# GCE 12yr — Comprehensive Visualization Notebook

Aggregates all final results from the 12yr pipeline, plus a set of diagnostic figures
ported from the 12yr validation notebook.

**Output plots** (saved to `./GCE_12yr_results_plots/`):

| #  | Plot                                | Source / inspiration                                  |
|----|-------------------------------------|-------------------------------------------------------|
| 01 | GCE counts map (3 E bands)          | Daylan+ 2014 Fig 7                                    |
| 02 | Masked counts map (PSC + disk)      | —                                                     |
| 03 | **SED decomposition (5 comp + GCE)**| 12yr V2 (per-bin fit coefficients applied)            |
| 04 | **80-model envelope + best-5 + σ_sys** | Cholis+ 2022 Fig 6 / Fig 12 (best-5 ±2σ style)     |
| 05 | Covariance matrix visualization     | new — heatmap + diagonal σ_sys profile                |
| 06 | **Multi-model coefficient comparison** | 12yr V10 (c_π+br, c_ICS, c_GCE, c_bub, c_iso)      |
| 07 | **Spatial residual maps + profiles**| 12yr V11 — (data − model)/data                        |
| 08 | bb̄ χ² contour (PPPC4)              | —                                                     |
| 09 | 4b χ² contour (MG5 SFDM)            | —                                                     |
| 10 | 4τ / 2b2τ χ² contours (MG5)         | —                                                     |
| 11 | bb̄ contour + dSph / p̄ overlay     | 12yr V22 / V26 (graceful fallback if files absent)    |
| 12 | Cholis 2022 published vs 12yr       | 12yr V24 + V25 (flux + DM contour)                    |
| 13 | Best-fit DM SED overlay             | —                                                     |

The cells use a defensive `SELECTED_MODEL` lookup, support multi-model comparison
(`MODELS_TO_COMPARE`), and gracefully skip overlays when external files aren't present.


In [ ]:
# Cell 1 — Imports + paths + constants
import os, sys, glob, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, FixedLocator, FormatStrFormatter
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import LogStretch, ImageNormalize
from scipy.interpolate import interp1d, RegularGridInterpolator

warnings.filterwarnings('ignore')

# === Paths ===
WORK_DIR = '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce'
if os.getcwd() != WORK_DIR:
    print(f'[chdir] {os.getcwd()} → {WORK_DIR}')
    os.chdir(WORK_DIR)
ANALYSIS_DIR = './GC_analysis_DR2'
RESULTS_DIR  = './results_12yr'
COV_DIR      = './results_cov_12yr'
PLOTS_DIR    = './GCE_12yr_results_plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

# Result file patterns
GCE_DAT_PATTERN = f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis.dat'
GCE_NPZ_PATTERN = f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis_fit.npz'
GCE_LH_PATTERN  = f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis_likelihood_value'
COV_NPZ         = f'{COV_DIR}/GCE_systematic_covariance_matrix_12yr.npz'

# Component fits file patterns (gtsrcmaps output, in ANALYSIS_DIR)
FRONT  = '_front'    # FRONT-only IRF
def comp_path(name, model=None, convol=True):
    """Build path for a component fits file.

    Examples:
      comp_path('pion', 'X')           -> ANALYSIS_DIR/GC_pion_modelX_12yr_front_clean.fits
      comp_path('pion', 'X', False)    -> ...GC_pion_modelX_12yr_front_clean_no_convol.fits
      comp_path('GCE')                 -> ANALYSIS_DIR/GC_GCE_model_12yr_front_clean.fits
                                          (GCE/bubble/isotropic don't take a Roman model)
    """
    suffix = 'clean' if convol else 'clean_no_convol'
    if model is None:
        return f'{ANALYSIS_DIR}/GC_{name}_model_12yr{FRONT}_{suffix}.fits'
    return f'{ANALYSIS_DIR}/GC_{name}_model{model}_12yr{FRONT}_{suffix}.fits'

# DM spectrum sources (paths from recent_updates)
PPPC4_CANDIDATES = [
    '/home/haebarg/GCE-Chi-square-fitting/PPPC4/particle_data',
    '/home/haebarg/GCE-Chi-square-fitting/PPPC4',
    '/home/haebarg/GCE-Chi-square-fitting',
]
PPPC4_DIR = None
for p in PPPC4_CANDIDATES:
    if os.path.exists(f'{p}/AtProduction_gammas.dat'):
        PPPC4_DIR = p
        break
if PPPC4_DIR is None:
    print('[warn] PPPC4 AtProduction_gammas.dat not found; defaulting to first candidate')
    PPPC4_DIR = PPPC4_CANDIDATES[0]

MG5_BASE = '/home/haebarg/MG5_aMC_v3_5_12'
MG5_CHANNEL_DIRS = {
    '4b':     f'{MG5_BASE}/Spectra_Data_sfdm_4b_r0.5',
    '4tau':   f'{MG5_BASE}/Spectra_Data_sfdm_4tau_r0.5',
    '2b2tau': f'{MG5_BASE}/Spectra_Data_sfdm_2b2tau_r0.5',
}

# === Physical constants (Sanghwan 16yr / Cholis convention) ===
J_FACTOR = 3.5251837158376415e+21   # GeV^2 cm^-5 sr  (NFW γ=1.2, 40°×40° masked)
SR       = 0.4288213187542626       # sr (40°×40° ROI solid angle, |b|>2° cut)

# Reference data — Cholis Zenodo (12yr published fluxes)
CHOLIS_REF_DIR = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'

# Cholis Fig 18 (DM contour reference) — same dir as 12yr if available
CHOLIS_FIG18_CANDIDATES = [
    '../GCE_TEMPLATES_FILES_v3/Figures_18_DM_contour',
    '/home/sanghwan/FermiLAT/Sanghwan/GCE_references',
]
CHOLIS_FIG18_DIR = next((p for p in CHOLIS_FIG18_CANDIDATES if os.path.exists(p)), None)

# External constraints (received from collaborators) — graceful fallback if absent
EXTERNAL_DATA_CANDIDATES = [
    '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_data',
    './external_constraints',
]
EXTERNAL_DATA_DIR = next((p for p in EXTERNAL_DATA_CANDIDATES if os.path.exists(p)), None)

# Roman numerals for 80 models
def roman(n):
    val = [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),(100,'C'),(90,'XC'),
           (50,'L'),(40,'XL'),(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
    out = ''
    for v, s in val:
        while n >= v:
            out += s; n -= v
    return out
ALL_MODELS = [roman(i) for i in range(1, 81)]

# === Sanity check ===
print(f'WORK_DIR        : {os.path.abspath(WORK_DIR)}')
print(f'RESULTS_DIR     : {RESULTS_DIR}')
print(f'ANALYSIS_DIR    : {ANALYSIS_DIR}')
print(f'PLOTS_DIR       : {PLOTS_DIR}')
print(f'COV_NPZ         : {COV_NPZ}')
print(f'PPPC4_DIR       : {PPPC4_DIR}')
print(f'MG5_BASE        : {MG5_BASE}')
print(f'CHOLIS_FIG18_DIR: {CHOLIS_FIG18_DIR}')
print(f'EXTERNAL_DATA   : {EXTERNAL_DATA_DIR}')
print(f'80 models       : {ALL_MODELS[:5]} ... {ALL_MODELS[-3:]}')
print(f'J_FACTOR (GeV²/cm⁵·sr) = {J_FACTOR:.3e}')
print(f'SR (sr)                = {SR:.4f}')

print(f'\n=== Path checks ===')
checks = [
    (RESULTS_DIR,                                'results dir'),
    (ANALYSIS_DIR,                               'analysis dir'),
    (f'{PPPC4_DIR}/AtProduction_gammas.dat',     'PPPC4 file'),
    (MG5_CHANNEL_DIRS['4b'],                     'MG5 4b dir'),
    (COV_NPZ,                                    'cov matrix npz'),
    (CHOLIS_REF_DIR,                             'Cholis Zenodo flux dir (optional)'),
    (CHOLIS_FIG18_DIR or '',                     'Cholis Fig 18 contour (optional)'),
]
for path, label in checks:
    ok = path and os.path.exists(path)
    print(f'  [{"OK" if ok else "MISS"}] {label}: {path}')
    
    

BIN_DIAG = 7   # diagnostic spotlight bin (used by cells 8g, 8h, 8i, 8r — E = 1.96 GeV in 14-bin grid)

## Model selection (V0_SETUP)

Defines `SELECTED_MODEL` (used by SED decomposition, residual map, single-model plots)
and `MODELS_TO_COMPARE` (used by the multi-model coefficient comparison).
Helper `list_available_models()` shows which `.dat` files exist on disk.

**Change `SELECTED_MODEL` and re-run the dependent cells.**


In [ ]:
# Cell 2 — SELECTED_MODEL + MODELS_TO_COMPARE (ported from 12yr V0_SETUP)
SELECTED_MODEL    = "I"                       # ← change to inspect a different model
MODELS_TO_COMPARE = ['I','X','XV','XLVIII','XLIX','LIII']         # ← coefficient comparison + best-N envelope

def list_available_models(verbose=True):
    """Return models with completed .dat on disk, sorted newest-first."""
    dats = glob.glob(GCE_DAT_PATTERN.format(model='*'))
    info = []
    for p in dats:
        # Roman label between 'GCE_model_' and '_front_12yr_cholis.dat'
        base = os.path.basename(p)
        lab  = base.replace('GCE_model_', '').replace('_front_12yr_cholis.dat', '')
        st   = os.stat(p)
        info.append((lab, p, st.st_size, st.st_mtime))
    info.sort(key=lambda r: -r[3])

    if verbose:
        if not info:
            print(f"[info] No completed .dat files in {RESULTS_DIR}/")
        else:
            print(f"  Found {len(info)} completed models:")
            print(f"  {'model':<10} {'size':>8}  {'modified':<20}")
            print('  ' + '-' * 50)
            for lab, p, size, mtime in info[:15]:
                ts = time.strftime('%Y-%m-%d %H:%M', time.localtime(mtime))
                print(f"  {lab:<10} {size:>8} B  {ts:<20}")
            if len(info) > 15:
                print(f"  ... and {len(info) - 15} more")
    return [lab for lab, *_ in info]

_avail = list_available_models(verbose=True)

if SELECTED_MODEL not in _avail and _avail:
    print(f"\n⚠ SELECTED_MODEL = {SELECTED_MODEL!r} not found in {RESULTS_DIR}/")
    print(f"  Falling back to {_avail[0]!r} (newest)")
    SELECTED_MODEL = _avail[0]
elif not _avail:
    print(f"\n⚠ No models available — cells that depend on data will skip gracefully.")
else:
    print(f"\n✓ SELECTED_MODEL    = {SELECTED_MODEL!r}")
    print(f"  MODELS_TO_COMPARE = {MODELS_TO_COMPARE}")


## DM Spectrum Loaders

Two sources for DM annihilation γ-ray spectra:

- **PPPC4** (Cirelli+ 2011): 2-body channels, with electroweak corrections
- **MG5 cascade** (MadGraph5_aMC@NLO): 4-body channels for SFDM model


In [ ]:
# Cell 3 — PPPC4 loader  (Sanghwan chi-square notebook style, channel index = 13 for bb̄)
def exctractcirellitable(DMmass, DMchannel, particle='gammas', EWcorr='Yes'):
    """Load PPPC4 dN/dE for given (mass, channel).

    DMchannel column index (with EW=Yes): 4=e⁺e⁻, 7=μ⁺μ⁻, 10=τ⁺τ⁻, 13=bb̄, 14=tt̄,
    17=W⁺W⁻, 20=ZZ, 22=gg, 23=hh.

    Returns: (E [GeV], dN/dE [1/GeV])
    """
    fname = (f'{PPPC4_DIR}/AtProduction_gammas.dat' if EWcorr == 'Yes'
             else f'{PPPC4_DIR}/AtProductionNoEW_gammas.dat')
    table = np.loadtxt(fname, skiprows=1)
    masses  = np.unique(table[:, 0])
    nearest = masses[np.argmin(np.abs(masses - DMmass))]
    rows    = table[table[:, 0] == nearest]
    log10x  = rows[:, 1]
    dNdlog10x = rows[:, DMchannel]
    x   = 10**log10x
    E   = x * nearest
    dNdE = dNdlog10x / (E * np.log(10))
    return E, dNdE

E_test, dNdE_test = exctractcirellitable(50.0, 13, 'gammas', 'Yes')
print(f'PPPC4 bb̄ at m_DM=50 GeV: {len(E_test)} points, '
      f'E ∈ [{E_test[0]:.2e}, {E_test[-1]:.2e}] GeV, peak dN/dE={dNdE_test.max():.3e}')


In [ ]:
# Cell 4 — MG5 cascade interpolator
_MG5_CACHE = {}

class MG5Interpolator:
    """2D interpolator on (DM_mass, log10(E)) grid for SFDM cascade spectra."""
    def __init__(self, mass, channel='4b', base_dir=None):
        self.mass = mass
        self.channel = channel
        if base_dir is None:
            base_dir = MG5_CHANNEL_DIRS[channel]
        self.base_dir = base_dir
        if channel not in _MG5_CACHE:
            self._load_table()
        self.interp     = _MG5_CACHE[channel]['interp']
        self.dm_masses  = _MG5_CACHE[channel]['masses']

    def _load_table(self):
        files = sorted(glob.glob(f'{self.base_dir}/MM_mpsi*GeV_*_photon.csv'))
        if not files:
            raise FileNotFoundError(f'No MG5 csv files in {self.base_dir}')
        masses, e_lists, dN_lists = [], [], []
        for f in files:
            base = os.path.basename(f)
            try:
                m = float(base.split('mpsi')[1].split('GeV')[0])
            except (IndexError, ValueError):
                continue
            try:
                data = np.loadtxt(f, delimiter=',', skiprows=1)
            except Exception:
                continue
            if data.size == 0:
                continue
            if data.ndim == 1:
                data = data.reshape(1, -1)
            masses.append(m); e_lists.append(data[:, 0]); dN_lists.append(data[:, 1])
        masses = np.array(masses)
        all_E = np.unique(np.concatenate(e_lists))
        all_E = all_E[all_E > 0]
        log10E_grid = np.linspace(np.log10(all_E.min()), np.log10(all_E.max()), 500)
        flux_mat = np.zeros((len(masses), len(log10E_grid)))
        for i, (e_arr, d_arr) in enumerate(zip(e_lists, dN_lists)):
            valid = e_arr > 0
            if valid.sum() < 2:
                continue
            f_int = interp1d(np.log10(e_arr[valid]), d_arr[valid],
                             bounds_error=False, fill_value=0.0)
            flux_mat[i] = f_int(log10E_grid)
        order = np.argsort(masses)
        masses, flux_mat = masses[order], flux_mat[order]
        interp = RegularGridInterpolator(
            (masses, log10E_grid), flux_mat,
            bounds_error=False, fill_value=0.0)
        _MG5_CACHE[self.channel] = {
            'interp': interp, 'masses': masses, 'log10E_grid': log10E_grid}
        print(f'  [MG5] loaded {self.channel}: {len(masses)} mass files, '
              f'm ∈ [{masses.min():.1f}, {masses.max():.1f}] GeV')

    def interpolated_table(self):
        log10E_grid = _MG5_CACHE[self.channel]['log10E_grid']
        E_arr = 10**log10E_grid
        pts = np.column_stack([np.full_like(log10E_grid, self.mass), log10E_grid])
        dNdE = self.interp(pts)
        dNdE = np.where(E_arr > self.mass, 0.0, dNdE)
        return E_arr, dNdE


# Pre-load all 3 MG5 channels (cache reused later)
for ch in ['4b', '4tau', '2b2tau']:
    try:
        E_t, dNdE_t = MG5Interpolator(50.0, ch).interpolated_table()
        n_nz = (dNdE_t > 0).sum()
        print(f'  [test] {ch} at m=50 GeV: {n_nz} non-zero pts, '
              f'peak dN/dE={dNdE_t.max():.3e}')
    except Exception as e:
        print(f'  [warn] {ch} load failed: {e}')


## GCE Flux + Component Templates + Covariance Loader

**This cell loads everything subsequent plots need:**

- All 80-model `.dat` files (E, flux, stat_err, lo, hi)
- Per-model `.npz` fit-coefficient files (when present)
- Component templates (π⁰+bremss, ICS, GCE, bubble, isotropic) for `SELECTED_MODEL`
- Per-model log-likelihood (for best-N ranking in Plot 4)
- Systematic covariance matrix → `cov_total = cov_stat + cov_sys`

Component-template loading uses the same convention as the main runner:
masked-disk-area-weighted average of `(component_map / exposure)` per energy bin.


In [ ]:
# Cell 5 — Load all 80 .dat, the cov matrix, and components for SELECTED_MODEL
# ----------------------------------------------------------------------------
# Output variables used downstream:
#   ALL .dat:       all_flux (dict: model -> {E,flux,stat_err,lower,upper}),
#                   all_loglike (dict: model -> sum log L)
#   Reference:      E, flux_X, stat_err
#   Cov:            cov_sys, sigma_sys, cov_total, inv_cov
#   Components:     pion, bremss, ics, GCE_t, bubble, isotropic, counts_per_exp,
#                   counts_per_exp_err, delta_E
#   Coefficients:   c_pion, c_ics, c_gce, c_bub, c_iso (from .npz, if present)
#   Multi-model:    coef_dict = {model: {c_pion, ..., c_iso}}

# ---- Load .dat for all 80 models ----
def load_model_flux(model):
    p = GCE_DAT_PATTERN.format(model=model)
    if not os.path.exists(p):
        return None
    d = np.loadtxt(p)
    return {
        'E':        d[:, 0],
        'flux':     d[:, 1],
        'stat_err': d[:, 2] if d.shape[1] > 2 else d[:, 1] * 0.1,
        'lower':    d[:, 3] if d.shape[1] > 3 else d[:, 1] - d[:, 2],
        'upper':    d[:, 4] if d.shape[1] > 4 else d[:, 1] + d[:, 2],
    }

all_flux    = {}
all_loglike = {}
for m in ALL_MODELS:
    res = load_model_flux(m)
    if res is None:
        continue
    all_flux[m] = res
    lh = GCE_LH_PATTERN.format(model=m)
    if os.path.exists(lh):
        try:
            all_loglike[m] = float(np.sum(np.loadtxt(lh)))
        except Exception:
            all_loglike[m] = np.nan
    else:
        all_loglike[m] = np.nan
print(f'Loaded {len(all_flux)} model .dat files; '
      f'{sum(np.isfinite(v) for v in all_loglike.values())} have log-likelihood')

# Reference (SELECTED_MODEL preferred, fallback to first available)
if SELECTED_MODEL in all_flux:
    _ref_model = SELECTED_MODEL
elif all_flux:
    _ref_model = next(iter(all_flux))
    print(f'[warn] SELECTED_MODEL={SELECTED_MODEL} missing; using {_ref_model} as ref')
else:
    _ref_model = None

if _ref_model:
    ref       = all_flux[_ref_model]
    E         = ref['E']
    flux_X    = ref['flux']
    stat_err  = ref['stat_err']
    n_bins    = len(E)
    print(f'Reference: Model {_ref_model}, '
          f'E ∈ [{E[0]:.3f}, {E[-1]:.1f}] GeV, peak flux={flux_X.max():.3e}')
else:
    E = flux_X = stat_err = None
    n_bins = 0

# ---- Load cov matrix ----
if os.path.exists(COV_NPZ):
    cov_data  = np.load(COV_NPZ)
    cov_sys   = cov_data['cov_matrix']
    sigma_sys = np.sqrt(np.diag(cov_sys))
    print(f'\nCov matrix: shape={cov_sys.shape}, '
          f'σ_sys peak={sigma_sys.max():.3e} at E={cov_data["E"][sigma_sys.argmax()]:.2f} GeV')
    if E is not None:
        cov_stat  = np.diag(stat_err**2)
        cov_total = cov_stat + cov_sys
        inv_cov   = np.linalg.inv(cov_total)
        print(f'cov_total condition number: {np.linalg.cond(cov_total):.2e}')
else:
    print(f'[warn] cov matrix not found: {COV_NPZ}')
    cov_sys = sigma_sys = cov_total = inv_cov = None


# ---- Load component templates for SELECTED_MODEL ----
def _build_component_loader():
    """Builds bin-wise (disk-mask-averaged) flux extractor for component cubes."""
    ccube_p   = f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits'
    expcube_p = f'{ANALYSIS_DIR}/GC_expcube_center_12yr{FRONT}_clean.fits'
    disk_p    = f'{ANALYSIS_DIR}/Model/GC_disk_mask_60x60_definitions.npy'
    for p, label in [(ccube_p, 'CCUBE'), (expcube_p, 'expcube'), (disk_p, 'disk mask')]:
        if not os.path.exists(p):
            print(f'  [warn] {label} missing: {p}')
            return None
    raw = fits.open(ccube_p)
    wcs = WCS(raw[0].header).dropaxis(2)
    W, H = raw[0].data.shape[1], raw[0].data.shape[2]
    # solid-angle per pixel
    srp = np.zeros((W, H))
    for i in range(H):
        for j in range(W):
            _, b = wcs.wcs_pix2world(j, i, 0)
            srp[i, j] = np.radians(0.1)**2 * np.cos(np.radians(b))
    exp = fits.open(expcube_p)[0].data[:, 100:500, 100:500] * srp[100:500, 100:500]
    disk = np.load(disk_p)[100:500, 100:500]
    n_E_loc = exp.shape[0]
    raw.close()

    def comp_flux_array(fname):
        out = np.zeros(n_E_loc)
        if not fname or not os.path.exists(fname):
            return out
        d = fits.open(fname)[0].data
        for i in range(n_E_loc):
            out[i] = np.sum(disk * (d[i][100:500, 100:500] / exp[i])) / np.sum(disk)
        return out
    return comp_flux_array, n_E_loc, exp, disk

_loader_info = _build_component_loader()
if _loader_info is None or _ref_model is None:
    pion = bremss = ics = GCE_t = bubble = isotropic = None
    counts_per_exp = counts_per_exp_err = delta_E = None
    print('\n[warn] component templates not loaded — SED decomposition + residual map will skip')
else:
    comp_flux_array, _n_E, _exp_arr, _disk_arr = _loader_info
    M = _ref_model
    pion      = comp_flux_array(comp_path('pion',          M, convol=False))
    bremss    = comp_flux_array(comp_path('bremss',        M, convol=False))
    ics       = comp_flux_array(comp_path('ics',           M, convol=False))
    GCE_t     = comp_flux_array(comp_path('GCE',           None, convol=False))
    bubble    = comp_flux_array(comp_path('fermi_bubble',  None, convol=False))
    isotropic = comp_flux_array(comp_path('isotropic',     None, convol=False))

    # Observed counts/exposure (for SED decomposition observed point)
    ccube_arr = fits.open(f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits')[0].data
    counts_per_exp     = np.zeros(_n_E)
    counts_per_exp_err = np.zeros(_n_E)
    for i in range(_n_E):
        c_in_roi = ccube_arr[i][100:500, 100:500]
        counts_per_exp[i]     = np.sum(_disk_arr * (c_in_roi / _exp_arr[i])) / np.sum(_disk_arr)
        counts_per_exp_err[i] = (np.sqrt(np.sum((np.sqrt(_disk_arr * c_in_roi) / _exp_arr[i])**2))
                                 / np.sum(_disk_arr))

    # Energy bin widths (from CCUBE EBOUNDS HDU, in keV → GeV)
    ebounds = fits.open(f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits')[1].data
    delta_E = np.array([(b['E_MAX'] - b['E_MIN']) * 1e-6 for b in ebounds])
    print(f'\nComponents loaded for Model {M}:')
    print(f'  π+br peak={(pion+bremss).max():.3e}, ICS peak={ics.max():.3e}, '
          f'GCE peak={GCE_t.max():.3e}')
    print(f'  bubble peak={bubble.max():.3e}, iso peak={isotropic.max():.3e}')


# ---- Load fit coefficients (.npz) for SELECTED_MODEL + comparison set ----
def _load_coefs(model, n):
    p = GCE_NPZ_PATTERN.format(model=model)
    if not os.path.exists(p):
        return None
    npz = np.load(p)
    fp = npz['fitted_params']
    if fp.ndim == 2 and fp.shape[0] == 5:
        # main-runner format (5, n_bins)
        return dict(c_pion=fp[0], c_ics=fp[1], c_gce=fp[2], c_bub=fp[3], c_iso=fp[4])
    elif fp.ndim == 1 and fp.size == 5 * n:
        # flat (5*n,) format (12yr)
        return dict(c_pion=fp[0*n:1*n], c_ics=fp[1*n:2*n], c_gce=fp[2*n:3*n],
                    c_bub=fp[3*n:4*n],  c_iso=fp[4*n:5*n])
    print(f'  [warn] unknown fitted_params shape for {model}: {fp.shape}')
    return None

c_pion = c_ics = c_gce = c_bub = c_iso = None
coef_dict = {}
if E is not None:
    for m in [SELECTED_MODEL] + [m for m in MODELS_TO_COMPARE if m != SELECTED_MODEL]:
        cd = _load_coefs(m, n_bins)
        if cd is not None:
            coef_dict[m] = cd
    if SELECTED_MODEL in coef_dict:
        c = coef_dict[SELECTED_MODEL]
        c_pion, c_ics, c_gce, c_bub, c_iso = (c['c_pion'], c['c_ics'], c['c_gce'],
                                              c['c_bub'], c['c_iso'])
        print(f'\nFit coefficients for Model {SELECTED_MODEL} (mean ± std):')
        print(f'  c_π+br = {c_pion.mean():.3f} ± {c_pion.std():.3f}')
        print(f'  c_ICS  = {c_ics.mean():.3f} ± {c_ics.std():.3f}')
        print(f'  c_GCE  = {c_gce.mean():.3f} ± {c_gce.std():.3f}')
        print(f'  c_bub  = {c_bub.mean():.3f} ± {c_bub.std():.3f}')
        print(f'  c_iso  = {c_iso.mean():.3f} ± {c_iso.std():.3f}')
    else:
        print(f'\n[info] no .npz coefficients for {SELECTED_MODEL} — '
              f'SED decomposition will use raw templates (c=1).')
    if len(coef_dict) > 1:
        print(f'\nLoaded coefficients for comparison: {list(coef_dict.keys())}')


# === [Patch] Cov fallback + chi^2 bin filter ===
# This block tries multiple cov sources, then falls back to stat-only.
# Defines CHI2_USE_BINS and inv_cov_fit unconditionally (for chi^2 cells).
if inv_cov is None and E is not None:
    # Try 1: archived (pre-bubble-fix) cov in standard location
    _archived_cov = './archive/pre_bubble_fix_cov/GCE_systematic_covariance_matrix_12yr.npz'
    if os.path.exists(_archived_cov):
        _ad = np.load(_archived_cov)
        cov_sys   = _ad['cov_matrix']
        sigma_sys = np.sqrt(np.diag(cov_sys))
        cov_total = np.diag(stat_err**2) + cov_sys
        inv_cov   = np.linalg.inv(cov_total)
        print(f'\n[temporary] Loaded ARCHIVED (pre-bubble-fix) cov matrix.')
    else:
        print(f'\n[note] no archived cov at {_archived_cov}')

    # Try 2: stat-only fallback (no systematic; chi^2 will be tight)
    if inv_cov is None:
        cov_sys   = np.zeros((len(E), len(E)))
        sigma_sys = np.zeros(len(E))
        cov_total = np.diag(stat_err**2)
        inv_cov   = np.linalg.inv(cov_total)
        print(f'[fallback] Using STAT-ERROR-ONLY chi^2 (no systematic). '
              f'Replace with proper cov matrix when ready.')

# Bin filter: paper-exact — use ALL 14 bins (Cholis+2022).
# (Earlier slice(2,None) was a temporary test of dropping the
#  DR2 mask boundary-stuck bins 0/1; reverted -> full 14-bin fit.)
CHI2_USE_BINS = slice(0, None)
if E is not None and inv_cov is not None:
    # Sub-cov for fit bins (2-13), then invert
    _cov_total_full = np.linalg.inv(inv_cov)
    _cov_total_fit  = _cov_total_full[CHI2_USE_BINS, CHI2_USE_BINS]
    inv_cov_fit = np.linalg.inv(_cov_total_fit)
    print(f'[fit] chi^2 will use bins {CHI2_USE_BINS.start}-{len(E)-1} '
          f'(E in [{E[CHI2_USE_BINS][0]:.2f}, {E[-1]:.1f}] GeV)')
else:
    inv_cov_fit = None
    print('[fit] inv_cov_fit = None (E or cov missing)')


## Plot 1 — GCE Map (Raw Counts)

Raw photon counts in 3 energy bands, Galactic coordinates centred on the GC
(Daylan+ 2014 Fig 7 style).


In [ ]:
# Cell 6 — Plot 1: GCE counts map (3 E bands)
ccube_path = f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits'
hdul = fits.open(ccube_path)
counts_cube = hdul[0].data
header      = hdul[0].header
n_E_full, ny, nx = counts_cube.shape

E_bounds = hdul[1].data
E_arr_ccube = np.array([np.sqrt(b['E_MIN'] * b['E_MAX'] * 1e-6) * 1e-3
                        for b in E_bounds])
print(f'CCUBE: shape={counts_cube.shape}, E ∈ [{E_arr_ccube[0]:.3f}, '
      f'{E_arr_ccube[-1]:.1f}] GeV')

# 3 E bands (Daylan-style)
bands = [
    (0.3, 1.0,  '0.3 - 1 GeV'),
    (1.0, 10.0, '1 - 10 GeV'),
    (10.0, 60.0,'10 - 60 GeV'),
]

extent = [-30, 30, -30, 30]   # 600 px × 0.1°/px = 60° centred on GC

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (e_lo, e_hi, label) in zip(axes, bands):
    sel = (E_arr_ccube >= e_lo) & (E_arr_ccube <= e_hi)
    summed = counts_cube[sel].sum(axis=0)
    im = ax.imshow(summed, origin='lower', cmap='magma',
                   extent=extent,
                   norm=LogNorm(vmin=max(1, summed.min() + 1), vmax=summed.max()))
    ax.set_title(label, fontsize=12)
    ax.set_xlabel(r'Galactic longitude $\ell$ [deg]')
    ax.set_ylabel('Galactic latitude $b$ [deg]')
    ax.invert_xaxis()  # ℓ increases to the left (astronomical convention)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='counts')

plt.suptitle('GCE region — raw photon counts (12yr, FRONT-only)', y=1.02, fontsize=13)
plt.tight_layout()
out = f'{PLOTS_DIR}/01_gce_map.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'[saved] {out}')
plt.show()


## Plot 2 — Masked Map

PSC + disk mask applied to the CCUBE, showing the area actually used in the fit.


In [ ]:
# Cell 7 — Plot 2: Masked counts map (PSC + disk mask)
psc_mask_path  = f'{ANALYSIS_DIR}/Model/GC_mask_60x60_definitions_DR2.npy'
disk_mask_path = f'{ANALYSIS_DIR}/Model/GC_disk_mask_60x60_definitions.npy'

psc_mask  = np.load(psc_mask_path)
disk_mask = np.load(disk_mask_path)

# Inner ROI 400×400 (40°×40°)
psc_inner   = psc_mask[:, 100:500, 100:500]
disk_inner  = disk_mask[100:500, 100:500]
counts_inner = counts_cube[:, 100:500, 100:500]
full_inner   = psc_inner * disk_inner[None, :, :]

extent_inner = [-20, 20, -20, 20]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (e_lo, e_hi, label) in zip(axes, bands):
    sel = (E_arr_ccube >= e_lo) & (E_arr_ccube <= e_hi)
    summed = counts_inner[sel].sum(axis=0)
    mask_b = full_inner[sel].max(axis=0)
    masked_disp = np.where(mask_b == 1, summed, np.nan)

    im = ax.imshow(masked_disp, origin='lower', cmap='viridis',
                   extent=extent_inner,
                   norm=LogNorm(vmin=1, vmax=summed.max()))
    frac_kept = mask_b.sum() / mask_b.size
    ax.set_title(f'{label}\n({frac_kept * 100:.1f}% pixels kept)', fontsize=11)
    ax.set_xlabel(r'Galactic longitude $\ell$ [deg]')
    ax.set_ylabel('Galactic latitude $b$ [deg]')
    ax.invert_xaxis()
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='counts')

plt.suptitle('Masked counts (PSC + disk mask applied, 12yr)', y=1.02, fontsize=13)
plt.tight_layout()
out = f'{PLOTS_DIR}/02_masked_map.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'[saved] {out}')
plt.show()


## Catalog comparison — 4FGL-DR2 vs DR2 point sources

Reads the DR2 point-source catalog (`gll_psc_v40.fit`) directly and computes the source counts and class breakdown in the analysis ROI. If a 4FGL-DR2 catalog file or the 12yr PSC XML is available, also lists the per-catalog totals for direct comparison.


In [ ]:
# Cell 7a - Catalog content comparison: 4FGL-DR2 vs DR2
# ---------------------------------------------------------
# Reads DR2 FITS catalog and (optionally) DR2 source XML to count sources in ROI,
# break down by association status (new vs existing in 4FGL family), source class,
# and Signif_Avg distribution.
import xml.etree.ElementTree as ET

# Catalog file paths — try common locations, graceful skip if missing
DR2_FITS_CANDIDATES = [
    f'{ANALYSIS_DIR}/Model/gll_psc_v40.fit',
    f'{ANALYSIS_DIR}/gll_psc_v40.fit',
    f'./gll_psc_v40.fit',
    '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce/gll_psc_v40.fit',
]
DR2_XML_CANDIDATES = [
    '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce/GC_analysis_sanghwan/Model/GC_psc_model_DR2.xml',
]

def _first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

fl16y_path = _first_existing(DR2_FITS_CANDIDATES)
dr2_path   = _first_existing(DR2_XML_CANDIDATES)

if fl16y_path is None:
    print('[skip] DR2 catalog FITS not found in candidate paths.')
    print('       Tried:')
    for p in DR2_FITS_CANDIDATES:
        print(f'         {p}')
else:
    print(f'DR2 catalog: {fl16y_path}')
    cat = fits.open(fl16y_path)
    psc = cat[1].data    # LAT_Point_Source_Catalog
    ext = cat[2].data    # ExtendedSources

    # Galactic coordinates with GLON wrap to [-180, 180]
    glon      = psc['GLON']
    glat      = psc['GLAT']
    glon_wrap = np.where(glon > 180, glon - 360, glon)
    ext_glon      = ext['GLON']
    ext_glat      = ext['GLAT']
    ext_glon_wrap = np.where(ext_glon > 180, ext_glon - 360, ext_glon)

    # ROI definitions
    roi60 = (np.abs(glon_wrap)     <= 30) & (np.abs(glat)     <= 30)
    roi40 = (np.abs(glon_wrap)     <= 20) & (np.abs(glat)     <= 20)
    ext60 = (np.abs(ext_glon_wrap) <= 30) & (np.abs(ext_glat) <= 30)
    ext40 = (np.abs(ext_glon_wrap) <= 20) & (np.abs(ext_glat) <= 20)

    n_psc_60 = int(roi60.sum()); n_ext_60 = int(ext60.sum())
    n_psc_40 = int(roi40.sum()); n_ext_40 = int(ext40.sum())

    print()
    print(f'DR2 total PSC      (full sky): {len(psc)}')
    print(f'DR2 total extended (full sky): {len(ext)}')
    print()
    print(f'In ROI 60deg x 60deg : {n_psc_60} PSC + {n_ext_60} extended = {n_psc_60 + n_ext_60}')
    print(f'In ROI 40deg x 40deg : {n_psc_40} PSC + {n_ext_40} extended = {n_psc_40 + n_ext_40}')

    # ASSOC_FGL based novelty: source has no 4FGL counterpart -> new in DR2
    def _empty_assoc(s):
        return s.strip() == '' or s.strip().lower() == 'none'
    assoc_60 = psc['ASSOC_FGL'][roi60]
    new_60 = np.array([_empty_assoc(s) for s in assoc_60])
    assoc_40 = psc['ASSOC_FGL'][roi40]
    new_40 = np.array([_empty_assoc(s) for s in assoc_40])

    print()
    print(f'Newly found in DR2 (no 4FGL_DR1/2/3 counterpart):')
    print(f'  ROI 60deg: {new_60.sum():4d} / {n_psc_60} ({100*new_60.sum()/n_psc_60:5.1f}%)')
    print(f'  ROI 40deg: {new_40.sum():4d} / {n_psc_40} ({100*new_40.sum()/n_psc_40:5.1f}%)')

    # CLASS1 distribution in ROI 60
    print()
    print('CLASS1 distribution in ROI 60deg (top 12):')
    class1 = psc['CLASS1'][roi60]
    cleaned = np.array([c.strip() if c.strip() else '(unassoc)' for c in class1])
    unique, counts = np.unique(cleaned, return_counts=True)
    sort_idx = np.argsort(-counts)
    for u, c in zip(unique[sort_idx][:12], counts[sort_idx][:12]):
        print(f'  {u:<15} {c:>4}')

    # Signif_Avg distribution
    sig60 = psc['Signif_Avg'][roi60]
    print()
    print('Signif_Avg (~ sqrt(TS)) distribution in ROI 60deg:')
    print(f'  median: {np.nanmedian(sig60):.1f}, max: {np.nanmax(sig60):.1f}')
    print(f'  Signif_Avg > 5  (TS > 25):  {(sig60 > 5).sum():4d}')
    print(f'  Signif_Avg > 7  (TS > 49):  {(sig60 > 7).sum():4d}    <- DR2 theta_s/theta_l split')
    print(f'  Signif_Avg > 10 (TS > 100): {(sig60 > 10).sum():4d}')

# DR2 XML count for direct comparison
if dr2_path is None:
    print()
    print('[note] 4FGL-DR2 PSC XML not found; DR2 stats only.')
else:
    print()
    print(f'4FGL-DR2 PSC XML (12yr analysis): {dr2_path}')
    tree = ET.parse(dr2_path)
    root = tree.getroot()
    dr2_sources = root.findall('source')
    n_dr2 = len(dr2_sources)
    print(f'  Number of <source> entries (point + diffuse + extended): {n_dr2}')

    # Count by point vs extended (the XML mixes them; differentiate by spatialModel)
    n_dr2_point = 0; n_dr2_ext = 0; n_dr2_diff = 0
    for s in dr2_sources:
        sm = s.find('spatialModel')
        if sm is None:
            continue
        sm_type = sm.get('type', '')
        if sm_type == 'SkyDirFunction':
            n_dr2_point += 1
        elif sm_type in ('SpatialMap', 'RadialDisk', 'RadialGaussian'):
            n_dr2_ext += 1
        else:
            n_dr2_diff += 1
    print(f'    point source (SkyDirFunction):     {n_dr2_point}')
    print(f'    extended (SpatialMap/Disk/Gauss):  {n_dr2_ext}')
    print(f'    other (diffuse / isotropic etc):   {n_dr2_diff}')

# Visualization: 3-panel summary of DR2 in ROI
if fl16y_path is not None:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Panel 1: source positions (l, b) in ROI 60, colored by new/existing
    ax = axes[0]
    glon_60 = glon_wrap[roi60]
    glat_60 = glat[roi60]
    ax.scatter(glon_60[~new_60], glat_60[~new_60], s=15, c='C0', alpha=0.7,
               label=f'Existing (4FGL counterpart): {(~new_60).sum()}')
    ax.scatter(glon_60[ new_60], glat_60[ new_60], s=30, c='C3', marker='x',
               label=f'New in DR2: {new_60.sum()}')
    # Extended sources too
    ax.scatter(ext_glon_wrap[ext60], ext_glat[ext60], s=80, marker='*',
               edgecolor='black', facecolor='gold', label=f'Extended: {n_ext_60}')
    # Inner 40 deg ROI box
    rect = plt.Rectangle((-20, -20), 40, 40, fill=False, ec='gray', ls='--', lw=1.5)
    ax.add_patch(rect)
    ax.set_xlim(30, -30); ax.set_ylim(-30, 30)
    ax.set_xlabel(r'GLON ($l$) [deg]'); ax.set_ylabel(r'GLAT ($b$) [deg]')
    ax.set_title(f'DR2 sources in ROI 60deg ({n_psc_60} PSC + {n_ext_60} ext)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

    # Panel 2: Signif_Avg histogram (log)
    ax = axes[1]
    sig_clean = sig60[~np.isnan(sig60)]
    bins = np.logspace(np.log10(3), np.log10(500), 40)
    ax.hist(sig_clean, bins=bins, color='steelblue', edgecolor='black', alpha=0.85)
    for v, lbl in [(5, 'TS=25'), (7, 'TS=49'), (10, 'TS=100')]:
        ax.axvline(v, ls='--', color='red', alpha=0.6)
        ax.text(v, ax.get_ylim()[1]*0.92, ' '+lbl, fontsize=9, color='red',
                rotation=90, va='top')
    ax.set_xscale('log')
    ax.set_xlabel('Signif_Avg ~ sqrt(TS)'); ax.set_ylabel('N sources')
    ax.set_title(f'Detection significance distribution (ROI 60deg)\nmedian = {np.nanmedian(sig60):.1f}')
    ax.grid(True, alpha=0.3)

    # Panel 3: CLASS1 breakdown bar chart
    ax = axes[2]
    top_n = 10
    cls_top = unique[sort_idx][:top_n]
    cnt_top = counts[sort_idx][:top_n]
    other = counts[sort_idx][top_n:].sum() if len(counts) > top_n else 0
    if other > 0:
        cls_top = np.append(cls_top, 'other')
        cnt_top = np.append(cnt_top, other)
    bars = ax.barh(np.arange(len(cls_top))[::-1], cnt_top, color='cornflowerblue', edgecolor='black')
    for bar, n in zip(bars, cnt_top):
        ax.text(bar.get_width() + max(cnt_top)*0.01, bar.get_y() + bar.get_height()/2,
                f'{n}', va='center', fontsize=9)
    ax.set_yticks(np.arange(len(cls_top))[::-1])
    ax.set_yticklabels(cls_top)
    ax.set_xlabel('N sources')
    ax.set_title(f'CLASS1 breakdown (ROI 60deg)')
    ax.grid(True, alpha=0.3, axis='x')

    fig.suptitle(f'DR2 catalog content in GC ROI', fontsize=13, fontweight='bold', y=1.00)
    fig.subplots_adjust(top=0.92, bottom=0.10, left=0.05, right=0.98, wspace=0.30)

    out = f'{PLOTS_DIR}/02a_catalog_DR2_breakdown.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'\nsaved: {out}')


## Plot 3 — SED Decomposition (NEW)

Single-model decomposition of the GCE region SED into the 5 background components
plus the GCE template, with fit coefficients applied. Ported from 12yr V2.

- π⁰+bremss, ICS, Fermi bubble, isotropic — backgrounds
- GCE — main signal (best-fit per-bin flux with 1σ band)
- Observed (CCUBE-masked) — black markers
- Sum of fitted components — gray dashed line (should overlap with observed)

Requires component templates and fit coefficients loaded in the previous cell.


In [ ]:
# Cell 8 — Plot 3: Single-model SED decomposition (5 components + GCE + observed)
if (E is None) or (pion is None) or (delta_E is None):
    print('[skip] missing E / components / delta_E — re-run Cell 5')
else:
    n = len(E)
    have_coefs = (c_pion is not None)
    if not have_coefs:
        c_pion = c_ics = c_gce = c_bub = c_iso = np.ones(n)
        title_tag = 'raw templates (c=1 fallback)'
    else:
        title_tag = 'fitted coefficients'

    E2_dE = E**2 / delta_E

    pb_sed  = c_pion * (pion + bremss) * E2_dE
    ics_sed = c_ics  * ics             * E2_dE
    bub_sed = c_bub  * bubble          * E2_dE
    iso_sed = c_iso  * isotropic       * E2_dE

    # observed = (counts/exp/sr) × E²/dE
    obs_E2dN = counts_per_exp * E2_dE

    fig, ax = plt.subplots(figsize=(10.5, 7))
    ax.plot(E, pb_sed,  color='red',     lw=1.6, label=r'$\pi^0$+bremss')
    ax.plot(E, ics_sed, color='blue',    lw=1.6, label='ICS')
    ax.plot(E, bub_sed, color='purple',  lw=1.6, label='Fermi bubble')
    ax.plot(E, iso_sed, color='green',   lw=1.6, label='Isotropic')

    # Observed
    ax.scatter(E, obs_E2dN, color='black', marker='o', s=32,
               zorder=10, label='Observed (CCUBE)')

    # GCE: use main .dat results (asymmetric error bars)
    ref = all_flux[SELECTED_MODEL]
    yerr_lo = np.maximum(ref['flux'] - ref['lower'], 0)
    yerr_hi = np.maximum(ref['upper'] - ref['flux'], 0)
    ax.errorbar(E, ref['flux'], yerr=[yerr_lo, yerr_hi],
                ls='-', marker='s', color='orange', ms=7, lw=2,
                elinewidth=2, capsize=4, capthick=2,
                label=f'GCE (Model {SELECTED_MODEL})')

    # Sum of fitted
    if have_coefs:
        total = pb_sed + ics_sed + bub_sed + iso_sed + ref['flux']
        ax.plot(E, total, color='gray', ls='--', lw=1.2, alpha=0.75,
                label='Sum of fitted components')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.3, 60);  ax.set_ylim(1e-8, 3e-5)
    ax.set_xlabel('E [GeV]', fontsize=12)
    ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=12)
    ax.set_title(f'SED decomposition — Model {SELECTED_MODEL}  '
                 f'(12yr, {title_tag})', fontsize=12)
    ax.grid(True, alpha=0.3, which='major')
    ax.legend(loc='upper right', ncol=2, fontsize=9, framealpha=0.92)

    plt.tight_layout()
    out = f'{PLOTS_DIR}/03_SED_decomposition_{SELECTED_MODEL}.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()

    # Brief quality check
    if have_coefs and not np.isnan(obs_E2dN).all():
        sum_tot = pb_sed + ics_sed + bub_sed + iso_sed + ref['flux']
        with np.errstate(invalid='ignore', divide='ignore'):
            ratio = sum_tot / obs_E2dN
        m_in = (E >= 1) & (E <= 10)
        if m_in.any():
            r_mean = np.nanmean(ratio[m_in])
            verdict = ('✓ converged' if 0.95 < r_mean < 1.05
                       else f'⚠ residual {(r_mean - 1)*100:+.0f}% (1-10 GeV)')
            print(f'  Sum/obs ratio (1-10 GeV mean): {r_mean:.3f} — {verdict}')


## Plot 3b — Per-model post-fit SED decomposition + selected-bin breakdown

Reimplements the 16yr main-notebook **cell-59** component SED (dropped during the 12yr pipeline migration; logic matches `export_postfit_sed.py` / `run_one_model.py` L500-519). Set `PLOT_MODEL` and `PLOT_BIN`:

- **Left** — full 14-bin post-fit decomposition: raw data (errorbar), π⁰+brem, ICS, GCE, bubble, iso each as `c·template·E²/ΔE` with asymmetric chain band (`fitted_params_lower/upper`), plus the summed total. Selected bin marked with a guide.
- **Right** — the selected bin's component breakdown with chain error bars + the per-bin coefficient values (surfaces the bin 0–1 `c_iso→0` / DR2-mask boundary-stuck pathology honestly, not hidden).

`SAVE_V28D=True` also writes the 16-column V28d post-fit table (restores the dropped pipeline export). Templates use `_no_convol`, `[100:500,100:500]` slice, disk-mask-only average, `/exp_cube` (sr folded in) — identical convention to Cell 5.

In [ ]:
# Cell 8c — Plot 3c: Fig 11 overlay vs Cholis 2022 Model I (paper-faithful 12yr comparison)
# Reads 2112_09706_Fig11_ModelI_ALL.txt (manual digitize of Cholis 2022 Fig 11,
# Model I, 4FGL-DR2 mask, |b|>2° ROI, post-fit dashed line + 2σ band).
# Overlays our 5-component post-fit SED + observed CCUBE flux + GCE result
# onto the Cholis paper bands.

FIG11_TXT = '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce/2112_09706_Fig11_ModelI_ALL.txt'
# (path: project knowledge file 또는 working dir 어디든 적당히)

if not os.path.exists(FIG11_TXT):
    # try project-knowledge mount path
    for cand in ['/mnt/project/2112_09706_Fig11_ModelI_ALL.txt',
                 './2112_09706_Fig11_ModelI_ALL.txt']:
        if os.path.exists(cand):
            FIG11_TXT = cand; break

if (E is None) or (pion is None) or (delta_E is None):
    print('[skip] 12yr components not loaded')
elif not os.path.exists(FIG11_TXT):
    print(f'[skip] Cholis Fig 11 data not found: {FIG11_TXT}')
else:
    fig11 = np.loadtxt(FIG11_TXT)
    E_p  = fig11[:, 0]
    pb_p, pb_lo, pb_hi   = fig11[:, 1], fig11[:, 2], fig11[:, 3]
    ic_p, ic_lo, ic_hi   = fig11[:, 4], fig11[:, 5], fig11[:, 6]
    bb_p, bb_lo, bb_hi   = fig11[:, 7], fig11[:, 8], fig11[:, 9]
    is_p, is_lo, is_hi   = fig11[:,10], fig11[:,11], fig11[:,12]
    gc_p, gc_lo, gc_hi   = fig11[:,13], fig11[:,14], fig11[:,15]

    # ----- our post-fit SED -----
    E2_dE = E**2 / delta_E
    have_coefs = (c_pion is not None)
    if not have_coefs:
        c_pion = c_ics = c_gce = c_bub = c_iso = np.ones(len(E))
    pb_our  = c_pion * (pion + bremss) * E2_dE
    ics_our = c_ics  * ics             * E2_dE
    bub_our = c_bub  * bubble          * E2_dE
    iso_our = c_iso  * isotropic       * E2_dE
    gce_our = c_gce  * GCE_t           * E2_dE     # = .dat 의 GCE flux 와 동일
    obs_our = counts_per_exp * E2_dE

    ref     = all_flux[SELECTED_MODEL]
    yerr_lo = np.maximum(ref['flux'] - ref['lower'], 0)
    yerr_hi = np.maximum(ref['upper'] - ref['flux'], 0)

    # ----- plot -----
    fig, ax = plt.subplots(figsize=(10, 7.5))

    # Cholis paper bands (2σ shaded + dashed center)
    band_kw = dict(alpha=0.18, lw=0)
    line_kw = dict(ls='--', lw=1.4, alpha=0.85)
    ax.fill_between(E_p, pb_lo, pb_hi, color='red',     **band_kw)
    ax.plot(E_p, pb_p, color='red',    label=r'Cholis: $\pi^0$+brem',  **line_kw)
    ax.fill_between(E_p, ic_lo, ic_hi, color='dodgerblue', **band_kw)
    ax.plot(E_p, ic_p, color='dodgerblue', label='Cholis: ICS',         **line_kw)
    ax.fill_between(E_p, bb_lo, bb_hi, color='purple',  **band_kw)
    ax.plot(E_p, bb_p, color='purple', label='Cholis: Bubble',          **line_kw)
    ax.fill_between(E_p, is_lo, is_hi, color='green',   **band_kw)
    ax.plot(E_p, is_p, color='green',  label='Cholis: Iso',             **line_kw)
    ax.fill_between(E_p, gc_lo, gc_hi, color='magenta', **band_kw)
    ax.plot(E_p, gc_p, color='magenta',label='Cholis: GCE',             **line_kw)

    # Our 12yr Model I bins (markers same color as Cholis)
    mk_kw = dict(ms=8, ls='-', lw=1.6, mew=0)
    ax.plot(E, pb_our,  marker='o', color='red',        label=r'12yr: $\pi^0$+brem', **mk_kw)
    ax.plot(E, ics_our, marker='s', color='dodgerblue', label='12yr: ICS',           **mk_kw)
    ax.plot(E, bub_our, marker='^', color='purple',     label='12yr: Bubble',        **mk_kw)
    ax.plot(E, iso_our, marker='D', color='green',      label='12yr: Iso',           **mk_kw)
    ax.errorbar(E, ref['flux'], yerr=[yerr_lo, yerr_hi],
                marker='*', ms=12, color='magenta', lw=1.8,
                elinewidth=1.6, capsize=3, label='12yr: GCE')
    ax.scatter(E, obs_our, marker='x', s=60, color='black',
               zorder=11, label='12yr: Observed (CCUBE)')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.3, 60); ax.set_ylim(1e-8, 3e-5)
    ax.set_xlabel('E [GeV]', fontsize=13)
    ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=13)
    ax.set_title('Fig 11 overlay — Cholis 2022 Model I (bands) vs haebarg 12yr Model I (markers)',
                 fontsize=12)
    ax.grid(True, which='major', alpha=0.3)
    ax.legend(loc='lower left', ncol=2, fontsize=8.5, framealpha=0.9)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/03c_Fig11_overlay_ModelI.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()

    # ----- numerical residual check (1–10 GeV mean ratio per component) -----
    from scipy.interpolate import interp1d
    def _interp(y_p): return interp1d(E_p, y_p, kind='linear',
                                       bounds_error=False, fill_value=np.nan)(E)
    pb_p_E  = _interp(pb_p); ic_p_E = _interp(ic_p); bb_p_E = _interp(bb_p)
    is_p_E  = _interp(is_p); gc_p_E = _interp(gc_p)
    m = (E >= 1) & (E <= 10)
    def _r(ours, paper): 
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.nanmean((ours / paper)[m])
    print(f'\n=== 1-10 GeV mean ratio (haebarg 12yr / Cholis paper) ===')
    print(f'  π⁰+brem : {_r(pb_our,  pb_p_E):.3f}')
    print(f'  ICS     : {_r(ics_our, ic_p_E):.3f}')
    print(f'  Bubble  : {_r(bub_our, bb_p_E):.3f}')
    print(f'  Iso     : {_r(iso_our, is_p_E):.3f}')
    print(f'  GCE     : {_r(ref["flux"], gc_p_E):.3f}')
    print(f'  (1.00 = perfect overlay; ±0.15 = within paper 2σ band typically)')

In [ ]:
# Cell 8g — (C1) Spatial pattern diagnostic
# Question: 우리 (π+brem) 과 ICS 의 spatial pattern 이 paper 와 어떻게 다른가?
# Approach: ROI 내 픽셀별 (pion+brem)/ICS ratio 의 분포 + 두 template 의 cross-correlation

if (E is not None) and (pion is not None):
    eb = BIN_DIAG
    ccube_p = f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits'
    pibr_p  = f'{ANALYSIS_DIR}/GC_pion_modelI_12yr{FRONT}_clean.fits'
    pibr_b  = f'{ANALYSIS_DIR}/GC_bremss_modelI_12yr{FRONT}_clean.fits'
    ics_p   = f'{ANALYSIS_DIR}/GC_ics_modelI_12yr{FRONT}_clean.fits'

    # load full 600×600 maps at bin 7
    pibr_full = (fits.open(pibr_p)[0].data[eb] + fits.open(pibr_b)[0].data[eb])
    ics_full  = fits.open(ics_p)[0].data[eb]
    data_full = fits.open(ccube_p)[0].data[eb]

    # ROI: [100:500, 100:500] (40×40 deg around GC)
    pibr_roi = pibr_full[100:500, 100:500]
    ics_roi  = ics_full[100:500, 100:500]
    data_roi = data_full[100:500, 100:500]

    # mask
    pm = np.load(f'{ANALYSIS_DIR}/Model/GC_mask_60x60_definitions_DR2.npy')[eb, 100:500, 100:500]
    dm = np.load(f'{ANALYSIS_DIR}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]
    full_mask = pm * dm

    # statistics inside mask
    mask_bool = (full_mask == 1)
    n_pix = mask_bool.sum()
    print(f'=== Spatial diagnostic, bin {eb} (E={E[eb]:.2f} GeV) ===')
    print(f'ROI pixels in mask: {n_pix} / {full_mask.size}')

    pi_in = pibr_roi[mask_bool]
    ic_in = ics_roi[mask_bool]
    dat_in = data_roi[mask_bool]

    # 1) global ratio
    print(f'\n--- global ROI sums (mask-averaged) ---')
    print(f'  Σ (pi+brem) inside mask: {pi_in.sum():.3e}')
    print(f'  Σ ICS       inside mask: {ic_in.sum():.3e}')
    print(f'  ratio (pi+brem) / ICS  : {pi_in.sum()/ic_in.sum():.3f}')
    print(f'  paper Fig 11 ratio at 2 GeV: {(7.96e-6 + 3.14e-6 * 0.7)/3.14e-6:.2f}')
    # ↑ paper 의 (π+brem)/ICS ratio 추정 (Fig 11 1.96 GeV 부근 값)

    # 2) cross-correlation
    pi_norm = (pi_in - pi_in.mean()) / pi_in.std()
    ic_norm = (ic_in - ic_in.mean()) / ic_in.std()
    cross_corr = np.mean(pi_norm * ic_norm)
    print(f'\n--- spatial correlation ---')
    print(f'  Pearson r((pi+brem), ICS) inside mask: {cross_corr:.4f}')
    print(f'  (1.0 = identical pattern; would make c_pion + c_ics fully degenerate)')

    # 3) pixel-wise ratio histogram
    px_ratio = pi_in / np.maximum(ic_in, 1e-30)
    print(f'\n--- pixel-wise (pi+brem)/ICS ratio inside mask ---')
    print(f'  mean   : {px_ratio.mean():.3f}')
    print(f'  median : {np.median(px_ratio):.3f}')
    print(f'  std    : {px_ratio.std():.3f}')
    print(f'  10/90%: {np.percentile(px_ratio, [10, 90])}')

    # 4) plot — 4 panel
    from matplotlib.colors import LogNorm
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    # panel A: pi+brem
    im0 = axes[0,0].imshow(np.where(mask_bool, pibr_roi, np.nan),
                            origin='lower', norm=LogNorm(), cmap='Reds')
    axes[0,0].set_title(f'(a) (π⁰+brem) inside mask, bin {eb}')
    plt.colorbar(im0, ax=axes[0,0])
    # panel B: ICS
    im1 = axes[0,1].imshow(np.where(mask_bool, ics_roi, np.nan),
                            origin='lower', norm=LogNorm(), cmap='Blues')
    axes[0,1].set_title(f'(b) ICS inside mask, bin {eb}')
    plt.colorbar(im1, ax=axes[0,1])
    # panel C: ratio (pi+brem)/ICS
    ratio_map = np.where(mask_bool, pibr_roi / np.maximum(ics_roi, 1e-30), np.nan)
    im2 = axes[1,0].imshow(ratio_map, origin='lower', cmap='RdBu_r',
                            vmin=0.5, vmax=3.0)
    axes[1,0].set_title(f'(c) (π⁰+brem)/ICS ratio (pixel-wise)')
    plt.colorbar(im2, ax=axes[1,0])
    # panel D: scatter pi+brem vs ICS
    ax3 = axes[1,1]
    ax3.scatter(ic_in, pi_in, s=0.3, alpha=0.2, color='black')
    _lo, _hi = ic_in.min(), ic_in.max()
    _x = np.linspace(_lo, _hi, 100)
    ax3.plot(_x, _x * (pi_in.sum()/ic_in.sum()), 'r-', lw=2,
              label=f'global ratio {pi_in.sum()/ic_in.sum():.2f}× ICS')
    ax3.plot(_x, _x * 1.0, 'g--', lw=1.5, alpha=0.6,
              label='1× ICS')
    ax3.plot(_x, _x * 2.53, 'm--', lw=1.5, alpha=0.6,
              label=f'paper (π+brem)/ICS ≈ 2.53')
    ax3.set_xscale('log'); ax3.set_yscale('log')
    ax3.set_xlabel('ICS pixel value'); ax3.set_ylabel('(π+brem) pixel value')
    ax3.set_title(f'(d) per-pixel correlation, r={cross_corr:.3f}')
    ax3.legend(loc='upper left', fontsize=9)
    ax3.grid(True, alpha=0.3)

    plt.suptitle(f'Spatial pattern of (π⁰+brem) vs ICS — Model I, bin {eb}', fontsize=13)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/03g_spatial_diag_bin{eb}.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'\n[saved] {out}')
    plt.show()

In [ ]:
# Cell 8h — (C2) + (C6): PSC mask impact + bin-wise ratio scan + N-S split
# Goal: locate where (pi+brem)/ICS mismatch with paper Fig 11 originates

if (E is not None) and (pion is not None) and os.path.exists(FIG11_TXT):
    from astropy.io import fits as _fits
    from astropy.wcs import WCS as _WCS
    
    # rebuild full per-bin template ROI sums (14 bins, all 600x600 ROI)
    pibr_p_full = _fits.open(f'{ANALYSIS_DIR}/GC_pion_modelI_12yr{FRONT}_clean.fits')[0].data
    bre_p_full  = _fits.open(f'{ANALYSIS_DIR}/GC_bremss_modelI_12yr{FRONT}_clean.fits')[0].data
    ics_p_full  = _fits.open(f'{ANALYSIS_DIR}/GC_ics_modelI_12yr{FRONT}_clean.fits')[0].data
    pm_full = np.load(f'{ANALYSIS_DIR}/Model/GC_mask_60x60_definitions_DR2.npy')
    dm_full = np.load(f'{ANALYSIS_DIR}/Model/GC_disk_mask_60x60_definitions.npy')

    # paper ratio per bin (interpolate Fig 11 to our E grid)
    fig11 = np.loadtxt(FIG11_TXT)
    E_p = fig11[:, 0]
    pb_p_arr = fig11[:, 1]
    ic_p_arr = fig11[:, 4]
    from scipy.interpolate import interp1d
    paper_pb_at_E  = interp1d(E_p, pb_p_arr, fill_value='extrapolate')(E)
    paper_ics_at_E = interp1d(E_p, ic_p_arr, fill_value='extrapolate')(E)
    paper_ratio_at_E = paper_pb_at_E / paper_ics_at_E
    
    # ROI [100:500, 100:500]
    n_bins = len(E)
    our_ratio_full      = np.zeros(n_bins)   # full mask (PSC + disk)
    our_ratio_disk_only = np.zeros(n_bins)   # disk only (no PSC)
    our_pb_sum  = np.zeros(n_bins)
    our_ics_sum = np.zeros(n_bins)
    
    dm = dm_full[100:500, 100:500]
    for eb in range(n_bins):
        pibr = (pibr_p_full[eb] + bre_p_full[eb])[100:500, 100:500]
        ic   = ics_p_full[eb][100:500, 100:500]
        pm   = pm_full[eb, 100:500, 100:500]
        full_mask = pm * dm
        # full mask
        m1 = (full_mask == 1)
        our_pb_sum[eb]  = pibr[m1].sum()
        our_ics_sum[eb] = ic[m1].sum()
        our_ratio_full[eb] = pibr[m1].sum() / ic[m1].sum()
        # disk only
        m2 = (dm == 1)
        our_ratio_disk_only[eb] = pibr[m2].sum() / ic[m2].sum()
    
    print(f'=== bin-wise (pi+brem)/ICS ROI ratio: ours vs paper Fig 11 ===')
    print(f'{"bin":>3} {"E[GeV]":>7} {"ours full":>10} {"ours disk":>10} '
          f'{"paper":>8} {"ratio_full/paper":>17} {"PSC effect":>11}')
    for i in range(n_bins):
        psc_effect = (our_ratio_full[i] / our_ratio_disk_only[i] - 1) * 100
        print(f'{i:>3} {E[i]:>7.3f} {our_ratio_full[i]:>10.3f} '
              f'{our_ratio_disk_only[i]:>10.3f} {paper_ratio_at_E[i]:>8.3f} '
              f'{our_ratio_full[i]/paper_ratio_at_E[i]:>17.3f} {psc_effect:>+10.1f}%')
    
    # ============================================================
    # N-S split at bin 7 (full mask)
    # ============================================================
    eb = BIN_DIAG
    ccube_p = f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits'
    ccu = _fits.open(ccube_p)
    wcs = _WCS(ccu[0].header).dropaxis(2)
    H, W = ccu[0].data.shape[1], ccu[0].data.shape[2]
    
    # build b map (Galactic latitude) for ROI pixels
    j_idx, i_idx = np.meshgrid(np.arange(W), np.arange(H))
    _, b_map = wcs.wcs_pix2world(j_idx, i_idx, 0)
    b_roi = b_map[100:500, 100:500]
    
    pibr_b7 = (pibr_p_full[eb] + bre_p_full[eb])[100:500, 100:500]
    ics_b7  = ics_p_full[eb][100:500, 100:500]
    pm_b7   = pm_full[eb, 100:500, 100:500]
    fm_b7   = pm_b7 * dm
    
    m_full = (fm_b7 == 1)
    m_N    = m_full & (b_roi > 0)
    m_S    = m_full & (b_roi < 0)
    print(f'\n=== bin 7 N-S split (full mask) ===')
    print(f'  pixels: N={m_N.sum()}, S={m_S.sum()}, total={m_full.sum()}')
    print(f'  (pi+brem)/ICS in N (b>0): {pibr_b7[m_N].sum()/ics_b7[m_N].sum():.3f}')
    print(f'  (pi+brem)/ICS in S (b<0): {pibr_b7[m_S].sum()/ics_b7[m_S].sum():.3f}')
    print(f'  (pi+brem) sum: N={pibr_b7[m_N].sum():.3e}, S={pibr_b7[m_S].sum():.3e}, '
          f'N/S={pibr_b7[m_N].sum()/pibr_b7[m_S].sum():.3f}')
    print(f'  ICS      sum: N={ics_b7[m_N].sum():.3e}, S={ics_b7[m_S].sum():.3e}, '
          f'N/S={ics_b7[m_N].sum()/ics_b7[m_S].sum():.3f}')
    print(f'  (1.00 = symmetric; deviation flags morphology asymmetry)')
    
    # ============================================================
    # plot — bin-wise ratio comparison
    # ============================================================
    fig, ax = plt.subplots(figsize=(11, 6.5))
    ax.plot(E, our_ratio_full,      marker='o', color='red',
            label='haebarg 12yr — full mask (PSC + disk)', lw=1.6, ms=8)
    ax.plot(E, our_ratio_disk_only, marker='s', color='salmon',
            label='haebarg 12yr — disk-only (no PSC mask)', lw=1.6, ms=8, ls='--')
    ax.plot(E, paper_ratio_at_E,    marker='*', color='magenta',
            label='Cholis paper Fig 11 (interpolated)', lw=2.0, ms=12)
    ax.set_xscale('log')
    ax.set_xlim(0.3, 50)
    ax.set_xlabel('E [GeV]', fontsize=13)
    ax.set_ylabel(r'(π⁰+brem) / ICS  ROI-sum ratio', fontsize=13)
    ax.set_title('Per-bin (π⁰+brem)/ICS ROI ratio — ours vs Cholis paper Fig 11', fontsize=12)
    ax.axhline(1, color='gray', alpha=0.3, lw=1)
    ax.axhline(2, color='gray', alpha=0.3, lw=1)
    ax.axhline(3, color='gray', alpha=0.3, lw=1)
    ax.grid(True, which='major', alpha=0.3)
    ax.legend(loc='best', fontsize=10, framealpha=0.93)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/03h_per_bin_ratio_comparison.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    print(f'\n[saved] {out}')
    plt.show()

In [ ]:
# Cell 8i — N-S asymmetry per component + exposure sanity check
# Goal: separate pi0 vs bremss N-S asymmetry, check ICS/GCE/bubble/iso/expcube symmetry

if (E is not None) and (pion is not None):
    eb = BIN_DIAG
    WORK = ANALYSIS_DIR
    
    # load full ROI for each component (no_convol = pre-PSF, cleaner morphology)
    paths = {
        'pi0':     f'{WORK}/GC_pion_modelI_12yr{FRONT}_clean.fits',
        'bremss':  f'{WORK}/GC_bremss_modelI_12yr{FRONT}_clean.fits',
        'ics':     f'{WORK}/GC_ics_modelI_12yr{FRONT}_clean.fits',
        'GCE':     f'{WORK}/GC_GCE_model_12yr{FRONT}_clean.fits',
        'bubble':  f'{WORK}/GC_fermi_bubble_model_12yr{FRONT}_clean.fits',
        'iso':     f'{WORK}/GC_isotropic_model_12yr{FRONT}_clean.fits',
    }
    # no_convol versions (orientation-pristine)
    paths_nc = {
        'pi0_nc':    f'{WORK}/GC_pion_modelI_12yr{FRONT}_clean_no_convol.fits',
        'bremss_nc': f'{WORK}/GC_bremss_modelI_12yr{FRONT}_clean_no_convol.fits',
        'ics_nc':    f'{WORK}/GC_ics_modelI_12yr{FRONT}_clean_no_convol.fits',
    }
    exp_p = f'{WORK}/GC_expcube_center_12yr{FRONT}_clean.fits'
    data_p = f'{WORK}/GC_ccube_12yr{FRONT}_clean.fits'
    
    # b map
    from astropy.io import fits as _fits
    from astropy.wcs import WCS as _WCS
    ccu = _fits.open(data_p)
    wcs = _WCS(ccu[0].header).dropaxis(2)
    H, W = ccu[0].data.shape[1], ccu[0].data.shape[2]
    j_idx, i_idx = np.meshgrid(np.arange(W), np.arange(H))
    _, b_map_full = wcs.wcs_pix2world(j_idx, i_idx, 0)
    b_roi = b_map_full[100:500, 100:500]
    
    # masks
    pm = np.load(f'{WORK}/Model/GC_mask_60x60_definitions_DR2.npy')[eb, 100:500, 100:500]
    dm = np.load(f'{WORK}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]
    fm = pm * dm
    
    m_full = (fm == 1)
    m_N = m_full & (b_roi > 0)
    m_S = m_full & (b_roi < 0)
    print(f'=== bin 7 N-S sum (full mask), each component separately ===')
    print(f'{"component":>12} {"N sum":>13} {"S sum":>13} {"N/S":>8} {"|asym|":>8}')
    
    for name, p in {**paths, **paths_nc}.items():
        try:
            d = _fits.open(p)[0].data[eb, 100:500, 100:500]
            n_sum = d[m_N].sum()
            s_sum = d[m_S].sum()
            ns_ratio = n_sum / s_sum if s_sum > 0 else np.nan
            asym = abs(ns_ratio - 1)
            print(f'{name:>12} {n_sum:>13.3e} {s_sum:>13.3e} {ns_ratio:>8.3f} {asym:>8.3f}')
        except FileNotFoundError:
            print(f'{name:>12} (missing)')
    
    # exposure cube N-S
    exp_cube = _fits.open(exp_p)[0].data[eb, 100:500, 100:500]
    n_exp = exp_cube[m_N].sum()
    s_exp = exp_cube[m_S].sum()
    print(f'\n  exposure       {n_exp:>13.3e} {s_exp:>13.3e} {n_exp/s_exp:>8.3f}'
          f'  ({(n_exp/s_exp-1)*100:+.1f}%)')
    
    # observed data N-S
    dat_cube = _fits.open(data_p)[0].data[eb, 100:500, 100:500]
    n_dat = dat_cube[m_N].sum()
    s_dat = dat_cube[m_S].sum()
    print(f'  observed (data) {n_dat:>12.3e} {s_dat:>13.3e} {n_dat/s_dat:>8.3f}'
          f'  ({(n_dat/s_dat-1)*100:+.1f}%)')
    
    print(f'\n--- interpretation guide ---')
    print(f'  exposure N/S deviation > 5%: gtexpcube2 anomaly, unusual')
    print(f'  ICS N/S ≈ 1: confirms what we saw earlier (ICS spatially symmetric)')
    print(f'  pi0 & bremss N/S both ≈ same large value: GALPROP run-level asymmetry')
    print(f'  pi0 N/S ≠ bremss N/S: per-component issue (one of them flipped/corrupted)')
    print(f'  observed data N/S asymmetry: tells which way galactic plane sky is asymmetric')
    
    # ============================================================
    # plot: per-component N-S split map at bin 7 (orientation-pristine = no_convol)
    # ============================================================
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    comps_to_plot = [
        ('pi0_nc',    'π⁰ (no_convol)',    'Reds'),
        ('bremss_nc', 'bremss (no_convol)', 'Oranges'),
        ('ics_nc',    'ICS (no_convol)',    'Blues'),
        ('GCE',       'GCE',                'PuRd'),
        ('bubble',    'Bubble',             'Purples'),
        ('iso',       'Isotropic',          'Greens'),
    ]
    
    from matplotlib.colors import LogNorm
    for ax, (key, title, cmap) in zip(axes.flat, comps_to_plot):
        if key in paths:
            p = paths[key]
        else:
            p = paths_nc[key]
        if not os.path.exists(p):
            ax.text(0.5, 0.5, f'(missing)\n{key}', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_axis_off()
            continue
        d = _fits.open(p)[0].data[eb, 100:500, 100:500]
        d_masked = np.where(m_full, d, np.nan)
        norm = LogNorm(vmin=max(d_masked[np.isfinite(d_masked)].min(), d_masked[np.isfinite(d_masked)].max()*1e-3),
                        vmax=d_masked[np.isfinite(d_masked)].max())
        im = ax.imshow(d_masked, origin='lower', norm=norm, cmap=cmap,
                        extent=[-20, 20, -20, 20])
        ax.axhline(0, color='black', alpha=0.7, lw=1)
        n_sum = d[m_N].sum()
        s_sum = d[m_S].sum()
        ns = n_sum / s_sum
        ax.set_title(f'{title}\nN/S = {ns:.3f}', fontsize=11)
        ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.suptitle(f'Per-component N-S asymmetry — Model I, bin {eb} (E={E[eb]:.2f} GeV)',
                  fontsize=13)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/03i_per_component_NS_split.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'\n[saved] {out}')
    plt.show()

In [ ]:
# Cell 8o — bin 7 의 (π+brem)/ICS ratio 변천: mask 단계 분리
eb = 7
from astropy.io import fits as _fits
import numpy as np

WORK = './GC_analysis_DR2'
pibr = (_fits.open(f'{WORK}/GC_pion_modelI_12yr_front_clean_no_convol.fits')[0].data[eb]
      + _fits.open(f'{WORK}/GC_bremss_modelI_12yr_front_clean_no_convol.fits')[0].data[eb])
ics  =  _fits.open(f'{WORK}/GC_ics_modelI_12yr_front_clean_no_convol.fits')[0].data[eb]
pibr_roi = pibr[100:500, 100:500]
ics_roi  = ics [100:500, 100:500]

dm = np.load(f'{WORK}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]
pm = np.load(f'{WORK}/Model/GC_mask_60x60_definitions_DR2.npy')[eb, 100:500, 100:500]

masks = [
    ('no mask',            np.ones_like(dm, dtype=bool)),
    ('disk only (|b|>2)',  dm == 1),
    ('PSC only',           pm == 1),
    ('disk + PSC (full)',  (dm * pm) == 1),
]
print(f'=== bin {eb} (π+brem)/ICS ratio under different masks ===')
print(f'{"mask":>25} {"pi+brem":>11} {"ICS":>11} {"ratio":>7} {"vs raw 1.954":>15}')
for name, m in masks:
    r = pibr_roi[m].sum() / ics_roi[m].sum()
    print(f'{name:>25} {pibr_roi[m].sum():>11.3e} {ics_roi[m].sum():>11.3e} '
          f'{r:>7.3f} {r/1.954:>15.3f}')

In [ ]:
# === Cell 8q v3 — paper c (disk mask APPLIED, paper §IV B convention) + raw ratio ===
import numpy as np
from scipy.interpolate import interp1d
from astropy.io import fits as _fits

# (1) paper Fig 11 post-fit
fig11 = np.loadtxt('./2112_09706_Fig11_ModelI_ALL.txt')
E_p   = fig11[:, 0]
paper_post = {'pi+brem': fig11[:,1], 'ICS': fig11[:,4],
              'Bub': fig11[:,7], 'Iso': fig11[:,10], 'GCE': fig11[:,13]}

# (2) paper Zenodo raw (Model I = 'bs')
RAW = '../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg'
raw_pi = _fits.open(f'{RAW}/pi0_bs_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')[0].data
raw_br = _fits.open(f'{RAW}/bremss_bs_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')[0].data
raw_ic = _fits.open(f'{RAW}/ICS_bs_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')[0].data

# (3) disk mask (|b|>=2°) + 40°×40° ROI on 240×240 grid (0.25°/pixel, center=120)
mask_b   = np.ones(240, dtype=bool); mask_b[112:128] = False    # excludes |b|<2°
disk_2d  = mask_b[:, None] * np.ones(240, dtype=bool)
disk_roi = disk_2d[40:200, 40:200]
n_unmsk  = disk_roi.sum()

def roi_disk_mean(cube):
    return cube[:, 40:200, 40:200][:, disk_roi].sum(axis=1) / n_unmsk

raw_pi_m = roi_disk_mean(raw_pi)
raw_br_m = roi_disk_mean(raw_br)
raw_ic_m = roi_disk_mean(raw_ic)

# (4) Interpolate 38-bin → paper Fig 11 grid (E_p)
E_38 = np.array([0.0438587, 0.0570013, 0.074082, 0.0962812, 0.125133, 0.162629,
                 0.211362, 0.274698, 0.357014, 0.463995, 0.603034, 0.783737,
                 1.01859, 1.32382, 1.72051, 2.23607, 2.90612, 3.77696, 4.90875,
                 6.37969, 8.2914, 10.776, 14.0051, 18.2018, 23.6561, 30.7448,
                 39.9576, 51.9312, 67.4927, 87.7174, 114.002, 148.164, 192.562,
                 250.265, 325.258, 422.724, 549.396, 714.027, 927.989])
E_38_ctr = np.sqrt(E_38[:-1] * E_38[1:])
def ip(y_38): return interp1d(E_38_ctr, y_38, fill_value='extrapolate', bounds_error=False)(E_p)

paper_raw_at_paper = {
    'pi+brem': ip(raw_pi_m) + ip(raw_br_m),
    'ICS'    : ip(raw_ic_m),
}
c_paper = {k: paper_post[k] / paper_raw_at_paper[k] for k in ['pi+brem', 'ICS']}

# (5) Cell 5 vars sanity + defensive ics re-extract
print('=== Cell 5 vars sanity ===')
for nm in ['E', 'delta_E', 'pion', 'bremss', 'ics', 'bubble', 'isotropic', 'GCE_t']:
    v = globals().get(nm)
    print(f'  {nm:<10} shape={getattr(v, "shape", None)}')

# Re-extract polluted components via Cell 5 helper
nE = len(E)
comp_flux_array = _loader_info[0]
def _safe(nm, comp_name, model_arg):
    v = globals().get(nm)
    if v is not None and v.shape == (nE,):
        return v
    print(f'  [re-extract] {nm} ← comp_flux_array("{comp_name}", {model_arg!r})')
    return comp_flux_array(comp_path(comp_name, model_arg, convol=False))

pion_v   = _safe('pion',      'pion',         SELECTED_MODEL)
bremss_v = _safe('bremss',    'bremss',       SELECTED_MODEL)
ics_v    = _safe('ics',       'ics',          SELECTED_MODEL)
bubble_v = _safe('bubble',    'fermi_bubble', None)
iso_v    = _safe('isotropic', 'isotropic',    None)
GCEt_v   = _safe('GCE_t',     'GCE',          None)

# (6) Our raw vs paper raw — direct ratio in 1-10 GeV mean
E2_dE = E**2 / delta_E
our_raw = {
    'pi+brem': (pion_v + bremss_v) * E2_dE,
    'ICS'    : ics_v               * E2_dE,
    'Bub'    : bubble_v            * E2_dE,
    'Iso'    : iso_v               * E2_dE,
    'GCE'    : GCEt_v              * E2_dE,
}
def ip_p2us(y_p): return interp1d(E_p, y_p, kind='linear', bounds_error=False, fill_value=np.nan)(E)
paper_raw_at_us = {k: ip_p2us(paper_raw_at_paper[k]) for k in ['pi+brem', 'ICS']}

m_us = (E >= 1) & (E <= 10)
m_p  = (E_p >= 1) & (E_p <= 10)

d = np.load('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')
c_us = d['fitted_params']

print('\n=== Paper c v3 (disk mask APPLIED) — 1-10 GeV mean ===')
print(f'{"comp":>10} {"c_paper":>10} {"our c":>10} {"raw_us/raw_paper":>20}')
for k, idx in [('pi+brem', 0), ('ICS', 1)]:
    cp = np.nanmean(c_paper[k][m_p])
    cu = np.nanmean(c_us[idx][m_us])
    rr = np.nanmean(our_raw[k][m_us] / paper_raw_at_us[k][m_us])
    print(f'{k:>10} {cp:>10.3f} {cu:>10.3f} {rr:>20.3f}')

print('\n=== Inferred c_paper for GCE/Bub/Iso (assuming raw_us = raw_paper) ===')
print(f'(reciprocal of cell 8e raw/dashed ratios:)')
for k, ratio_8e in [('GCE', 0.320), ('Bub', 2.965), ('Iso', 0.621)]:
    c_inferred = 1.0 / ratio_8e
    idx = {'GCE': 2, 'Bub': 3, 'Iso': 4}[k]
    cu = np.nanmean(c_us[idx][m_us])
    print(f'  {k:>4}: c_paper ≈ {c_inferred:.3f}    our c = {cu:.3f}    '
          f'{"match" if abs(c_inferred - cu) < 0.3 else "DIFF"}')

print('\n=== Conclusion ===')
print('  If raw_us/raw_paper ≈ 1.0:')
print('    → raw template paper-faithful (V37/V12d extends to ROI-mean level)')
print('    → swap (PB/ICS) is PURE fit dynamics, not template normalize')
print('    → GCE/Bub/Iso already in agreement → ranking inversion driver = PB/ICS分配')

In [ ]:
# Cell 8s — paper Cholis Zenodo Model I GCE flux 직접 비교
# paper-faithful .dat: E(GeV), E²·dPhi/dE_best, 1sigma_low, 1sigma_high (40x40 inner, masked_disk)
import numpy as np
import matplotlib.pyplot as plt

ZE = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
p_I    = np.loadtxt(f'{ZE}/GCE_ModelI_flux_Inner40x40_masked_disk.dat')
p_best = np.loadtxt(f'{ZE}/GCE_BestFitModel_flux_Inner40x40_masked_disk.dat')
print(f'paper Model I shape: {p_I.shape}, cols: E, best, lo, hi')

# 우리 12yr Model I .dat
o_I = np.loadtxt('./results_12yr/GCE_model_I_front_12yr_cholis.dat')
print(f'our   Model I shape: {o_I.shape}')

# overlay
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
ax = axes[0]
ax.fill_between(p_I[:, 0], p_I[:, 2], p_I[:, 3], alpha=0.3, color='magenta', label='paper Model I (1σ stat)')
ax.plot(p_I[:, 0], p_I[:, 1], 'D-', color='magenta', ms=7, label='paper Model I best')
ax.errorbar(o_I[:, 0], o_I[:, 1], yerr=[o_I[:, 1]-o_I[:, 3], o_I[:, 4]-o_I[:, 1]],
            fmt='o-', color='blue', ms=7, capsize=3, label='12yr Model I (this work)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.25, 60); ax.set_ylim(1e-8, 3e-6)
ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'$E^2 d\Phi/dE$ [GeV/cm²/s/sr]')
ax.set_title('paper Model I (Zenodo) vs our 12yr Model I')
ax.legend(loc='lower left'); ax.grid(True, alpha=0.3)

# ratio
ax = axes[1]
from scipy.interpolate import interp1d
p_at_us = interp1d(p_I[:, 0], p_I[:, 1], fill_value='extrapolate')(o_I[:, 0])
ratio = o_I[:, 1] / p_at_us
ax.plot(o_I[:, 0], ratio, 'o-', color='purple', ms=8, lw=2)
ax.axhline(1.0, color='gray', alpha=0.6)
ax.axhspan(0.9, 1.1, alpha=0.15, color='green', label='±10%')
ax.set_xscale('log')
ax.set_xlim(0.25, 60); ax.set_ylim(0.5, 1.6)
ax.set_xlabel('E [GeV]'); ax.set_ylabel('our / paper')
ax.set_title('GCE flux ratio: 12yr / paper')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = './GCE_12yr_results_plots/03s_paper_modelI_vs_ours.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'[saved] {out}')

# bin-by-bin 정량
print(f'\n=== bin-by-bin GCE flux comparison (12yr / paper Model I) ===')
print(f'{"bin":>3} {"E[GeV]":>7} {"paper":>11} {"ours":>11} {"ratio":>7}')
for i in range(len(o_I)):
    print(f'{i:>3} {o_I[i,0]:>7.3f} {p_at_us[i]:>11.3e} {o_I[i,1]:>11.3e} {ratio[i]:>7.3f}')

# 1-10 GeV mean
m = (o_I[:, 0] >= 1) & (o_I[:, 0] <= 10)
print(f'\n1-10 GeV mean ratio (12yr / paper Model I): {np.mean(ratio[m]):.3f}')
print(f'(메모리 #13: V49g 결론 "GCE Sang와 정합" 의 12yr-direct paper verification)')

In [ ]:
# Cell 8s2 — Zenodo overlay for best 5 GDE models (X, XV, XLVIII, XLIX, LIII)
# yerr 음수 clip + 발생 bin print (별도 진단 영역)
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

ZE = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
BEST_MODELS = ['X', 'XV', 'XLVIII', 'XLIX', 'LIII']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

print('=== yerr 음수 발생 bin (clip 적용 부분) ===')
for i, N in enumerate(BEST_MODELS):
    ax = axes_flat[i]
    p = np.loadtxt(f'{ZE}/GCE_Model{N}_flux_Inner40x40_masked_disk.dat')
    o = np.loadtxt(f'./results_12yr/GCE_model_{N}_front_12yr_cholis.dat')
    raw_lo = o[:, 1] - o[:, 3]
    raw_hi = o[:, 4] - o[:, 1]
    neg_lo = np.where(raw_lo < 0)[0].tolist()
    neg_hi = np.where(raw_hi < 0)[0].tolist()
    if neg_lo or neg_hi:
        print(f'  Model {N:>7}: lo<0 at bins {neg_lo}, hi<0 at bins {neg_hi}')
    yerr_lo = np.clip(raw_lo, 0, None)
    yerr_hi = np.clip(raw_hi, 0, None)
    ax.fill_between(p[:, 0], p[:, 2], p[:, 3], alpha=0.3, color='magenta',
                    label=f'paper Model {N} (1σ stat)')
    ax.plot(p[:, 0], p[:, 1], 'D-', color='magenta', ms=7,
            label=f'paper Model {N} best')
    ax.errorbar(o[:, 0], o[:, 1],
                yerr=[yerr_lo, yerr_hi],
                fmt='o-', color='blue', ms=7, capsize=3,
                label=f'12yr Model {N} (this work)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.25, 60); ax.set_ylim(1e-8, 3e-6)
    ax.set_xlabel('E [GeV]')
    ax.set_ylabel(r'$E^2 d\Phi/dE$ [GeV/cm²/s/sr]')
    ax.set_title(f'Model {N}')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, alpha=0.3)

axes_flat[5].axis('off')
plt.tight_layout()
out = './GCE_12yr_results_plots/03s2_paper_best5_vs_ours.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'\n[saved] {out}')

# 1-10 GeV mean ratio (검토 용)
print('\n=== 1-10 GeV mean GCE flux ratio (12yr / paper Zenodo) ===')
for N in BEST_MODELS:
    p = np.loadtxt(f'{ZE}/GCE_Model{N}_flux_Inner40x40_masked_disk.dat')
    o = np.loadtxt(f'./results_12yr/GCE_model_{N}_front_12yr_cholis.dat')
    p_at_us = interp1d(p[:, 0], p[:, 1], fill_value='extrapolate')(o[:, 0])
    ratio = o[:, 1] / p_at_us
    m = (o[:, 0] >= 1) & (o[:, 0] <= 10)
    print(f'  Model {N:>7}: {ratio[m].mean():.3f}')

In [ ]:
# Cell 8s4 — Zenodo + Sanghwan overlay for best 5 GDE models (X, XV, XLVIII, XLIX, LIII)
# 3-way: paper Zenodo band + 우리 12yr + Sanghwan 12yr. high-E (>10 GeV) 정량 추가.
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

ZE   = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
SANG = '../GCE_12yr_data/GCE_models_12yr_cholis_sanghwan'
BEST_MODELS = ['X', 'XV', 'XLVIII', 'XLIX', 'LIII']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

# Sanghwan .dat column 구조 1회 확인 (첫 model)
_s0 = np.loadtxt(f'{SANG}/GCE_model_{BEST_MODELS[0]}_12yr_cholis.dat')
print(f'=== Sanghwan .dat shape: {_s0.shape} (col 0=E, col 1=flux 가정) ===')

print('\n=== yerr 음수 발생 bin (clip 적용 부분) ===')
for i, N in enumerate(BEST_MODELS):
    ax = axes_flat[i]
    p = np.loadtxt(f'{ZE}/GCE_Model{N}_flux_Inner40x40_masked_disk.dat')
    o = np.loadtxt(f'./results_12yr/GCE_model_{N}_front_12yr_cholis.dat')
    s = np.loadtxt(f'{SANG}/GCE_model_{N}_12yr_cholis.dat')
    raw_lo = o[:, 1] - o[:, 3]
    raw_hi = o[:, 4] - o[:, 1]
    neg_lo = np.where(raw_lo < 0)[0].tolist()
    neg_hi = np.where(raw_hi < 0)[0].tolist()
    if neg_lo or neg_hi:
        print(f'  Model {N:>7}: lo<0 at bins {neg_lo}, hi<0 at bins {neg_hi}')
    yerr_lo = np.clip(raw_lo, 0, None)
    yerr_hi = np.clip(raw_hi, 0, None)

    ax.fill_between(p[:, 0], p[:, 2], p[:, 3], alpha=0.3, color='magenta',
                    label=f'paper Model {N} (1σ stat)')
    ax.plot(p[:, 0], p[:, 1], 'D-', color='magenta', ms=7,
            label=f'paper Model {N} best')
    ax.errorbar(o[:, 0], o[:, 1],
                yerr=[yerr_lo, yerr_hi],
                fmt='o-', color='blue', ms=7, capsize=3,
                label=f'12yr Model {N} (this work)')
    ax.plot(s[:, 0], s[:, 1], '^--', color='green', ms=6, alpha=0.8,
            label=f'Sanghwan 12yr Model {N}')

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.25, 60); ax.set_ylim(1e-8, 3e-6)
    ax.set_xlabel('E [GeV]')
    ax.set_ylabel(r'$E^2 d\Phi/dE$ [GeV/cm²/s/sr]')
    ax.set_title(f'Model {N}')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, alpha=0.3)

axes_flat[5].axis('off')
plt.tight_layout()
out = './GCE_12yr_results_plots/03s2_paper_best5_vs_ours_vs_sang.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'\n[saved] {out}')

# 정량 — 1-10 GeV main + high-E (>10 GeV) 영역별, 우리 & Sanghwan 둘 다 vs paper Zenodo
print('\n=== mean GCE flux ratio (vs paper Zenodo) ===')
print(f'{"model":>7} | {"우리 1-10":>10} {"Sang 1-10":>10} | '
      f'{"우리 >10":>10} {"Sang >10":>10} | {"우리/Sang >10":>13}')
for N in BEST_MODELS:
    p = np.loadtxt(f'{ZE}/GCE_Model{N}_flux_Inner40x40_masked_disk.dat')
    o = np.loadtxt(f'./results_12yr/GCE_model_{N}_front_12yr_cholis.dat')
    s = np.loadtxt(f'{SANG}/GCE_model_{N}_12yr_cholis.dat')
    p_at_o = interp1d(p[:, 0], p[:, 1], fill_value='extrapolate')(o[:, 0])
    p_at_s = interp1d(p[:, 0], p[:, 1], fill_value='extrapolate')(s[:, 0])
    s_at_o = interp1d(s[:, 0], s[:, 1], fill_value='extrapolate')(o[:, 0])
    r_o = o[:, 1] / p_at_o
    r_s = s[:, 1] / p_at_s
    r_os = o[:, 1] / s_at_o          # 우리 / Sanghwan (직접)
    m_lo_o = (o[:, 0] >= 1) & (o[:, 0] <= 10)
    m_hi_o = (o[:, 0] > 10)
    m_lo_s = (s[:, 0] >= 1) & (s[:, 0] <= 10)
    m_hi_s = (s[:, 0] > 10)
    print(f'{N:>7} | {r_o[m_lo_o].mean():>10.3f} {r_s[m_lo_s].mean():>10.3f} | '
          f'{r_o[m_hi_o].mean():>10.3f} {r_s[m_hi_s].mean():>10.3f} | '
          f'{r_os[m_hi_o].mean():>13.3f}')

# bin-별 우리/Sanghwan ratio (high-E 집중)
print('\n=== bin-별 우리/Sanghwan ratio (Model X, high-E 집중) ===')
N = 'X'
o = np.loadtxt(f'./results_12yr/GCE_model_{N}_front_12yr_cholis.dat')
s = np.loadtxt(f'{SANG}/GCE_model_{N}_12yr_cholis.dat')
s_at_o = interp1d(s[:, 0], s[:, 1], fill_value='extrapolate')(o[:, 0])
print(f'{"bin":>3} {"E[GeV]":>8} {"우리":>11} {"Sang":>11} {"우리/Sang":>10}')
for j in range(len(o)):
    print(f'{j:>3} {o[j,0]:>8.3f} {o[j,1]:>11.3e} {s_at_o[j]:>11.3e} '
          f'{o[j,1]/s_at_o[j]:>10.3f}')

In [ ]:
# Cell 8s5 — 4-way GCE flux overlay (Model I): Cholis + our 14-bin + HIS METHOD@our env + his saved v3
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

ZE       = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
RETRY    = '../12yr_retry'
SANG_OLD = '../GCE_12yr_data/GCE_models_12yr_cholis_sanghwan'

cho  = np.loadtxt(f'{ZE}/GCE_ModelI_flux_Inner40x40_masked_disk.dat')            # [E, flux, lo, hi]      14-bin
ours = np.loadtxt('./results_12yr/GCE_model_I_front_12yr_cholis.dat')            # [E, flux, _, lo, hi]   14-bin
hisE = np.loadtxt(f'{RETRY}/his_v3_run/pipeline_outputs/modelI_v3/'
                  'GCE_model_I_12yr_front_clean_v3_unbound_legacy_modelI_legacy_average_sr1.dat')   # [E,flux,err,lo,hi] 17-bin  HIS METHOD @ our env
hisS = np.loadtxt(f'{RETRY}/sanghwan_codex_v3/results_v3/'
                  'GCE_model_I_12yr_front_clean_v3_unbound_legacy_modelI_legacy_average_sr1.dat')   # his saved v3        17-bin
try:
    sold = np.loadtxt(f'{SANG_OLD}/GCE_model_I_12yr_cholis.dat')                 # old Sanghwan Gen2 (optional)
except Exception:
    sold = None

print(f'Cholis ModelI : {cho.shape}  E {cho[0,0]:.3f}-{cho[-1,0]:.1f} GeV')
print(f'our 14-bin    : {ours.shape} E {ours[0,0]:.3f}-{ours[-1,0]:.1f} GeV')
print(f'HIS @ our env : {hisE.shape} E {hisE[0,0]:.3f}-{hisE[-1,0]:.1f} GeV')
print(f'his saved v3  : {hisS.shape} E {hisS[0,0]:.3f}-{hisS[-1,0]:.1f} GeV')

def clip_err(d, c_f=1, c_lo=3, c_hi=4):
    return np.clip(d[:,c_f]-d[:,c_lo], 0, None), np.clip(d[:,c_hi]-d[:,c_f], 0, None)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

ax = axes[0]
ax.fill_between(cho[:,0], cho[:,2], cho[:,3], alpha=0.25, color='magenta', label='Cholis Model I (1$\\sigma$ stat)')
ax.plot(cho[:,0], cho[:,1], 'D-', color='magenta', ms=6, label='Cholis Model I best')
lo, hi = clip_err(ours)
ax.errorbar(ours[:,0], ours[:,1], yerr=[lo,hi], fmt='o-', color='blue', ms=6, capsize=3, label='our 14-bin reproduce')
loH, hiH = clip_err(hisE)
ax.errorbar(hisE[:,0], hisE[:,1], yerr=[loH,hiH], fmt='s-', color='darkorange', ms=7, capsize=3, lw=2, label='HIS method @ our env (17-bin)')
ax.plot(hisS[:,0], hisS[:,1], 'x--', color='black', ms=7, alpha=0.7, label='his saved v3 (17-bin)')
if sold is not None:
    ax.plot(sold[:,0], sold[:,1], '^:', color='green', ms=5, alpha=0.5, label='old Sanghwan Gen2')
ax.axvline(52, color='gray', ls=':', alpha=0.5)
ax.text(52, 1.6e-6, ' Cholis high-E edge', color='gray', fontsize=8, rotation=90, va='top')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.25, 400); ax.set_ylim(3e-9, 3e-6)
ax.set_xlabel('E [GeV]'); ax.set_ylabel(r'$E^2 d\Phi/dE$ [GeV/cm$^2$/s/sr]')
ax.set_title('Model I — GCE flux overlay (his method @ our env vs Cholis vs our 14-bin)')
ax.legend(loc='lower left', fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
cho_at = lambda E: interp1d(cho[:,0], cho[:,1], fill_value='extrapolate')(E)
def ratio_in_range(d):
    m = (d[:,0] >= cho[0,0]) & (d[:,0] <= cho[-1,0])
    return d[m,0], d[m,1]/cho_at(d[m,0])
Eo, ro = ratio_in_range(ours)
Eh, rh = ratio_in_range(hisE)
Es, rs = ratio_in_range(hisS)
ax.plot(Eo, ro, 'o-', color='blue',       ms=7, lw=2,   label='our 14-bin / Cholis')
ax.plot(Eh, rh, 's-', color='darkorange', ms=8, lw=2.2, label='HIS method @ our env / Cholis')
ax.plot(Es, rs, 'x--', color='black',      ms=7, alpha=0.7, label='his saved v3 / Cholis')
ax.axhline(1.0, color='gray', alpha=0.7)
ax.axhspan(0.9, 1.1, alpha=0.12, color='green', label='$\\pm$10%')
ax.axvspan(7, 52, alpha=0.08, color='red', label='high-E (7-52 GeV)')
ax.set_xscale('log')
ax.set_xlim(0.25, 60); ax.set_ylim(0.4, 2.0)
ax.set_xlabel('E [GeV]'); ax.set_ylabel('flux / Cholis Model I')
ax.set_title('ratio vs Cholis — high-E excess: methodological or our-pipeline-specific?')
ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
out = './GCE_12yr_results_plots/03s5_hismethod_vs_cholis_vs_ours.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'[saved] {out}')

In [ ]:
# Cell 8s6 — quantitative: reproducibility (his method @ our env vs his saved v3) + region means vs Cholis
import numpy as np
from scipy.interpolate import interp1d

ZE='../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'; RETRY='../12yr_retry'
cho  = np.loadtxt(f'{ZE}/GCE_ModelI_flux_Inner40x40_masked_disk.dat')
ours = np.loadtxt('./results_12yr/GCE_model_I_front_12yr_cholis.dat')
hisE = np.loadtxt(f'{RETRY}/his_v3_run/pipeline_outputs/modelI_v3/GCE_model_I_12yr_front_clean_v3_unbound_legacy_modelI_legacy_average_sr1.dat')
hisS = np.loadtxt(f'{RETRY}/sanghwan_codex_v3/results_v3/GCE_model_I_12yr_front_clean_v3_unbound_legacy_modelI_legacy_average_sr1.dat')

print('=== (1) reproducibility: HIS method @ our env  vs  his saved v3  (same 17-bin grid) ===')
print(f'{"bin":>3} {"E[GeV]":>8} {"@our env":>11} {"saved v3":>11} {"our/saved":>10}')
assert hisE.shape[0]==hisS.shape[0], 'bin count mismatch between the two 17-bin runs'
rep = hisE[:,1]/hisS[:,1]
for j in range(len(hisE)):
    print(f'{j:>3} {hisE[j,0]:>8.3f} {hisE[j,1]:>11.3e} {hisS[j,1]:>11.3e} {rep[j]:>10.3f}')
print(f'  -> reproduction ratio: mean={rep.mean():.3f}, max|dev|={np.max(np.abs(rep-1)):.3f}  (emcee scatter floor)')

cho_at = lambda E: interp1d(cho[:,0], cho[:,1], fill_value='extrapolate')(E)
def region_means(d, label):
    E, f = d[:,0], d[:,1]; r = f/cho_at(E); inr = (E>=cho[0,0])&(E<=cho[-1,0])
    for name, m in [('1-10 GeV',(E>=1)&(E<=10)),
                    ('7-52 GeV (high-E)',(E>=7)&(E<=52)),
                    ('>10 GeV',(E>10)&inr)]:
        if m.any():
            print(f'  {label:<22} {name:<20}: <flux/Cholis> = {r[m].mean():.3f}  (n={int(m.sum())})')
print('\n=== (2) mean flux / Cholis Model I  — the high-E excess test ===')
region_means(ours, 'our 14-bin reproduce')
region_means(hisE, 'HIS method @ our env')
region_means(hisS, 'his saved v3')
print('\nReading: HIS-method 7-52 ratio ~ our-14bin ratio (both >1)  => excess is methodological.')
print('         HIS-method ~1 while our-14bin >1                    => excess is our-14bin-pipeline-specific.')

In [ ]:
# Cell 8s7 — his-method per-bin fit coefficients (from metadata) + GCE-unbounded-prior behavior
import json, os, numpy as np
RETRY='../12yr_retry'
meta = json.load(open(f'{RETRY}/his_v3_run/pipeline_outputs/modelI_v3/'
                      'GCE_model_I_12yr_fit_metadata_v3_unbound_legacy_modelI_legacy_average_sr1.json'))
comps = meta['components']
print(f'components: {comps}')
print(f'prior={meta["gce_prior_mode"]}  constraint={meta["constraint_mode"]}  flux_mode={meta["flux_mode"]}')
print(f'\n{"bin":>3} {"E[GeV]":>8} ' + ' '.join(f'{c:>11}' for c in comps) + f' {"gce_lo":>9} {"gce_hi":>9}')
for res in meta['results']:
    p, lo, hi = res['params'], res['lower'], res['upper']
    row = ' '.join(f'{p[c]:>11.4f}' for c in comps)
    flag = '  <-- c_gce lower<0 (unbounded prior active)' if lo['gce'] < 0 else ''
    print(f'{res["energy_gev"]:>8.3f}'.rjust(12) + f' {row} {lo["gce"]:>9.3f} {hi["gce"]:>9.3f}{flag}')
print('\nGCE-only-unbounded prior: c_gce may dip <0 at weak-signal high-E bins; c_iso/others stay >=0.')

ournpz = GCE_NPZ_PATTERN.format(model='I')
if os.path.exists(ournpz):
    z = np.load(ournpz, allow_pickle=True)
    print(f'\n[our 14-bin reproduce npz] {ournpz}\n  keys: {list(z.keys())}')
    print('  (wire c_gce/c_iso arrays from these keys for a direct coefficient comparison)')
else:
    print(f'\n[our reproduce npz not found: {ournpz}]')

In [ ]:
# Cell 8s8 (FIXED) — localize the ~8% high-E residual with CORRECT flux decomposition.
# Discovery (from run_one_model): npz named keys GCE/pion/bremss/ics/bubble/isotropic are
# TEMPLATE INTENSITIES  T_k = disk-mask-avg( no_convol_map / (exposure*sr_pixel) ),
# NOT the E^2 dN/dE flux. The publication flux is  flux_k = c_k * T_k * E^2/dE,
# where c_k = fitted_params (MAP, the (5,n) array). In particular flux_gce == the .dat col 1.
import json, os, numpy as np
import matplotlib.pyplot as plt

RETRY = '../12yr_retry'
COMPS = ['pion_bremss', 'ics', 'gce', 'bubble', 'isotropic']     # fit-coeff order; GCE = idx 2

# ---------- our 14-bin reproduce (npz + .dat) ----------
z   = np.load(GCE_NPZ_PATTERN.format(model='I'), allow_pickle=True)
Eo  = np.asarray(z['E']).ravel()
dEo = np.asarray(z['delta_E']).ravel()
n   = len(Eo)
cpo = np.asarray(z['fitted_params']).reshape(5, n)               # MAP coeffs [pb,ics,gce,bub,iso]
T   = {'pion_bremss': np.asarray(z['pion']).ravel() + np.asarray(z['bremss']).ravel(),
       'ics':         np.asarray(z['ics']).ravel(),
       'gce':         np.asarray(z['GCE']).ravel(),
       'bubble':      np.asarray(z['bubble']).ravel(),
       'isotropic':   np.asarray(z['isotropic']).ravel()}
flux_o = {c: cpo[k] * T[c] * Eo**2 / dEo for k, c in enumerate(COMPS)}     # E^2 dN/dE per comp

gce_dat = np.loadtxt(GCE_DAT_PATTERN.format(model='I'))          # [E, flux, std, lo, hi]
dev = np.max(np.abs(flux_o['gce'] / gce_dat[:, 1] - 1))
print(f'[self-check] reconstructed GCE flux (c_gce*T_gce*E^2/dE) vs .dat col1: max rel-dev = {dev:.2e}')
print(f'             -> {"PASS: named key = template intensity confirmed" if dev < 1e-3 else "CHECK"}')

# ---------- his method @ our env (metadata coeffs + .dat flux) ----------
meta = json.load(open(f'{RETRY}/his_v3_run/pipeline_outputs/modelI_v3/'
                      'GCE_model_I_12yr_fit_metadata_v3_unbound_legacy_modelI_legacy_average_sr1.json'))
ch = np.array([[r['params'][c] for c in COMPS] for r in meta['results']]).T  # (5,17)
fh = np.loadtxt(f'{RETRY}/his_v3_run/pipeline_outputs/modelI_v3/'
                'GCE_model_I_12yr_front_clean_v3_unbound_legacy_modelI_legacy_average_sr1.dat')[:, 1]

# ---------- bins 0-13 align (centres match <0.5%); compare by index ----------
iref = int(np.argmin(np.abs(Eo - 1.5)))
cr   = cpo / ch[:, :n]
cr_n = cr / cr[:, iref:iref+1]                                    # per-comp coeff ratio (norm)
fr   = gce_dat[:, 1] / fh[:n]
fr_n = fr / fr[iref]                                              # GCE flux ratio from .dat (norm)

# ---------- plot ----------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
col = {'pion_bremss': 'saddlebrown', 'ics': 'teal', 'gce': 'crimson', 'bubble': 'purple', 'isotropic': 'darkorange'}
ax = axes[0]
for k, c in enumerate(COMPS):
    ax.plot(Eo, cr_n[k], 'o-', color=col[c], ms=5, label=c)
ax.axhline(1.0, color='gray', alpha=0.6); ax.axvspan(7, 52, alpha=0.08, color='red', label='high-E (7-52)')
ax.set_xscale('log'); ax.set_xlabel('E [GeV]'); ax.set_ylabel(f'coeff ratio ours/his, norm @{Eo[iref]:.1f} GeV')
ax.set_title('per-component fit-coefficient ratio (normalized) — which component gives way at high-E')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axes[1]
ax.plot(Eo, cr_n[2], 's-', color='crimson', ms=7, lw=2, label='GCE coeff ratio (norm)')
ax.plot(Eo, fr_n,    'D--', color='black',  ms=6,    label='GCE flux ratio (.dat, norm)')
ax.axhline(1.0, color='gray', alpha=0.6); ax.axvspan(7, 52, alpha=0.08, color='red', label='high-E (7-52)')
ax.set_xscale('log'); ax.set_xlabel('E [GeV]'); ax.set_ylabel(f'ratio ours/his, norm @{Eo[iref]:.1f} GeV')
ax.set_title('GCE: coefficient vs flux ratio (both reliable now; should overlap)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = './GCE_12yr_results_plots/03s8_coeff_localization_FIXED.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); print(f'[saved] {out}')

# ---------- table ----------
print(f'\n=== GCE coeff & flux ratios (ours/his), norm @ {Eo[iref]:.2f} GeV ===')
print(f'{"E[GeV]":>8} {"cGCE_o":>8} {"cGCE_h":>8} {"c_rn":>7} {"flux_rn":>8}')
for j in range(n):
    print(f'{Eo[j]:>8.3f} {cpo[2, j]:>8.4f} {ch[2, j]:>8.4f} {cr_n[2, j]:>7.3f} {fr_n[j]:>8.3f}')
m = (Eo >= 7) & (Eo <= 52)
print(f'\n7-52 GeV mean: GCE coeff ratio(norm)={cr_n[2, m].mean():.3f}, flux ratio(norm)={fr_n[m].mean():.3f}'
      f'  (match => high-E rise is FIT-LEVEL, template*binning flat)')

# ---------- our per-component E^2 dN/dE flux at high-E (CORRECT units) ----------
print('\n=== our 14-bin per-component flux E^2 dN/dE [GeV/cm^2/s/sr] (E >= 4 GeV) ===')
print(f'{"E[GeV]":>8} ' + ' '.join(f'{c:>11}' for c in COMPS) + f' {"sum_bkg":>11}')
for j in range(n):
    if Eo[j] >= 4:
        vals = [flux_o[c][j] for c in COMPS]
        sum_bkg = vals[0] + vals[1] + vals[3] + vals[4]
        print(f'{Eo[j]:>8.3f} ' + ' '.join(f'{v:>11.3e}' for v in vals) + f' {sum_bkg:>11.3e}')

In [ ]:
# Cell 8s9 (v3) — upstream-input comparison at high-E for ALL components (Model I), local staged copy.
# v2 result: data identical (1.000), ICS template flat 0.988 (gauge), disk mask identical -> none of
# these drives the +8%. v3 extends the per-bin intensity comparison to all six templates to find any
# component whose ours/his ratio is NON-FLAT in energy (a genuine high-E driver, vs a flat offset that
# the fit coefficient absorbs). For each component we report flatness = <ratio>_highE / <ratio>_lowE.
# Bins 0-13 align (centres <0.5%); compare by index. (IndexError in v2 fixed: his arrays sliced [:n].)
import os, numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS

OUR_DIR = ANALYSIS_DIR
HIS_DIR = './his_v3_products_modelI'

# component -> (our filename, his filename). pion/bremss/ics carry the Roman model; GCE/bubble/iso don't.
COMP = {
    'pion':   'GC_pion_modelI_12yr_front_clean_no_convol.fits',
    'bremss': 'GC_bremss_modelI_12yr_front_clean_no_convol.fits',
    'ics':    'GC_ics_modelI_12yr_front_clean_no_convol.fits',
    'GCE':    'GC_GCE_model_12yr_front_clean_no_convol.fits',
    'bubble': 'GC_fermi_bubble_model_12yr_front_clean_no_convol.fits',
    'iso':    'GC_isotropic_model_12yr_front_clean_no_convol.fits',
}
CCUBE_O = f'{OUR_DIR}/GC_ccube_12yr_front_clean.fits'; EXP_O = f'{OUR_DIR}/GC_expcube_center_12yr_front_clean.fits'
CCUBE_H = f'{HIS_DIR}/GC_ccube_12yr_front_clean.fits'; EXP_H = f'{HIS_DIR}/GC_expcube_center_12yr_front_clean.fits'

print('=== file presence ===')
ok = True
for c, f in COMP.items():
    po, ph = f'{OUR_DIR}/{f}', f'{HIS_DIR}/{f}'
    eo, eh = os.path.exists(po), os.path.exists(ph)
    print(f'  {c:7s} our[{"OK" if eo else "MISS"}] his[{"OK" if eh else "MISS"}]  {f}')
    ok = ok and eo and eh
for nm, p in [('our_ccube', CCUBE_O), ('our_exp', EXP_O), ('his_ccube', CCUBE_H), ('his_exp', EXP_H)]:
    e = os.path.exists(p); print(f'  {nm:9s} [{"OK" if e else "MISS"}] {p}'); ok = ok and e
assert ok, 'stage all his templates first (run the staging step)'

# --- sr/pixel + disk mask (common = OURS, since v2 showed his disk mask == ours) ---
def roi_sr(dl, db, b): return np.radians(dl) * np.radians(db) * np.cos(np.radians(b))
hdr = fits.open(CCUBE_O)[0].header; Wm = WCS(hdr).dropaxis(2)
ny, nx = fits.open(CCUBE_O)[0].data.shape[1:]
srpix = np.zeros((ny, nx))
for i in range(ny):
    for j in range(nx):
        _, b = Wm.wcs_pix2world(j, i, 0); srpix[i, j] = roi_sr(0.1, 0.1, b)
srpix = srpix[100:500, 100:500]
disk = np.load(f'{OUR_DIR}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]

def E_centers(c):
    eb = fits.open(c)[1].data; return np.array([np.sqrt(r[2]*r[1]*1e-6)*1e-3 for r in eb])
def Tavg(map_path, exp_path):
    d = fits.open(map_path)[0].data; e = fits.open(exp_path)[0].data[:, 100:500, 100:500] * srpix
    return np.array([np.sum(disk*(d[i, 100:500, 100:500]/e[i]))/np.sum(disk) for i in range(d.shape[0])])

Eo = E_centers(CCUBE_O); Eh = E_centers(CCUBE_H); n = len(Eo)
ratios = {}
for c, f in COMP.items():
    To = Tavg(f'{OUR_DIR}/{f}', EXP_O)
    Th = Tavg(f'{HIS_DIR}/{f}', EXP_H)
    ratios[c] = (To[:n] / Th[:n])
To = Tavg(CCUBE_O, EXP_O); Th = Tavg(CCUBE_H, EXP_H); ratios['data'] = To[:n] / Th[:n]
# combined pion+bremss intensity ratio (the actual fit component)
pb_o = Tavg(f'{OUR_DIR}/{COMP["pion"]}', EXP_O)[:n] + Tavg(f'{OUR_DIR}/{COMP["bremss"]}', EXP_O)[:n]
pb_h = Tavg(f'{HIS_DIR}/{COMP["pion"]}', EXP_H)[:n] + Tavg(f'{HIS_DIR}/{COMP["bremss"]}', EXP_H)[:n]
ratios['pion+brem'] = pb_o / pb_h

cols = ['pion', 'bremss', 'pion+brem', 'ics', 'GCE', 'bubble', 'iso', 'data']
print(f'\n=== per-bin intensity ratio ours/his (common disk mask = OURS) ===')
print(f'{"i":>2} {"E[GeV]":>8} ' + ' '.join(f'{c:>10}' for c in cols))
for i in range(n):
    fl = ' <-hi' if Eo[i] >= 7 else ''
    print(f'{i:>2} {Eo[i]:>8.3f} ' + ' '.join(f'{ratios[c][i]:>10.4f}' for c in cols) + fl)

# --- flatness: <ratio>_highE(7-52) / <ratio>_lowE(1-5) ; ~1 = flat (gauge), !=1 = energy-shape driver ---
lo = (Eo >= 1) & (Eo <= 5); hi = (Eo >= 7) & (Eo <= 52)
print(f'\n=== flatness = <ratio>_highE(7-52) / <ratio>_lowE(1-5)  [~1.000 = flat offset, harmless] ===')
flat = {}
for c in cols:
    flat[c] = ratios[c][hi].mean() / ratios[c][lo].mean()
    tag = '   <== NON-FLAT (energy-shape difference, +8% candidate)' if abs(flat[c]-1) > 0.01 else ''
    print(f'  {c:10s}: lowE={ratios[c][lo].mean():.4f}  highE={ratios[c][hi].mean():.4f}  flatness={flat[c]:.4f}{tag}')

# --- morphology @ 16 GeV (bin 12) for ICS (control) + any NON-FLAT component ---
suspects = ['ics'] + [c for c in ['pion', 'bremss', 'GCE', 'bubble', 'iso'] if abs(flat.get(c, 1)-1) > 0.01]
kb = 12
for c in suspects:
    f = COMP[c]
    io = fits.open(f'{OUR_DIR}/{f}')[0].data[kb, 100:500, 100:500]
    ih = fits.open(f'{HIS_DIR}/{f}')[0].data[kb, 100:500, 100:500]
    so, sh = np.sum(io*disk), np.sum(ih*disk)
    io_n = io/so if so else io; ih_n = ih/sh if sh else ih
    with np.errstate(divide='ignore', invalid='ignore'):
        rat = np.where((ih_n != 0) & (disk > 0), io_n/ih_n, np.nan)
    gm = disk > 0
    print(f'\n[{c}] morphology @ {Eo[kb]:.0f} GeV (normalized to unit disk-sum):')
    print(f'  our N/S={np.nansum((io_n*disk)[200:])/np.nansum((io_n*disk)[:200]):.3f} '
          f'L/R={np.nansum((io_n*disk)[:,200:])/np.nansum((io_n*disk)[:,:200]):.3f} | '
          f'his N/S={np.nansum((ih_n*disk)[200:])/np.nansum((ih_n*disk)[:200]):.3f} '
          f'L/R={np.nansum((ih_n*disk)[:,200:])/np.nansum((ih_n*disk)[:,:200]):.3f}')
    print(f'  morph ratio our/his over disk: median={np.nanmedian(rat[gm]):.3f} '
          f'5-95%=[{np.nanpercentile(rat[gm],5):.3f}, {np.nanpercentile(rat[gm],95):.3f}]')

# --- summary plot: all-component intensity ratios vs E ---
plt.figure(figsize=(9, 5.5))
for c in ['pion+brem', 'ics', 'GCE', 'bubble', 'iso', 'data']:
    plt.plot(Eo, ratios[c], 'o-', ms=4, label=c)
plt.axhline(1.0, color='gray', alpha=.6); plt.axvspan(7, 52, alpha=.08, color='red', label='high-E')
plt.xscale('log'); plt.xlabel('E [GeV]'); plt.ylabel('intensity ratio ours/his')
plt.title('per-component template & data intensity ratio (ours/his) — flat = harmless, sloped = driver')
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=.3); plt.tight_layout()
out = './GCE_12yr_results_plots/03s9_all_component_ratios.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); print(f'\n[saved] {out}')

In [ ]:
# Cell 8s10 — visualize the two templates with non-trivial ours/his differences:
#   bubble (morphology DIFFERS) and GCE (normalization differs ~2.18x; morphology expected identical).
# The fit is PER-ENERGY-BIN, so only the SPATIAL SHAPE within a bin matters (per-bin coeff absorbs any
# overall normalization). We normalize each map to unit disk-sum and compare shapes + latitude profiles.
import os, numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

OUR_DIR = ANALYSIS_DIR
HIS_DIR = './his_v3_products_modelI'
disk = np.load(f'{OUR_DIR}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]
kb = 12   # ~16 GeV (high-E)

files = {'bubble': 'GC_fermi_bubble_model_12yr_front_clean_no_convol.fits',
         'GCE':    'GC_GCE_model_12yr_front_clean_no_convol.fits'}

def load_norm(d, fname):
    m = fits.open(f'{d}/{fname}')[0].data[kb, 100:500, 100:500].astype(float)
    s = np.sum(m * disk)
    return m / s if s else m

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for r, (comp, fn) in enumerate(files.items()):
    o = load_norm(OUR_DIR, fn); h = load_norm(HIS_DIR, fn)
    with np.errstate(divide='ignore', invalid='ignore'):
        rat = np.where((h != 0) & (disk > 0), o / h, np.nan)
    vmax = np.nanpercentile(np.r_[o[disk > 0], h[disk > 0]], 99)
    im0 = axes[r, 0].imshow(o * disk, origin='lower', vmax=vmax); axes[r, 0].set_title(f'{comp} OURS (norm) ~16 GeV'); plt.colorbar(im0, ax=axes[r, 0], fraction=.046)
    im1 = axes[r, 1].imshow(h * disk, origin='lower', vmax=vmax); axes[r, 1].set_title(f'{comp} HIS (norm)'); plt.colorbar(im1, ax=axes[r, 1], fraction=.046)
    im2 = axes[r, 2].imshow(rat, origin='lower', vmin=0.5, vmax=1.5, cmap='RdBu_r'); axes[r, 2].set_title(f'{comp} ratio our/his'); plt.colorbar(im2, ax=axes[r, 2], fraction=.046)
    gm = disk > 0
    ns_o = np.nansum((o * disk)[200:]) / np.nansum((o * disk)[:200])
    ns_h = np.nansum((h * disk)[200:]) / np.nansum((h * disk)[:200])
    ident = (abs(np.nanmedian(rat[gm]) - 1) < 0.02) and (np.nanstd(rat[gm]) < 0.05)
    print(f'{comp:7s}: morph ratio over disk median={np.nanmedian(rat[gm]):.3f} std={np.nanstd(rat[gm]):.3f}  '
          f'N/S our={ns_o:.3f} his={ns_h:.3f}  ->  shape {"IDENTICAL" if ident else "DIFFERS"}')
plt.tight_layout(); out = './GCE_12yr_results_plots/03s10_bubble_gce_templates.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); print(f'[saved] {out}')

# latitude profiles (sum over longitude within disk), ours vs his
fig2, ax2 = plt.subplots(1, 2, figsize=(13, 4.5))
for comp, fn in files.items():
    o = load_norm(OUR_DIR, fn); h = load_norm(HIS_DIR, fn)
    axx = ax2[0] if comp == 'bubble' else ax2[1]
    axx.plot(np.nansum(o * disk, axis=1), label='ours')
    axx.plot(np.nansum(h * disk, axis=1), '--', label='his')
    axx.set_title(f'{comp} latitude profile (sum over l) ~16 GeV')
    axx.set_xlabel('row (b index; 0=bottom .. 400=top)'); axx.legend(); axx.grid(alpha=.3)
plt.tight_layout(); out2 = './GCE_12yr_results_plots/03s10_lat_profiles.png'
plt.savefig(out2, dpi=130, bbox_inches='tight'); print(f'[saved] {out2}')

In [ ]:
# ======================================================================
# Cell — best-5 GCE flux: 12.7yr re-fit vs Cholis same-model 1σ band
#   panels 0-4: per-model SED overlay (our M vs Cholis M band)
#   panel  5  : ratio our/Cholis_best (same-model), 저-E·고-E 영역 음영
#   our .dat: E, flux(maxL), std, lo(16th), hi(84th) | cholis: E, best, 1σ_lo, 1σ_hi
# ======================================================================
import os, numpy as np
import matplotlib.pyplot as plt

ZE     = globals().get('CHOLIS_REF_DIR', '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra')
_PLOTS = globals().get('PLOTS_DIR', './GCE_12yr_results_plots'); os.makedirs(_PLOTS, exist_ok=True)
BEST = ['X', 'XV', 'XLVIII', 'XLIX', 'LIII']
COL  = dict(zip(BEST, ['C0', 'C1', 'C2', 'C3', 'C4']))

def our_dat(M):
    for p in (f'./results_12yr/GCE_model_{M}_front_12yr_cholis.dat',
              f'./GCE_model_{M}_front_12yr_cholis.dat'):
        if os.path.exists(p): return np.loadtxt(p)
    return None
def chol_dat(M):
    p = f'{ZE}/GCE_Model{M}_flux_Inner40x40_masked_disk.dat'   # model-named (rank명 GCE_BestFitModel 쓰지 말 것)
    return np.loadtxt(p) if os.path.exists(p) else None

fig, axes = plt.subplots(2, 3, figsize=(18, 10)); ax = axes.flatten()

# --- panels 0-4: per-model same-model SED overlay ---
for i, M in enumerate(BEST):
    a = ax[i]; o = our_dat(M); c = chol_dat(M)
    if o is None or c is None:
        a.text(0.5, 0.5, f'{M}: file missing', ha='center', transform=a.transAxes); continue
    cE, cB, cLo, cHi = c[:, 0], c[:, 1], c[:, 2], c[:, 3]
    oE, oF, oLo, oHi = o[:, 0], o[:, 1], o[:, 3], o[:, 4]
    a.fill_between(cE, cLo, cHi, color='magenta', alpha=0.22, label=f'Cholis {M} 1σ (Fig19)')
    a.plot(cE, cB, 'D-', color='magenta', ms=5, lw=1.2, label=f'Cholis {M} best')
    a.errorbar(oE, oF, yerr=[np.clip(oF - oLo, 0, None), np.clip(oHi - oF, 0, None)],
               fmt='o-', color='navy', ms=5, lw=1.4, capsize=2.5, label='this work 12.7yr')
    a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(0.25, 60); a.set_ylim(1e-8, 1.2e-6)
    a.set_xlabel('E [GeV]'); a.set_ylabel(r'$E^2 d\Phi/dE$ [GeV cm$^{-2}$s$^{-1}$sr$^{-1}$]')
    a.set_title(f'Model {M}  (same-model)'); a.grid(True, which='both', alpha=0.25)
    a.legend(fontsize=8, loc='lower left')

# --- panel 5: ratio our/Cholis_best (same-model) for all best-5 ---
a = ax[5]
ref = chol_dat('XLIX')
if ref is not None:
    rE, rB, rLo, rHi = ref[:, 0], ref[:, 1], ref[:, 2], ref[:, 3]
    a.fill_between(rE, rLo / rB, rHi / rB, color='gray', alpha=0.25,
                   label='Cholis XLIX 1σ/best (대표 band)')
a.axhline(1.0, color='k', lw=0.9, ls='--')
a.axvspan(0.25, 1.05, color='C0', alpha=0.06)   # 저-E (bins 0-4)
a.axvspan(5.0, 60,    color='C3', alpha=0.06)    # 고-E (bins 11-13)
for M in BEST:
    o = our_dat(M); c = chol_dat(M)
    if o is None or c is None: continue
    a.plot(c[:, 0], o[:, 1] / c[:, 1], '-o', color=COL[M], ms=4, lw=1.3, label=M)
a.text(0.42, 2.45, '저-E\n(0-4)',   fontsize=8, color='C0', ha='center')
a.text(20,   2.45, '고-E\n(11-13)', fontsize=8, color='C3', ha='center')
a.set_xscale('log'); a.set_xlim(0.25, 60); a.set_ylim(0.0, 2.7)
a.set_xlabel('E [GeV]'); a.set_ylabel('our / Cholis_best (same-model)')
a.set_title('ratio: mid-E~1 (in), 저-E 모델의존, 고-E 공통초과')
a.grid(True, which='both', alpha=0.25); a.legend(fontsize=8, ncol=2, loc='upper center')

plt.suptitle('best-5 GCE flux — 12.7yr re-fit vs Cholis same-model 1σ band', y=1.005, fontsize=13)
plt.tight_layout()
out = f'{_PLOTS}/17_best5_vs_cholis_samemodel_12.7yr.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); print(f'[saved] {out}')
plt.show()

In [ ]:
# high-E 차이 origin — bin 구조 layer 추적
import numpy as np
from astropy.io import fits

print('=== 각 pipeline file 의 energy bin 수 ===')
files = {
    'MapCube (pion modelX)': './MapCubes/pion_mapcube_modelX.fits',
    'CCUBE'                : './GC_analysis_DR2/GC_ccube_12yr_front_clean.fits',
    'GCE conv map (modelX)': None,  # 아래서 glob
    'bin_definitions'      : './GC_analysis_DR2/bin_definitions.fits',
}

import glob
# GCE conv map / srcmap 후보 찾기
for pat in ['./GC_analysis_DR2/GC_*GCE*modelX*clean*.fits',
            './GC_analysis_DR2/GC_*modelX*srcmap*.fits',
            './GC_analysis_DR2/GC_Extended_srcmap*model_X*.fits']:
    hits = glob.glob(pat)
    for h in hits:
        files[f'srcmap/conv: {h.split("/")[-1]}'] = h

for label, path in files.items():
    if path is None:
        continue
    try:
        with fits.open(path) as hdul:
            shapes = []
            for hdu in hdul:
                if hdu.data is not None and hasattr(hdu.data, 'shape'):
                    shapes.append(f'{hdu.name}:{hdu.data.shape}')
                # energy axis table
                if hdu.data is not None and getattr(hdu.data, 'dtype', None) is not None \
                   and hdu.data.dtype.names and 'Energy' in (hdu.data.dtype.names or ()):
                    ne = len(hdu.data['Energy'])
                    shapes.append(f'[{hdu.name} Energy: {ne} entries]')
            print(f'  {label:<45}: {shapes}')
    except Exception as e:
        print(f'  {label:<45}: ERROR {e}')

# CCUBE 의 energy axis 직접
print('\n=== CCUBE energy axis ===')
with fits.open('./GC_analysis_DR2/GC_ccube_12yr_front_clean.fits') as h:
    print(f'  primary data shape: {h[0].data.shape}  (axis 0 = energy)')
    for hdu in h:
        if hdu.data is not None and getattr(hdu.data,'dtype',None) is not None \
           and hdu.data.dtype.names:
            print(f'  HDU {hdu.name}: cols={hdu.data.dtype.names}, nrows={len(hdu.data)}')

In [ ]:
# 38-bin MapCube → 15-bin srcmap 변환의 high-E 적분 방식 역산
import numpy as np
from astropy.io import fits

# 1) MapCube native 38-bin energy + data (pion, model X)
with fits.open('./MapCubes/pion_mapcube_modelX.fits') as h:
    mc_data = h[0].data                      # (38, 240, 240)
    mc_E = h['ENERGIES'].data['Energy']      # MeV, 38 entries
print(f'MapCube: {mc_data.shape}, E[MeV] {mc_E[0]:.1f}~{mc_E[-1]:.1f}')

# 2) srcmap pion component 15-bin + EBOUNDS
sp = './GC_analysis_DR2/GC_Extended_srcmap_12yr_front_clean_model_X.fits'
with fits.open(sp) as h:
    sm_pion = h['pion'].data                 # (15, 600, 600)
    eb = h['EBOUNDS'].data
    e_min = eb['E_MIN']; e_max = eb['E_MAX'] # keV (CCUBE 단위)
print(f'srcmap pion: {sm_pion.shape}, EBOUNDS {len(eb)} bins')
print(f'EBOUNDS E_min[0]={e_min[0]:.1f} keV, E_max[-1]={e_max[-1]:.1f} keV')

# 3) analysis 14-bin edges (GeV) + bin-center
edges_GeV = np.concatenate([e_min, [e_max[-1]]]) * 1e-6
ctr_GeV = np.sqrt(edges_GeV[:-1] * edges_GeV[1:])
print(f'\nanalysis 14-bin: edges {edges_GeV[0]:.4f}~{edges_GeV[-1]:.4f} GeV')

# 4) 각 analysis bin 이 native 38-bin 의 어느 index 를 cover 하는지
mc_E_GeV = mc_E * 1e-3
print(f'\n=== analysis bin → native 38-bin index mapping ===')
print(f'{"bin":>3} {"E_lo":>8} {"E_hi":>8} {"native idx":>20} {"n_native":>9}')
for b in range(14):
    lo, hi = edges_GeV[b], edges_GeV[b+1]
    idx = np.where((mc_E_GeV >= lo) & (mc_E_GeV < hi))[0]
    print(f'{b:>3} {lo:>8.3f} {hi:>8.3f} {str(idx.tolist()):>20} {len(idx):>9}')

# 5) srcmap component 의 bin 11-13 spatial sum (ROI 합) vs native MapCube 적분
#    (단위 다름 — srcmap=counts, MapCube=flux. 비교는 bin간 ratio shape)
print(f'\n=== srcmap pion bin sum (spatial total, 15-bin) ===')
sm_sum = sm_pion.reshape(15, -1).sum(axis=1)
for j in range(15):
    print(f'  srcmap bin {j:>2}: sum={sm_sum[j]:.4e}')

# native 38-bin spatial sum
mc_sum = mc_data.reshape(38, -1).sum(axis=1)
print(f'\n=== native bin 11-13 영역의 38-bin index + sum ===')
for b in [11, 12, 13]:
    lo, hi = edges_GeV[b], edges_GeV[b+1]
    idx = np.where((mc_E_GeV >= lo) & (mc_E_GeV < hi))[0]
    print(f'  analysis bin {b} ({lo:.2f}-{hi:.2f} GeV): '
          f'native idx {idx.tolist()}, native sums {[f"{mc_sum[k]:.3e}" for k in idx]}')

In [ ]:
# bin 11-13 의 gtsrcmaps 합산 방식 역산
# srcmap bin sum vs native 3-bin 의 여러 합산 가정 비교
import numpy as np
from astropy.io import fits

with fits.open('./MapCubes/pion_mapcube_modelX.fits') as h:
    mc_data = h[0].data
    mc_E = h['ENERGIES'].data['Energy'] * 1e-3   # GeV

sp = './GC_analysis_DR2/GC_Extended_srcmap_12yr_front_clean_model_X.fits'
with fits.open(sp) as h:
    sm_pion = h['pion'].data
    eb = h['EBOUNDS'].data
edges = np.concatenate([eb['E_MIN'], [eb['E_MAX'][-1]]]) * 1e-6  # GeV

mc_sum = mc_data.reshape(38, -1).sum(axis=1)   # native flux sum per bin
sm_sum = sm_pion.reshape(15, -1).sum(axis=1)   # srcmap counts sum per bin

# srcmap 은 counts, MapCube 는 flux → 절대 비교 불가.
# 대신 "bin 11-13 영역의 spectral shape 가 srcmap 에서 어떻게 보존되는지" 를
# bin 10 (native 1-bin, 1:1) 대비 ratio 로 정규화하여 비교.

print('=== native MapCube: analysis bin 별 합산 가정들 (GeV/cm²/s/sr 적분) ===')
print(f'{"bin":>3} {"E_lo-E_hi":>14} {"native_sum":>12} {"trapz_dE":>12} {"geom_ctr":>12}')
for b in range(11, 14):
    lo, hi = edges[b], edges[b+1]
    idx = np.where((mc_E >= lo) & (mc_E < hi))[0]
    Es = mc_E[idx]
    vals = mc_sum[idx]                                  # E²dN/dE per native bin
    # (a) 단순 합
    simple = vals.sum()
    # (b) trapezoidal in E (실제 flux integral ∝ ∫dN/dE dE)
    dNdE = vals / Es**2                                 # E²dN/dE → dN/dE
    trapz = np.trapz(dNdE, Es)                          # ∫dN/dE dE = photon flux
    # (c) bin-center geometric mean 1점 (gtsrcmaps 가 이렇게 하면 over)
    ctr = np.sqrt(lo*hi)
    geom = np.interp(ctr, mc_E, mc_sum)
    print(f'{b:>3} {lo:>6.2f}-{hi:>6.2f} {simple:>12.4e} {trapz:>12.4e} {geom:>12.4e}')

print('\n=== srcmap counts: bin 10(1:1) 대비 bin 11-13 정규화 ===')
# bin 10 srcmap idx (analysis 14-bin → srcmap 15-bin: 0-13 대응 가정)
for b in [10, 11, 12, 13]:
    print(f'  srcmap bin {b}: {sm_sum[b]:.4e}  (ratio to bin10: {sm_sum[b]/sm_sum[10]:.4f})')

# paper Zenodo 의 GDE high-E shape 와 비교 (post-fit, 우리 .dat 가 아닌 GDE 직접)
# → GDE template 단계 비교라 paper Zenodo GCE .dat 와는 다른 quantity.
# 대신 우리 .dat 의 PB+ICS high-E shape 가 over 인지 (GCE 가 아닌 GDE over 확인)
print('\n=== 우리 .dat 의 component 별 high-E ratio (vs Sanghwan) — GCE vs GDE 분리 ===')
print('  (이미 메모리: GCE c 흡수. GDE(PB/ICS) 가 over 면 grouping origin 확정)')

In [ ]:
# gtsrcmaps MapCubeFunction 적분 vs paper native-group 직접 계산
import numpy as np
from astropy.io import fits

# MapCube native 38-bin (pion, model X) + Zenodo README 의 native edges
with fits.open('./MapCubes/pion_mapcube_modelX.fits') as h:
    mc = h[0].data                          # (38, 240, 240), flux E²dN/dE
    mc_Ectr = h['ENERGIES'].data['Energy']  # MeV, geometric centers

# Zenodo README native edges (MeV) — 38 bins
native_edges = np.array([
    43.8587, 57.0013, 74.082, 96.2812, 125.133, 162.629, 211.362, 274.698,
    357.014, 463.995, 603.034, 783.737, 1018.59, 1323.82, 1720.51, 2236.07,
    2906.12, 3776.96, 4908.75, 6379.69, 8291.4, 10776.0, 14005.1, 18201.8,
    23656.1, 30744.8, 39957.6, 51931.2, 67492.7, 87717.4, 114002.0, 148164.0,
    192562.0, 250265.0, 325258.0, 422724.0, 549396.0, 714027.0, 927989.0])  # 39 edges

# analysis 14-bin edges (MeV)
ana_edges = np.array([274.698, 357.014, 463.995, 603.034, 783.737, 1018.59,
                      1323.82, 1720.51, 2236.07, 2906.12, 3776.96, 4908.75,
                      10776.0, 23700.0, 51931.2])  # 15 edges (bin 11-13 wide)

# 각 native bin 의 flux integral: ∫dN/dE dE = (E²dN/dE / E²) 적분
# E²dN/dE = mc (per native bin, at center). dN/dE = mc / Ectr².
# 한 native bin 의 photon flux = dN/dE × ΔE_native
mc_spatial_sum = mc.reshape(38, -1).sum(axis=1)     # ROI-total E²dN/dE per native bin
native_dE = np.diff(native_edges)                    # MeV, 38 widths
dNdE_native = mc_spatial_sum / mc_Ectr**2            # ROI-total dN/dE per native bin
photon_flux_native = dNdE_native * native_dE         # ROI-total photon flux per native bin

print('=== paper native-group 방식: analysis bin 11-13 = native 3-bin 묶기 ===')
print(f'{"ana_bin":>7} {"native idx":>16} {"Σ photon_flux":>15} {"⟨E²dN/dE⟩_group":>18}')
for b in range(11, 14):
    lo, hi = ana_edges[b], ana_edges[b+1]
    idx = np.where((mc_Ectr >= lo) & (mc_Ectr < hi))[0]
    # paper group: native 3-bin 의 photon flux 합 → analysis bin 의 평균 E²dN/dE 복원
    grouped_photon_flux = photon_flux_native[idx].sum()
    ana_dE = hi - lo
    ana_Ectr = np.sqrt(lo*hi)
    # analysis bin 의 E²dN/dE = (photon flux / ΔE) × E²
    grouped_E2dNdE = (grouped_photon_flux / ana_dE) * ana_Ectr**2
    print(f'{b:>7} {str(idx.tolist()):>16} {grouped_photon_flux:>15.4e} {grouped_E2dNdE:>18.4e}')

# gtsrcmaps 가 만든 srcmap 의 bin 11-13 (counts) — 이미 적분된 결과
sp = './GC_analysis_DR2/GC_Extended_srcmap_12yr_front_clean_model_X.fits'
with fits.open(sp) as h:
    sm = h['pion'].data.reshape(15, -1).sum(axis=1)
print(f'\n=== gtsrcmaps srcmap counts (bin 11-13) ===')
for b in [11, 12, 13]:
    print(f'  srcmap bin {b}: {sm[b]:.4e}')

# 핵심 비교: bin 10 (1:1 native) 기준으로 정규화한 bin11-13 spectral shape
# paper-group 방식 vs gtsrcmaps 방식의 bin10 대비 비율 차이
print(f'\n=== bin 10 대비 spectral shape (paper-group vs gtsrcmaps) ===')
# bin 10 paper-group (native 1-bin)
b10_lo, b10_hi = ana_edges[10], ana_edges[11]
i10 = np.where((mc_Ectr >= b10_lo) & (mc_Ectr < b10_hi))[0]
pf10 = photon_flux_native[i10].sum()
print(f'  {"bin":>3} {"paper-group/bin10":>18} {"gtsrcmaps/bin10":>16} {"ratio(pg/gt)":>13}')
for b in [11, 12, 13]:
    lo, hi = ana_edges[b], ana_edges[b+1]
    idx = np.where((mc_Ectr >= lo) & (mc_Ectr < hi))[0]
    pf = photon_flux_native[idx].sum()
    pg_norm = pf / pf10
    gt_norm = sm[b] / sm[10]
    print(f'  {b:>3} {pg_norm:>18.4f} {gt_norm:>16.4f} {pg_norm/gt_norm:>13.4f}')

In [ ]:
# wide-bin E assignment 확인 — 우리 .dat E vs paper Zenodo E vs Table III/IV E
import numpy as np
ZE = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
o = np.loadtxt('./results_12yr/GCE_model_X_front_12yr_cholis.dat')
p = np.loadtxt(f'{ZE}/GCE_ModelX_flux_Inner40x40_masked_disk.dat')

ana_edges = np.array([0.274698,0.357014,0.463995,0.603034,0.783737,1.01859,
                      1.32382,1.72051,2.23607,2.90612,3.77696,4.90875,
                      10.776,23.700,51.9312])
geom = np.sqrt(ana_edges[:-1]*ana_edges[1:])

print(f'{"bin":>3} {"우리E":>9} {"paperE":>9} {"geom√":>9} {"E_diff%":>8} {"E²비(p/o)":>10}')
for b in range(14):
    eo, ep = o[b,0], p[b,0]
    print(f'{b:>3} {eo:>9.4f} {ep:>9.4f} {geom[b]:>9.4f} '
          f'{(eo-ep)/ep*100:>+8.2f} {(ep/eo)**2:>10.4f}')

# bin 11-13 에서 paper flux 를 우리 E 로 재평가하면 ratio 변하나
print('\n=== bin 11-13: E 정의 차이가 flux ratio 에 주는 영향 ===')
for b in [11,12,13]:
    eo, ep = o[b,0], p[b,0]
    fo, fp = o[b,1], p[b,1]
    print(f'bin {b}: 우리/paper flux = {fo/fp:.3f}, '
          f'E² 보정시 = {fo/fp * (eo/ep)**2:.3f}  (우리E={eo:.2f}, paperE={ep:.2f})')

In [ ]:
# exposure cube high-E 거동 — bin 11-13 이 native 3-bin 영역에서 정상인가
import numpy as np
from astropy.io import fits

with fits.open('./GC_analysis_DR2/GC_expcube_center_12yr_front_clean.fits') as h:
    exp = h['PRIMARY'].data            # (14,600,600) cm²·s
    exp_E = h['ENERGIES'].data['Energy']  # MeV

roi = slice(100,500)
exp_roi = exp[:, roi, roi].mean(axis=(1,2))   # 14, mean exposure per bin

print(f'{"bin":>3} {"E[GeV]":>9} {"exposure":>12} {"exp/exp[10]":>12} {"bin width비":>12}')
ana_edges = np.array([0.274698,0.357014,0.463995,0.603034,0.783737,1.01859,
                      1.32382,1.72051,2.23607,2.90612,3.77696,4.90875,
                      10.776,23.700,51.9312])
dE = np.diff(ana_edges)
for b in range(14):
    print(f'{b:>3} {exp_E[b]*1e-3:>9.3f} {exp_roi[b]:>12.4e} '
          f'{exp_roi[b]/exp_roi[10]:>12.4f} {dE[b]/dE[10]:>12.4f}')

# 핵심: exposure 가 bin width 에 비례해 커지는지 (wide bin 이면 exposure 도 커야 정상)
# bin 11-13 은 native 3-bin width → exposure 도 ~3× 커야 정상.
# 만약 exposure 가 width 만큼 안 커지면 flux=counts/exp 에서 over.
print('\n=== bin 11-13: exposure 가 wide bin 을 반영하나 ===')
for b in [11,12,13]:
    print(f'bin {b}: exp/exp[10]={exp_roi[b]/exp_roi[10]:.3f}, '
          f'width/width[10]={dE[b]/dE[10]:.3f}, '
          f'ratio={(exp_roi[b]/exp_roi[10])/(dE[b]/dE[10]):.3f}')

In [ ]:
# pregroup14 가 conv map 을 실제로 바꿨나 (gtsrcmaps 덮어쓰기 여부)
import numpy as np
from astropy.io import fits

roi = slice(100,500)
for comp in ['pion','bremss','ics']:
    f0 = f'./GC_analysis_DR2/GC_{comp}_modelX_12yr_front_clean.fits'
    f1 = f'./GC_analysis_DR2/GC_{comp}_modelX_pregroup14_12yr_front_clean.fits'
    d0 = fits.open(f0)[0].data[:, roi, roi].sum(axis=(1,2))
    d1 = fits.open(f1)[0].data[:, roi, roi].sum(axis=(1,2))
    print(f'\n=== {comp} conv map (기존 vs pregroup14) ROI sum ===')
    print(f'{"bin":>3} {"기존":>11} {"pregroup14":>11} {"ratio":>8}')
    for b in [10,11,12,13]:
        print(f'{b:>3} {d0[b]:>11.4e} {d1[b]:>11.4e} {d1[b]/d0[b]:>8.4f}')

In [ ]:
import numpy as np, glob, os

MODEL = 'X'

# ---------- 1. 우리 best-fit post-fit GCE flux (.dat) ----------
our_dat_path = f'./results_12yr/GCE_model_{MODEL}_front_12yr_cholis.dat'
g = np.loadtxt(our_dat_path)
E, ours, our_std = g[:, 0], g[:, 1], g[:, 2]   # 5-col: E, flux, std, lo, hi
n = len(E)

# ---------- 2. paper Zenodo Model X .dat (4-col: E, best, lo, hi) ----------
zen_dir = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
zen_cands = [
    f'{zen_dir}/GCE_Model_{MODEL}_flux_Inner40x40_masked_disk.dat',
    f'{zen_dir}/GCE_Model{MODEL}_flux_Inner40x40_masked_disk.dat',
    f'{zen_dir}/GCE_BestFitModel_flux_Inner40x40_masked_disk.dat',
]
zen_path = next((p for p in zen_cands if os.path.exists(p)), None)
if zen_path is None:
    hits = glob.glob(f'{zen_dir}/GCE_*flux_Inner40x40_masked_disk.dat')
    print('[paper .dat 자동탐색 실패] 후보 목록 — 아래에서 Model X(best-fit) 파일 확인:')
    for h in sorted(hits):
        print('   ', h)
    paper = None
else:
    pg = np.loadtxt(zen_path)
    paper = pg[:len(E), 1]
    print(f'[paper .dat] {zen_path}')

# ---------- 3. fit.npz: fitted_params (5,14) + GCE 행 자동 식별 ----------
npz_cands = glob.glob(f'./results_12yr/GCE_model_{MODEL}_front_12yr_cholis_fit.npz') \
          + glob.glob(f'./GCE_model_{MODEL}_front_12yr_cholis_fit.npz')
npz_path = npz_cands[0] if npz_cands else None
fp = None
if npz_path:
    z = np.load(npz_path)
    print(f'\n[fit.npz] {npz_path}  keys={list(z.keys())}')
    fp = z['fitted_params']            # 기대 shape (5,14)
    print('fitted_params shape =', fp.shape)
    # GCE 행 자동 식별: ours = c_row * template * E^2/dE 이므로
    # template = ours / (c_row * E^2/dE) 가 가장 "평탄(energy-independent)"한 행이 GCE
    # delta_E 가 없으면 E^2/dE 공통인자는 행 식별에 무관 → ours/c_row 의 변동계수 최소 행을 채택
    cv = []
    for k in range(fp.shape[0]):
        c = fp[k]
        with np.errstate(divide='ignore', invalid='ignore'):
            t = np.where(c > 0, ours / c, np.nan)
        cv.append(np.nanstd(t) / np.nanmean(t) if np.nanmean(t) else np.inf)
    gce_row = int(np.nanargmin(cv))
    print(f'GCE 행 자동식별 = index {gce_row} (변동계수 {cv[gce_row]:.3f}); '
          f'메모리 표기는 index 2')
else:
    gce_row = 2
    print('\n[fit.npz 없음] GCE 행 = index 2 (메모리값)로 가정, c 출력 생략')

# ---------- 출력 1: 확인 1 (post-fit flux 기울기) ----------
print('\n=== 확인 1: post-fit GCE flux (bin 10-13) ===')
print(f'{"bin":>3} {"E[GeV]":>9} {"ours":>11} {"paper":>11} {"ratio":>7}')
for j in range(10, n):
    pv = paper[j] if paper is not None else np.nan
    r = ours[j] / pv if (paper is not None and pv) else np.nan
    print(f'{j:>3} {E[j]:>9.3f} {ours[j]:>11.3e} {pv:>11.3e} {r:>7.3f}')

def logslope(E_, F_, i0, i1):  # bin i0→i1 의 log-log 평균 기울기
    m = (F_ > 0)
    if not (m[i0] and m[i1]):
        return np.nan
    return (np.log(F_[i1]) - np.log(F_[i0])) / (np.log(E[i1]) - np.log(E[i0]))

print(f'\nbin10→13 log-slope  ours = {logslope(E, ours, 10, n-1):+.3f}')
if paper is not None:
    print(f'bin10→13 log-slope paper = {logslope(E, paper, 10, n-1):+.3f}'
          '   (paper가 더 음수면 paper가 더 가파름 = 우리 over)')

# ---------- 출력 2: 확인 3 + 테스트 B 선행 (5 component high-E c) ----------
if fp is not None:
    print('\n=== 확인 3 + 테스트 B: fitted_params 5 component, high-E bins ===')
    hdr = '  '.join(f'b{j}' for j in range(10, n))
    print(f'{"row":>3}(role)        ' + hdr)
    for k in range(fp.shape[0]):
        role = 'GCE' if k == gce_row else ''
        vals = '  '.join(f'{fp[k, j]:.3g}' for j in range(10, n))
        print(f'{k:>3}{role:>6}        {vals}')
    print(f'\nc_GCE 전체 14-bin (row {gce_row}):')
    print('  ' + '  '.join(f'{fp[gce_row, j]:.3g}' for j in range(n)))

In [ ]:
import numpy as np, glob, os

n_expected = 14
# ---------- Model I fit.npz / .dat 탐색 ----------
npz_I = (glob.glob('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')
         + glob.glob('./GCE_model_I_front_12yr_cholis_fit.npz'))
dat_I = (glob.glob('./results_12yr/GCE_model_I_front_12yr_cholis.dat')
         + glob.glob('./GCE_model_I_front_12yr_cholis.dat'))
assert npz_I and dat_I, f'Model I 파일 못 찾음: npz={npz_I} dat={dat_I}'
z = np.load(npz_I[0]); print('[fit.npz]', npz_I[0])
E, dE = z['E'], z['delta_E']
n = len(E)
gce_dat = np.loadtxt(dat_I[0])[:n, 1]

# ---------- 키가 post-fit flux인지 확인 ----------
ratio_check = z['GCE'][:3] / gce_dat[:3]
print('z[GCE]/dat_GCE (앞3) =', np.round(ratio_check, 4),
      '→ ~1이면 키=post-fit flux (아래 그대로 사용), 아니면 매핑 재검토 필요')

# ---------- 우리 5 component post-fit flux ----------
ours = {'gas': z['pion'] + z['bremss'], 'ICS': z['ics'],
        'bub': z['bubble'], 'iso': z['isotropic'], 'GCE': z['GCE']}

# ---------- Cholis Fig 11 Model I digitize → 14-bin (log-log interp) ----------
fig_cands = (glob.glob('2112_09706_Fig11_ModelI_ALL.txt')
             + glob.glob('./*/2112_09706_Fig11_ModelI_ALL.txt')
             + glob.glob('../**/2112_09706_Fig11_ModelI_ALL.txt'))
assert fig_cands, 'Fig11 digitize 파일 못 찾음 — 경로 지정 필요'
fig = np.loadtxt(fig_cands[0]); print('[Fig11]', fig_cands[0])
Ef = fig[:, 0]
def li(col):  # log-log 보간
    return 10 ** np.interp(np.log10(E), np.log10(Ef), np.log10(fig[:, col]))
cholis = {'gas': li(1), 'ICS': li(4), 'bub': li(7), 'iso': li(10), 'GCE': li(13)}

# ---------- 잔차 표 ----------
order = ['gas', 'ICS', 'bub', 'iso', 'GCE']
D = {c: ours[c] - cholis[c] for c in order}
print('\n=== Δ = ours − cholis (post-fit flux, GeV/cm²/s/sr) ===')
print(f'{"bin":>3}{"E":>7} | ' + ' '.join(f'{c:>10}' for c in order)
      + f' | {"GCE/cholis":>10}')
for j in range(n):
    row = ' '.join(f'{D[c][j]:>+10.2e}' for c in order)
    print(f'{j:>3}{E[j]:>7.2f} | {row} | {ours["GCE"][j]/cholis["GCE"][j]:>10.3f}')

# ---------- anti-correlation 진단 ----------
bi = D['bub'] + D['iso']
print('\n=== 사용자 가설 (GCE ↔ bubble+iso anti-corr) ===')
print(f'corr(Δ_GCE, Δ_bub+Δ_iso)      = {np.corrcoef(D["GCE"], bi)[0,1]:+.3f}'
      '   (음수 = anti-corr = 가설 지지)')
print(f'corr(Δ_GCE, Σ_others 4개)      = '
      f'{np.corrcoef(D["GCE"], D["gas"]+D["ICS"]+bi)[0,1]:+.3f}')
print(f'부호 anti-match (GCE vs bub+iso) = '
      f'{np.mean(np.sign(D["GCE"])==-np.sign(bi))*100:.0f}%')
# high-E (bin 11-13) 집중 확인
he = slice(11, n)
print(f'\nhigh-E(bin11-13) Δ_GCE   = {D["GCE"][he]}')
print(f'high-E(bin11-13) Δ_bub   = {D["bub"][he]}')
print(f'high-E(bin11-13) Δ_iso   = {D["iso"][he]}')
print(f'high-E(bin11-13) Δ_gas   = {D["gas"][he]}')
print(f'high-E(bin11-13) Δ_ICS   = {D["ICS"][he]}')

In [ ]:
import numpy as np, glob

# ---------- Model I fit.npz ----------
npz_I = (glob.glob('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')
         + glob.glob('./GCE_model_I_front_12yr_cholis_fit.npz'))
dat_I = (glob.glob('./results_12yr/GCE_model_I_front_12yr_cholis.dat')
         + glob.glob('./GCE_model_I_front_12yr_cholis.dat'))
assert npz_I and dat_I
z = np.load(npz_I[0]); print('[fit.npz]', npz_I[0])
E, dE = z['E'], z['delta_E']; n = len(E); E2dE = E**2 / dE
fp = z['fitted_params']          # 베이스 확정 순서 [gas, ics, GCE, bubble, iso]
crow = {'gas': 0, 'ICS': 1, 'GCE': 2, 'bub': 3, 'iso': 4}
keys = {'gas': z['pion'] + z['bremss'], 'ICS': z['ics'], 'GCE': z['GCE'],
        'bub': z['bubble'], 'iso': z['isotropic']}

# ---------- 올바른 post-fit flux = c × key × E²/ΔE ----------
ours = {c: fp[crow[c]] * keys[c] * E2dE for c in keys}

# ---------- SANITY: ours[GCE] 가 .dat 과 일치해야 공식이 맞다 ----------
gce_dat = np.loadtxt(dat_I[0])[:n, 1]
sane = ours['GCE'] / gce_dat
print('SANITY ours[GCE]/dat_GCE =', np.round(sane, 4))
assert np.allclose(sane, 1.0, rtol=0.05), '공식 불일치 — 행 순서/E²ΔE 재점검 필요, 중단'
print('  → 공식 OK, 5-component post-fit flux 신뢰 가능\n')

# ---------- Cholis Fig 11 Model I digitize → 14-bin (log-log interp) ----------
fig_cands = (glob.glob('2112_09706_Fig11_ModelI_ALL.txt')
             + glob.glob('./*/2112_09706_Fig11_ModelI_ALL.txt'))
assert fig_cands, 'Fig11 digitize 파일 경로 지정 필요'
fig = np.loadtxt(fig_cands[0]); Ef = fig[:, 0]
li = lambda col: 10 ** np.interp(np.log10(E), np.log10(Ef), np.log10(fig[:, col]))
cholis = {'gas': li(1), 'ICS': li(4), 'bub': li(7), 'iso': li(10), 'GCE': li(13)}

order = ['gas', 'ICS', 'bub', 'iso', 'GCE']
# ---------- ① ratio (정합도, 1이 목표) ----------
print('=== ours / cholis (1 = 정합) ===')
print(f'{"bin":>3}{"E":>7} | ' + ' '.join(f'{c:>8}' for c in order))
for j in range(n):
    print(f'{j:>3}{E[j]:>7.2f} | '
          + ' '.join(f'{ours[c][j]/cholis[c][j]:>8.3f}' for c in order))

# ---------- ② Δ = ours − cholis (보상 관계, 같은 flux 단위) ----------
D = {c: ours[c] - cholis[c] for c in order}
print('\n=== Δ = ours − cholis (GeV/cm²/s/sr) ===')
print(f'{"bin":>3}{"E":>7} | ' + ' '.join(f'{c:>10}' for c in order))
for j in range(n):
    print(f'{j:>3}{E[j]:>7.2f} | ' + ' '.join(f'{D[c][j]:>+10.2e}' for c in order))

# ---------- ③ 보상 관계 진단 ----------
bi = D['bub'] + D['iso']
print('\n=== 사용자 가설: GCE over ↔ (bubble+iso) under ===')
print(f'corr(Δ_GCE, Δ_bub+Δ_iso)   = {np.corrcoef(D["GCE"], bi)[0,1]:+.3f} (음수=anti=지지)')
print(f'corr(Δ_GCE, Σ others 4개)   = '
      f'{np.corrcoef(D["GCE"], D["gas"]+D["ICS"]+bi)[0,1]:+.3f}')
print(f'부호 anti-match (GCE vs bub+iso) = '
      f'{np.mean(np.sign(D["GCE"])==-np.sign(bi))*100:.0f}%')
print('\n전 component 합 보존 체크 ΣΔ (작아야 = data 보존):')
print('  ' + '  '.join(f'b{j}:{sum(D[c][j] for c in order):+.1e}' for j in range(10, n)))

In [ ]:
import glob, numpy as np
from astropy.io import fits

WORK  = './GC_analysis_DR2'        # run_one_model.py L115
front = '_front'                   # L82
MODEL = 'I'

# best-fit params: (5,14) = [c_gas, c_ics, c_GCE, c_bubble, c_iso]  (L598 unpack / L784 reshape)
fp = np.load(glob.glob('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')[0])['fitted_params']
n  = fp.shape[1]

# 마스크: per-bin psc + energy-독립 disk  (L591-593)
psc_all = np.load(f'{WORK}/Model/GC_mask_60x60_definitions_DR2.npy')          # (14,600,600)
disk    = np.load(f'{WORK}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500]

def slc(path, ebin):   # L576-589 와 동일한 [ebin,100:500,100:500] slice
    return fits.open(path)[0].data[ebin, 100:500, 100:500]

print(f'{"bin":>3} {"Σmodel_cnt":>14} {"Σdata_cnt":>14} {"(m-d)/d":>9}')
resid = np.zeros(n)
for ebin in range(n):
    data   = slc(f'{WORK}/GC_ccube_12yr{front}_clean.fits', ebin)
    pionbr = (slc(f'{WORK}/GC_pion_model{MODEL}_12yr{front}_clean.fits', ebin)
              + slc(f'{WORK}/GC_bremss_model{MODEL}_12yr{front}_clean.fits', ebin))
    ics    = slc(f'{WORK}/GC_ics_model{MODEL}_12yr{front}_clean.fits', ebin)
    GCE    = slc(f'{WORK}/GC_GCE_model_12yr{front}_clean.fits', ebin)            # model suffix 없음
    bubble = slc(f'{WORK}/GC_fermi_bubble_model_12yr{front}_clean.fits', ebin)   # model suffix 없음
    iso    = slc(f'{WORK}/GC_isotropic_model_12yr{front}_clean.fits', ebin)      # model suffix 없음

    full_mask = psc_all[ebin, 100:500, 100:500] * disk                          # L591-593
    cg, ci, cgce, cb, ciso = fp[:, ebin]                                        # [gas, ics, GCE, bubble, iso]
    # model counts = L600-604 (exp_cube 곱 없음, convol map 그대로 counts)
    C = cg * pionbr + ci * ics + cgce * GCE + ciso * iso + cb * bubble
    m  = (full_mask == 1)                                                       # L607
    Cm = C[m].sum()
    Dm = data[m].sum()
    resid[ebin] = (Cm - Dm) / Dm
    print(f'{ebin:>3} {Cm:>14.1f} {Dm:>14.1f} {resid[ebin]:>+9.4f}')

print('\n=== high-E (bin 11-13) total-counts 잔차 ===')
print('  (m-d)/d :', np.round(resid[11:], 4))
print('  ~0  → model=data 확정 → ΣΔ>0 은 Fig11 digitize 오차 (ICS over = artifact),')
print('         추적 대상은 GCE high-E over(Zenodo) 하나로 좁혀짐')
print('  양수 → 진짜 over-fit → 더 깊이')
print(f'\n전 bin |(m-d)/d| 최대 = {np.abs(resid).max():.4f} @ bin {np.abs(resid).argmax()}')

In [ ]:
import glob, numpy as np
from astropy.io import fits
from astropy.wcs import WCS

WORK='./GC_analysis_DR2'; front='_front'; MODEL='I'   # ← 'X' 가능

# ---- best-fit params (fit.npz; [gas,ics,GCE,bubble,iso]) ----
npz = np.load(glob.glob(f'./results_12yr/GCE_model_{MODEL}_front_12yr_cholis_fit.npz')[0])
fp = npz['fitted_params']; E = npz['E']; n = len(E)

# ---- mask (Likelihood L591-593) + GLON 방향 라벨용 WCS ----
disk = np.load(f'{WORK}/Model/GC_disk_mask_60x60_definitions.npy')[100:500,100:500]
psc  = np.load(f'{WORK}/Model/GC_mask_60x60_definitions_DR2.npy')[:,100:500,100:500]
_w = WCS(fits.open(f'{WORK}/GC_ccube_12yr{front}_clean.fits')[0].header).dropaxis(2)
l_left,_  = _w.wcs_pix2world(100, 300, 0)    # 슬라이스 좌단(열 인덱스 0 ↔ 원본 col 100)
l_right,_ = _w.wcs_pix2world(499, 300, 0)    # 슬라이스 우단
l_left  = l_left-360  if l_left>180  else l_left
l_right = l_right-360 if l_right>180 else l_right
print(f'[GLON 방향] 슬라이스 좌측 열 = l {l_left:+.1f}°, 우측 열 = l {l_right:+.1f}°  '
      f'(좌측이 +l 이면 표준 galactic 관례)')

def cmap(name,e): return fits.open(f'{WORK}/{name}')[0].data[e,100:500,100:500]

W = 400; halfL = slice(0, W//2); halfR = slice(W//2, W)   # 열(GLON) 좌/우 절반

def asym(arr, fmask):
    """A = Σ(좌측 절반)/Σ(우측 절반), unmasked 픽셀만. >1=좌측 밝음, <1=우측 밝음."""
    m = (fmask==1)
    a = np.where(m, arr, 0.0)
    L = a[:, halfL].sum(); R = a[:, halfR].sum()
    return L/R if R else np.nan

print(f'\n{"bin":>3}{"E":>7} | {"data":>7} {"PB":>7} {"ICS":>7} {"GCE":>7} {"bub":>7} {"iso":>7} | '
      f'{"noGCE잔차":>9} | {"방향:data/PB/잔차":>16}')
for e in range(n):
    cg,ci,cgce,cb,ciso = fp[:,e]
    PB  = cmap(f'GC_pion_model{MODEL}_12yr{front}_clean.fits',e)+cmap(f'GC_bremss_model{MODEL}_12yr{front}_clean.fits',e)
    ICS = cmap(f'GC_ics_model{MODEL}_12yr{front}_clean.fits',e)
    GCE = cmap(f'GC_GCE_model_12yr{front}_clean.fits',e)
    BUB = cmap(f'GC_fermi_bubble_model_12yr{front}_clean.fits',e)
    ISO = cmap(f'GC_isotropic_model_12yr{front}_clean.fits',e)
    DAT = cmap(f'GC_ccube_12yr{front}_clean.fits',e)
    fmask = psc[e]*disk
    full   = cg*PB+ci*ICS+cgce*GCE+ciso*ISO+cb*BUB
    no_gce = full - cgce*GCE
    resid_gce = DAT - no_gce                      # GCE 귀속 잔차 (후보2)
    aD,aPB = asym(DAT,fmask), asym(cg*PB,fmask)
    aIC,aG = asym(ci*ICS,fmask), asym(cgce*GCE,fmask)
    aB,aI  = asym(cb*BUB,fmask), asym(ciso*ISO,fmask)
    aR     = asym(resid_gce,fmask)
    # 방향 부호: >1 → 'L'(좌측 밝음), <1 → 'R'
    sgn = lambda x: 'L' if x>1 else 'R'
    print(f'{e:>3}{E[e]:>7.2f} | {aD:>7.3f} {aPB:>7.3f} {aIC:>7.3f} {aG:>7.3f} {aB:>7.3f} {aI:>7.3f} | '
          f'{aR:>9.3f} | {sgn(aD)}/{sgn(aPB)}/{sgn(aR)}')

print('\n판정:')
print('  - GCE/bub/iso 의 A ≈ 1 (좌우대칭) 이어야 정상 (sanity)')
print('  - PB 비대칭 방향(L/R) 이 data 와 같으면 flip 정렬 일관; 반대면 PB flip 어긋남')
print('  - high-E(bin11-13) 에서 GCE 귀속 잔차 방향이 data 와 반대로 튀면,')
print('    fit 이 PB 정렬오차를 GCE 비대칭으로 떠안은 신호')
print('  → 베이스(REF phase2 §1.1, PROJECT_STATE §3)의 flip 규약과 대조해 정렬 일관성 판정')

In [ ]:
# Test A Step 1 — Cell 35 source + spectrum file extension + MapCube energy axis
import os
import numpy as np
from astropy.io import fits

# 1) extend_spectrum_files.py source view (PL extension 의 정확한 form)
for path in ['./extend_spectrum_files.py', './scripts/extend_spectrum_files.py']:
    if os.path.exists(path):
        print(f'=== {path} ===')
        with open(path) as f:
            print(f.read())
        break
else:
    print('!! extend_spectrum_files.py 위치 미확인 - ls 로 확인 필요')

# 2) spectrum file 의 PL extension 양상 (current vs .original backup)
for spec in ['fermi_bubble_spectrum.txt', 'isotropic_spectrum_ff.txt']:
    p_cur  = f'./{spec}'
    p_orig = f'./{spec}.original'
    print(f'\n=== {spec} ===')
    for label, p in [('current', p_cur), ('original', p_orig)]:
        if os.path.exists(p):
            d = np.loadtxt(p)
            print(f'  {label:>8}: {d.shape[0]:>4} rows, '
                  f'E [GeV] {d[0,0]:.4f} ~ {d[-1,0]:.4f}, '
                  f'last 5 E: {d[-5:, 0]}')
        else:
            print(f'  {label:>8}: not found ({p})')

# 3) MapCube energy axis (38-bin native or PL-extended?)
p = './MapCubes/pion_mapcube_modelX.fits'
if os.path.exists(p):
    with fits.open(p) as h:
        print(f'\n=== {p} ===')
        print(f'data shape: {h[0].data.shape}  (axis=0 = energy)')
        for ext_idx in range(len(h)):
            ext = h[ext_idx]
            print(f'  HDU[{ext_idx}] name={ext.name!r}', end='')
            if ext.data is not None and hasattr(ext.data, 'dtype') and ext.data.dtype.names:
                print(f', cols={ext.data.dtype.names}')
                if 'Energy' in (ext.data.dtype.names or ()):
                    e = ext.data['Energy']
                    print(f'    Energy [MeV]: {e[0]:.1f} ~ {e[-1]:.1f} ({len(e)} entries)')
            else:
                print()

In [ ]:
# Test B — 14-bin grouping verification (no fit needed)
# 우리 bin_definitions.fits vs Cholis Zenodo 14-bin grouping 직접 비교
import numpy as np
from astropy.io import fits

# 우리 bin_definitions.fits (keV 단위, baseline 메모리 #5 명시 TUNIT2=keV)
bd = fits.open('./GC_analysis_DR2/bin_definitions.fits')
print('=== bin_definitions.fits structure ===')
bd.info()
ebounds = bd[1].data if len(bd) > 1 else bd[0].data
print(f'\nColumns: {ebounds.dtype.names}')
print(f'Number of bins: {len(ebounds)}')

# Try common column names
for col_min, col_max in [('E_MIN', 'E_MAX'), ('CHANNEL_MIN', 'CHANNEL_MAX')]:
    if col_min in ebounds.dtype.names:
        edges_input = np.concatenate([ebounds[col_min], [ebounds[col_max][-1]]])
        break
else:
    edges_input = None
    print('!! Unknown column names — manual inspection needed')

if edges_input is not None:
    # unit detection: > 1000 → keV, < 100 → GeV
    if edges_input.max() > 1000:
        edges_GeV = edges_input * 1e-6
        print(f'Unit detected: keV → converted to GeV')
    else:
        edges_GeV = edges_input
        print(f'Unit detected: GeV (no conversion)')
    print(f'우리 14-bin edges [GeV]: {edges_GeV}')

# Cholis Zenodo 14-bin grouping (Cholis_ZENODO_README + paper Table III)
# bin label : native bins (38-bin) : E_min[MeV], E_ctr[MeV], E_max[MeV]
cholis_table = [
    ( 0, [ 7],       274.698,   357.014),
    ( 1, [ 8],       357.014,   463.995),
    ( 2, [ 9],       463.995,   603.034),
    ( 3, [10],       603.034,   783.737),
    ( 4, [11],       783.737,  1018.59),
    ( 5, [12],      1018.59,   1323.82),
    ( 6, [13],      1323.82,   1720.51),
    ( 7, [14],      1720.51,   2236.07),
    ( 8, [15],      2236.07,   2906.12),
    ( 9, [16],      2906.12,   3776.96),
    (10, [17],      3776.96,   4908.75),
    (11, [18,19,20], 4908.75, 10776.0),
    (12, [21,22,23],10776.0,  23656.1),
    (13, [24,25,26],23656.1,  51931.2),
]

print(f'\n=== Comparison: 우리 vs Cholis Zenodo 14-bin edges (GeV) ===')
print(f'{"bin":>3} {"native":<14} {"E_min_GeV":>12} {"E_max_GeV":>12} '
      f'{"우리_min":>12} {"우리_max":>12} {"min_diff%":>10} {"max_diff%":>10}')
for b, native, emin, emax in cholis_table:
    em_GeV = emin * 1e-3
    eM_GeV = emax * 1e-3
    if edges_input is not None and b+1 < len(edges_GeV):
        u_min = edges_GeV[b]
        u_max = edges_GeV[b+1]
        d_min = (u_min - em_GeV) / em_GeV * 100
        d_max = (u_max - eM_GeV) / eM_GeV * 100
        print(f'{b:>3} {str(native):<14} {em_GeV:>12.5f} {eM_GeV:>12.5f} '
              f'{u_min:>12.5f} {u_max:>12.5f} {d_min:>+10.3f} {d_max:>+10.3f}')
    else:
        print(f'{b:>3} {str(native):<14} {em_GeV:>12.5f} {eM_GeV:>12.5f} '
              f'{"N/A":>12} {"N/A":>12} {"-":>10} {"-":>10}')

# bin 11-13 의 grouping 양상 (high-E over 의 origin 추적)
print(f'\n=== bin 11-13 (high-E grouping 의 native bin 수) ===')
print('  bin 11: native [18,19,20] (4.9-10.8 GeV)  ← 3 native bins → 1 우리 bin')
print('  bin 12: native [21,22,23] (10.8-23.7 GeV) ← 3 native bins → 1 우리 bin')
print('  bin 13: native [24,25,26] (23.7-51.9 GeV) ← 3 native bins → 1 우리 bin')
print('  (bin 0-10 은 1 native = 1 우리, bin 11-13 은 3 native = 1 우리)')

In [ ]:
# ============================================================================
# Cell 8t — Self-contained likelihood probe (Eq.13 reimplementation)
# ============================================================================
# Likelihood class + dependencies 직접 paste (run_one_model.py L530-680 verbatim).
# import 안 함 → run_one_model.py module-level XML build side-effect 회피.

import os, numpy as np
from astropy.io import fits as _fits
from astropy.wcs import WCS as _WCS
from scipy.special import gammaln
from scipy.interpolate import interp1d as _ip

# ---- (1) globals setup: cell 1/5 에 있는 것 우선, 없으면 default ----
WORK_DIR = globals().get('WORK_DIR', '.')
front    = globals().get('FRONT', '_front')
ANALYSIS = globals().get('ANALYSIS_DIR', f'{WORK_DIR}/GC_analysis_DR2')

# ---- (2) psc_mask + disk_mask (full 600x600, slice 100:500 inside Likelihood) ----
psc_mask  = np.load(f'{ANALYSIS}/Model/GC_mask_60x60_definitions_DR2.npy').astype(int)
disk_mask = np.load(f'{ANALYSIS}/Model/GC_disk_mask_60x60_definitions.npy').astype(int)

# ---- (3) steradian_per_pixel (full 600x600 from ccube WCS) ----
_ccube_p = f'{ANALYSIS}/GC_ccube_12yr{front}_clean.fits'
_rm  = _fits.open(_ccube_p)
_wcs = _WCS(_rm[0].header).dropaxis(2)
_W, _H = np.shape(_rm[0].data[0])
steradian_per_pixel = np.zeros([_W, _H])
for i in range(_H):
    for j in range(_W):
        _, _b = _wcs.wcs_pix2world(j, i, 0)
        steradian_per_pixel[i, j] = np.radians(0.1) ** 2 * np.cos(np.radians(_b))
_rm.close()

# ---- (4) external constraints (bubble + iso) ----
def log_factorial(O):
    return gammaln(np.asarray(O, dtype=float) + 1.0)

_bc = np.loadtxt(f'{ANALYSIS}/Model/bubble_constraints.txt')
bubble_flux_data        = _ip(_bc[:,0], _bc[:,1], fill_value='extrapolate', kind='quadratic')(E)
bubble_lower_error_data = _ip(_bc[:,0], _bc[:,2], fill_value='extrapolate', kind='quadratic')(E)
bubble_upper_error_data = _ip(_bc[:,0], _bc[:,3], fill_value='extrapolate', kind='quadratic')(E)

_ic = np.loadtxt(f'{ANALYSIS}/Model/iso_constraints_full_err.txt')
isotropic_flux_data        = (E ** 2) * _ip(_ic[:,0], _ic[:,1], fill_value='extrapolate', kind='quadratic')(E)
isotropic_lower_error_data = (E ** 2) * _ip(_ic[:,0], _ic[:,2], fill_value='extrapolate', kind='quadratic')(E)
isotropic_upper_error_data = (E ** 2) * _ip(_ic[:,0], _ic[:,3], fill_value='extrapolate', kind='quadratic')(E)

# ---- (5) Likelihood class — verbatim from run_one_model.py L570-653 ----
class Likelihood:
    def __init__(self, model, energy_bin):
        self.model       = model
        self.energy_bin  = energy_bin
        self.data        = _fits.open(f'{WORK_DIR}/GC_ccube_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
        self.pion_bremss = (_fits.open(f'{WORK_DIR}/GC_pion_model{model}_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
                          + _fits.open(f'{WORK_DIR}/GC_bremss_model{model}_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500])
        self.ics    = _fits.open(f'{WORK_DIR}/GC_ics_model{model}_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
        self.GCE    = _fits.open(f'{WORK_DIR}/GC_GCE_model_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
        self.bubble = _fits.open(f'{WORK_DIR}/GC_fermi_bubble_model_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
        self.iso    = _fits.open(f'{WORK_DIR}/GC_isotropic_model_12yr{front}_clean.fits')[0].data[energy_bin, 100:500, 100:500]
        self.iso_no_convol    = _fits.open(f'{WORK_DIR}/GC_isotropic_model_12yr{front}_clean_no_convol.fits')[0].data[energy_bin, 100:500, 100:500]
        self.bubble_no_convol = _fits.open(f'{WORK_DIR}/GC_fermi_bubble_model_12yr{front}_clean_no_convol.fits')[0].data[energy_bin, 100:500, 100:500]
        self.E       = E
        self.delta_E = delta_E
        self.exp_cube = (_fits.open(f'{WORK_DIR}/GC_expcube_center_12yr{front}_clean.fits')[0].data[energy_bin]
                         * steradian_per_pixel)[100:500, 100:500]
        _psc_mask = psc_mask[energy_bin]
        self.disk_mask = disk_mask
        self.full_mask = _psc_mask * disk_mask
        _obs_masked = self.data[self.full_mask == 1].astype(float)
        self.observed_log_factorial_masked = log_factorial(_obs_masked)

    def likelihood_constrained(self, parameter_set):
        pion_bremss_param, ics_param, GCE_param, bubble_param, isotropic_param = parameter_set
        expected_pixel = (pion_bremss_param * self.pion_bremss
                          + ics_param         * self.ics
                          + GCE_param         * self.GCE
                          + isotropic_param   * self.iso
                          + bubble_param      * self.bubble)
        observed_pixel = self.data
        observed_pixel = observed_pixel[self.full_mask == 1]
        expected_pixel = expected_pixel[self.full_mask == 1]
        if (expected_pixel < 0).any():
            return np.inf
        observed_log_expected = observed_pixel * np.log(expected_pixel)
        lhd = 2 * (expected_pixel - observed_log_expected + self.observed_log_factorial_masked)

        isotropic = (np.sum(self.full_mask * (self.iso_no_convol) / self.exp_cube)
                     * isotropic_param / np.sum(self.full_mask))
        isotropic_sed = (self.E[self.energy_bin] ** 2) * isotropic / (self.delta_E[self.energy_bin])
        bubble = (np.sum(self.full_mask * (self.bubble_no_convol) / self.exp_cube)
                  * bubble_param / np.sum(self.full_mask))
        bubble_sed = (self.E[self.energy_bin] ** 2) * bubble / (self.delta_E[self.energy_bin])

        larger_error = max([bubble_upper_error_data[self.energy_bin],
                            bubble_lower_error_data[self.energy_bin]])
        if bubble_flux_data[self.energy_bin] < bubble_sed:
            chi2_bubble = ((bubble_sed - bubble_flux_data[self.energy_bin])
                           / bubble_upper_error_data[self.energy_bin]) ** 2
        elif bubble_flux_data[self.energy_bin] > bubble_sed:
            chi2_bubble = ((bubble_sed - bubble_flux_data[self.energy_bin])
                           / bubble_lower_error_data[self.energy_bin]) ** 2
        else:
            chi2_bubble = ((bubble_sed - bubble_flux_data[self.energy_bin])
                           / larger_error) ** 2

        isotropic_larger_error = max([isotropic_lower_error_data[self.energy_bin],
                                      isotropic_upper_error_data[self.energy_bin]])
        if isotropic_flux_data[self.energy_bin] < isotropic_sed:
            chi2_isotropic = ((isotropic_flux_data[self.energy_bin] - isotropic_sed)
                              / isotropic_upper_error_data[self.energy_bin]) ** 2
        elif isotropic_flux_data[self.energy_bin] > isotropic_sed:
            chi2_isotropic = ((isotropic_flux_data[self.energy_bin] - isotropic_sed)
                              / isotropic_lower_error_data[self.energy_bin]) ** 2
        else:
            chi2_isotropic = ((isotropic_flux_data[self.energy_bin] - isotropic_sed)
                              / isotropic_larger_error) ** 2

        return np.sum(lhd) + chi2_bubble + chi2_isotropic


# ---- (6) Probe bin 7: paper c vs our fit c ----
BIN_PROBE = 7

# paper c at bin 7 — interp from c_paper dict (cell 8q v3)
c_PB_paper  = float(_ip(E_p, c_paper['pi+brem'], bounds_error=False, fill_value=np.nan)(E[BIN_PROBE]))
c_ICS_paper = float(_ip(E_p, c_paper['ICS'],     bounds_error=False, fill_value=np.nan)(E[BIN_PROBE]))
# GCE/Bub/Iso paper c: 1-10 GeV mean from cell 8e raw/dashed reciprocal (per-bin variant 없음)
c_GCE_paper, c_bub_paper, c_iso_paper = 3.13, 0.34, 1.61
paper_c = [c_PB_paper, c_ICS_paper, c_GCE_paper, c_bub_paper, c_iso_paper]

npz = np.load(f'./results_12yr/GCE_model_{SELECTED_MODEL}_front_12yr_cholis_fit.npz')
our_c = list(npz['fitted_params'][:, BIN_PROBE])

print(f'=== bin {BIN_PROBE} (E = {E[BIN_PROBE]:.3f} GeV)  Model {SELECTED_MODEL} ===\n')
print(f'  {"param":<8} {"paper c":>11} {"our fit c":>11} {"ratio":>8}')
for i, name in enumerate(['c_PB', 'c_ICS', 'c_GCE', 'c_bub', 'c_iso']):
    r = our_c[i] / paper_c[i] if paper_c[i] != 0 else float('nan')
    print(f'  {name:<8} {paper_c[i]:>11.4f} {our_c[i]:>11.4f} {r:>8.3f}')

print(f'\nBuilding Likelihood({SELECTED_MODEL!r}, {BIN_PROBE}) ...')
#lh = Likelihood(SELECTED_MODEL, BIN_PROBE)
print('  built.')

#v_ours  = float(lh.likelihood_constrained(our_c))
#v_paper = float(lh.likelihood_constrained(paper_c))
#delta   = v_paper - v_ours

#print(f'\n=== Eq.13 (-2 log L) values, larger = worse fit ===')
#print(f'  @ our  c = {v_ours:>16.4f}')
#print(f'  @ paper c = {v_paper:>16.4f}')
#print(f'  Δ        = {delta:>+16.4f}  (paper - ours)')

#mlh = float(npz['max_likelihood'][BIN_PROBE])
#print(f'\n  .npz max_likelihood[bin {BIN_PROBE}]  = {mlh:.4f}')
#print(f'  -2 × mlh                        = {-2*mlh:.4f}   (expected ≈ v_ours)')

#abs_delta = abs(delta)
#print(f'\n=== Interpretation ===')
#if delta > 0:
#    print(f'  paper c is WORSE by Δ(-2logL) = {delta:.2f}  →  √Δ = {np.sqrt(delta):.2f}σ')
#else:
#    print(f'  paper c is BETTER by |Δ| = {-delta:.2f}  →  emcee missed the better mode')

#print(f'\n=== Scenario ===')
#if abs_delta < 5:
#    print(f'  |Δ| < 5: same likelihood valley → (A) emcee random mode collapse')
#elif abs_delta < 50:
#    print(f'  |Δ| 5-50: distinct nearby modes / minor code diff')
#else:
#    print(f'  |Δ| > 50: paper c forbidden in our likelihood → likelihood code body differs')

In [ ]:
# Cell 8t-fix2 — mask slice fix + WORK_DIR override + sanity-checked probe

WORK_DIR = './GC_analysis_DR2'   # run_one_model.py L115 와 일치
front = FRONT

# Mask re-load with correct slicing (run_one_model.py L492-493 verbatim)
psc_mask  = np.load(f'{WORK_DIR}/Model/GC_mask_60x60_definitions_DR2.npy')[:, 100:500, 100:500].astype(int)
disk_mask = np.load(f'{WORK_DIR}/Model/GC_disk_mask_60x60_definitions.npy')[100:500, 100:500].astype(int)

# ---- Sanity checks (fail-fast before Likelihood build) ----
import os as _os
_paths = [
    f'{WORK_DIR}/GC_ccube_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_pion_model{SELECTED_MODEL}_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_bremss_model{SELECTED_MODEL}_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_ics_model{SELECTED_MODEL}_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_GCE_model_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_fermi_bubble_model_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_isotropic_model_12yr{front}_clean.fits',
    f'{WORK_DIR}/GC_isotropic_model_12yr{front}_clean_no_convol.fits',
    f'{WORK_DIR}/GC_fermi_bubble_model_12yr{front}_clean_no_convol.fits',
    f'{WORK_DIR}/GC_expcube_center_12yr{front}_clean.fits',
]
_missing = [p for p in _paths if not _os.path.exists(p)]
assert not _missing, f'missing files:\n  ' + '\n  '.join(_missing)
assert psc_mask.shape == (len(E), 400, 400), f'psc_mask shape {psc_mask.shape}, expected (14, 400, 400)'
assert disk_mask.shape == (400, 400),        f'disk_mask shape {disk_mask.shape}, expected (400, 400)'
assert steradian_per_pixel.shape == (600, 600), \
    f'steradian_per_pixel shape {steradian_per_pixel.shape}, expected (600, 600)  (Likelihood slices internally)'
print(f'[sanity OK] all 10 FITS files present, masks {psc_mask.shape}/{disk_mask.shape}, '
      f'unmasked pixels @ bin {BIN_PROBE}: {(psc_mask[BIN_PROBE] * disk_mask).sum()}')

# ---- Build + probe ----
print(f'\nBuilding Likelihood({SELECTED_MODEL!r}, {BIN_PROBE}) ...')
lh = Likelihood(SELECTED_MODEL, BIN_PROBE)
print(f'  built: data {lh.data.shape}, full_mask sum {lh.full_mask.sum()}, '
      f'exp_cube range [{lh.exp_cube.min():.2e}, {lh.exp_cube.max():.2e}]')

v_ours  = float(lh.likelihood_constrained(our_c))
v_paper = float(lh.likelihood_constrained(paper_c))
delta   = v_paper - v_ours

print(f'\n=== Eq.13 (-2 log L) values, larger = worse fit ===')
print(f'  @ our  c  = {v_ours:>16.4f}')
print(f'  @ paper c = {v_paper:>16.4f}')
print(f'  Δ         = {delta:>+16.4f}  (paper - ours)')

mlh = float(npz['max_likelihood'][BIN_PROBE])
print(f'\n  .npz max_likelihood[bin {BIN_PROBE}] = {mlh:.4f}')
print(f'  -2 × mlh                       = {-2*mlh:.4f}   (sanity: should ≈ v_ours)')
print(f'  v_ours / (-2 × mlh) - 1        = {v_ours / (-2*mlh) - 1:+.6f}  (≈0 → convention matches)')

abs_delta = abs(delta)
print(f'\n=== Interpretation ===')
if delta > 0:
    print(f'  paper c is WORSE by Δ(-2logL) = {delta:.2f}  →  √Δ = {np.sqrt(delta):.2f}σ')
else:
    print(f'  paper c is BETTER by |Δ| = {-delta:.2f}  →  emcee missed the better mode')

print(f'\n=== Scenario ===')
if abs_delta < 5:
    print(f'  |Δ| < 5  : same likelihood valley → (A) emcee random mode collapse')
elif abs_delta < 50:
    print(f'  |Δ| 5-50 : distinct nearby modes / minor code diff')
else:
    print(f'  |Δ| > 50 : paper c forbidden in our likelihood → likelihood code body differs')

In [ ]:
# Cell 8t-decompose — Δ(-2logL) 의 Poisson vs χ²_ext 기여 분해
# Likelihood.likelihood_constrained 의 internal 값을 instrumented 재현 (run_one_model.py L595-653 동일)

def _decomposed_likelihood(lh, params):
    """Likelihood.likelihood_constrained 와 bit-for-bit 동일하되 세 부분 반환."""
    pb, ic, gc, bu, iso = params
    expected = pb * lh.pion_bremss + ic * lh.ics + gc * lh.GCE + iso * lh.iso + bu * lh.bubble
    obs = lh.data
    obs_m = obs[lh.full_mask == 1]
    exp_m = expected[lh.full_mask == 1]
    if (exp_m < 0).any():
        return dict(poisson=np.inf, chi2_bub=0.0, chi2_iso=0.0, total=np.inf)
    lhd = 2 * (exp_m - obs_m * np.log(exp_m) + lh.observed_log_factorial_masked)
    poisson_sum = float(np.sum(lhd))
    # bubble SED
    eb = lh.energy_bin
    bub_count = (np.sum(lh.full_mask * lh.bubble_no_convol / lh.exp_cube) * bu) / np.sum(lh.full_mask)
    bub_sed   = (lh.E[eb] ** 2) * bub_count / lh.delta_E[eb]
    if bubble_flux_data[eb] < bub_sed:
        chi2_b = ((bub_sed - bubble_flux_data[eb]) / bubble_upper_error_data[eb]) ** 2
    elif bubble_flux_data[eb] > bub_sed:
        chi2_b = ((bub_sed - bubble_flux_data[eb]) / bubble_lower_error_data[eb]) ** 2
    else:
        chi2_b = 0.0
    # iso SED
    iso_count = (np.sum(lh.full_mask * lh.iso_no_convol / lh.exp_cube) * iso) / np.sum(lh.full_mask)
    iso_sed   = (lh.E[eb] ** 2) * iso_count / lh.delta_E[eb]
    if isotropic_flux_data[eb] < iso_sed:
        chi2_i = ((isotropic_flux_data[eb] - iso_sed) / isotropic_upper_error_data[eb]) ** 2
    elif isotropic_flux_data[eb] > iso_sed:
        chi2_i = ((isotropic_flux_data[eb] - iso_sed) / isotropic_lower_error_data[eb]) ** 2
    else:
        chi2_i = 0.0
    return dict(poisson=poisson_sum, chi2_bub=float(chi2_b), chi2_iso=float(chi2_i),
                total=poisson_sum + chi2_b + chi2_i,
                bub_sed=float(bub_sed), iso_sed=float(iso_sed))

d_ours  = _decomposed_likelihood(lh, our_c)
d_paper = _decomposed_likelihood(lh, paper_c)

print(f'=== Δ(-2logL) decomposition at bin {BIN_PROBE} (E = {E[BIN_PROBE]:.3f} GeV) ===\n')
print(f'  {"term":<14} {"our c":>14} {"paper c":>14} {"Δ (paper-ours)":>18}')
for k in ['poisson', 'chi2_bub', 'chi2_iso', 'total']:
    delta_k = d_paper[k] - d_ours[k]
    print(f'  {k:<14} {d_ours[k]:>14.4f} {d_paper[k]:>14.4f} {delta_k:>+18.4f}')

print(f'\n  bubble_sed comparison:')
print(f'    @ our c  : bub_sed = {d_ours["bub_sed"]:.3e},  '
      f'paper data = {bubble_flux_data[BIN_PROBE]:.3e},  '
      f'σ_low/high = {bubble_lower_error_data[BIN_PROBE]:.3e}/{bubble_upper_error_data[BIN_PROBE]:.3e}')
print(f'    @ paper c: bub_sed = {d_paper["bub_sed"]:.3e}')

print(f'\n  iso_sed comparison:')
print(f'    @ our c  : iso_sed = {d_ours["iso_sed"]:.3e},  '
      f'paper data = {isotropic_flux_data[BIN_PROBE]:.3e},  '
      f'σ_low/high = {isotropic_lower_error_data[BIN_PROBE]:.3e}/{isotropic_upper_error_data[BIN_PROBE]:.3e}')
print(f'    @ paper c: iso_sed = {d_paper["iso_sed"]:.3e}')

print(f'\n=== Conclusion based on dominant term ===')
delta_poisson = d_paper['poisson'] - d_ours['poisson']
delta_chi2    = (d_paper['chi2_bub'] + d_paper['chi2_iso']) - (d_ours['chi2_bub'] + d_ours['chi2_iso'])
if abs(delta_chi2) > 2 * abs(delta_poisson):
    print(f'  χ²_external dominates ({delta_chi2:+.1f} vs Poisson {delta_poisson:+.1f})')
    print(f'  → paper c의 bub/iso template SED 가 우리 template normalize 위에서 external constraint 와 불일치')
    print(f'  → root cause = template normalize convention (Phase 6 §6.4), NOT fit dynamics')
elif abs(delta_poisson) > 2 * abs(delta_chi2):
    print(f'  Poisson dominates ({delta_poisson:+.1f} vs χ²_ext {delta_chi2:+.1f})')
    print(f'  → paper c 가 우리 Poisson likelihood 위에서 진정한 worse minimum')
    print(f'  → root cause = likelihood code body 또는 fit dynamics')
else:
    print(f'  Mixed: Poisson {delta_poisson:+.1f}, χ²_ext {delta_chi2:+.1f}')

In [ ]:
import numpy as np
d = np.load('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')
print('=== .npz keys + shapes ===')
for k in d.files:
    a = d[k]
    print(f'  {k:<24} shape={a.shape}   dtype={a.dtype}')
print(f'\n=== fitted_params shape interpretation ===')
fp = d['fitted_params']
print(f'  shape={fp.shape}')
print(f'  if (5, 14): 5 free params (PB, ICS, GCE, Bub, Iso) — paper Eq.7 convention')
print(f'  if (6, 14): 6 free params (Pi0, Brem, ICS, GCE, Bub, Iso) separately')
print(f'  fp[:, 7] (bin 7) = {fp[:, 7] if fp.ndim == 2 else fp}')

In [ ]:
from astropy.io import fits

print('=== srcmap FITS provenance check (Model I, FRONT, bin 7) ===\n')
for c in ['pion', 'bremss', 'ics']:
    p = f'./GC_analysis_DR2/GC_{c}_modelI_12yr_front_clean.fits'
    h = fits.open(p)[0].header
    # HISTORY 카드 + 일반 keyword 둘 다 훑기
    relevant = []
    for k in ('HISTORY', 'COMMENT'):
        if k in h:
            relevant += [(k, str(v)) for v in h[k]]
    relevant += [(k, str(v)) for k, v in h.items()
                 if k in ('CMAPFILE', 'SRCMAP', 'INFILE') or 'MAP' in k.upper()]
    print(f'--- file: {p} ---')
    for k, line in relevant:
        if 'Map_flux' in line or 'mapcube' in line.lower() or 'cmap' in line.lower() \
           or '.fits' in line or 'INFILE' in k or 'SRCMAP' in k:
            print(f'  [{k}] {line}')
    print()

In [ ]:
# Cell 8u — Sang c probe (bin 7) : PB/ICS-only swap, isolate PB↔ICS axis cost

# Sang c at bin 7 (Model I), from memory #21: /home/sanghwan/.../cell 45 output
# Full 5-comp 모르는 상태 — GCE/Bub/Iso 는 our_c 유지 → PB/ICS axis 단독 probe.
sang_pb_bin7  = 0.962
sang_ics_bin7 = 1.023

sang_partial_c = np.array([sang_pb_bin7, sang_ics_bin7, our_c[2], our_c[3], our_c[4]])

v_sang  = float(lh.likelihood_constrained(sang_partial_c))
delta_s = v_sang - v_ours
d_sang  = _decomposed_likelihood(lh, sang_partial_c)

print(f'=== Sang c probe (PB/ICS swap only) at bin {BIN_PROBE} (E = {E[BIN_PROBE]:.3f} GeV) ===\n')
print(f'              {"PB":>8} {"ICS":>8} {"GCE":>8} {"Bub":>8} {"Iso":>8}')
print(f'  our   c  : {our_c[0]:>8.3f} {our_c[1]:>8.3f} {our_c[2]:>8.3f} {our_c[3]:>8.3f} {our_c[4]:>8.3f}')
print(f'  paper c  : {paper_c[0]:>8.3f} {paper_c[1]:>8.3f} {paper_c[2]:>8.3f} {paper_c[3]:>8.3f} {paper_c[4]:>8.3f}')
print(f'  Sang  c* : {sang_partial_c[0]:>8.3f} {sang_partial_c[1]:>8.3f} {sang_partial_c[2]:>8.3f} {sang_partial_c[3]:>8.3f} {sang_partial_c[4]:>8.3f}   (* GCE/Bub/Iso = our_c)')

print(f'\n  v_ours  = {v_ours:>14.4f}')
print(f'  v_paper = {v_paper:>14.4f}   Δ = {v_paper - v_ours:>+12.4f}   √|Δ| = {np.sqrt(abs(v_paper - v_ours)):>6.2f}σ')
print(f'  v_sang  = {v_sang:>14.4f}   Δ = {delta_s:>+12.4f}   √|Δ| = {np.sqrt(abs(delta_s)):>6.2f}σ')

print(f'\n  Sang c decomposition: poisson={d_sang["poisson"]:.2f}  χ²_bub={d_sang["chi2_bub"]:.2f}  χ²_iso={d_sang["chi2_iso"]:.2f}')

print(f'\n=== Branch decision ===')
if abs(delta_s) > 100:
    print(f'  |Δ_sang| = {abs(delta_s):.1f} > 100  →  H1 (PB/ICS spatial distinguishability)')
elif abs(delta_s) < 10:
    print(f'  |Δ_sang| = {abs(delta_s):.1f} < 10   →  H3 (spatial residual map + emcee mode trap check)')
else:
    print(f'  |Δ_sang| = {abs(delta_s):.1f} (intermediate)  →  full 5-comp probe (B) 필요')

In [ ]:
# Cell 8v — H1: PB/ICS conv map pixel-level shape comparison (bin 7)
from astropy.io import fits

US_DIR = './GC_analysis_DR2'
SG_DIR = '/home/sanghwan/FermiLAT/Sanghwan/GC_analysis'

def load_bin(d, comp, b):
    p = f'{d}/GC_{comp}_modelI_12yr_front_clean.fits'
    return fits.open(p)[0].data[b][100:500, 100:500]   # match Likelihood slicing

b = BIN_PROBE
us_pb_full = load_bin(US_DIR, 'pion', b) + load_bin(US_DIR, 'bremss', b)
us_ic_full = load_bin(US_DIR, 'ics',  b)
sg_pb_full = load_bin(SG_DIR, 'pion', b) + load_bin(SG_DIR, 'bremss', b)
sg_ic_full = load_bin(SG_DIR, 'ics',  b)

mask = (psc_mask[b] * disk_mask).astype(bool)
us_pb, us_ic = us_pb_full[mask], us_ic_full[mask]
sg_pb, sg_ic = sg_pb_full[mask], sg_ic_full[mask]

def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a*b).sum() / np.sqrt((a**2).sum() * (b**2).sum()))

print(f'=== bin {b} (E={E[b]:.3f} GeV)  |  masked pixels: {mask.sum()} ===\n')
print(f'(1) Sum ratio (us/sang)        PB={us_pb.sum()/sg_pb.sum():.4f}   ICS={us_ic.sum()/sg_ic.sum():.4f}')
print(f'\n(2) Same-component corr (us↔sg)   PB={corr(us_pb, sg_pb):.4f}   ICS={corr(us_ic, sg_ic):.4f}')
print(f'\n(3) Within-env PB↔ICS distinguishability (lower = more distinguishable)')
print(f'    us:   corr(PB, ICS) = {corr(us_pb, us_ic):.4f}')
print(f'    sang: corr(PB, ICS) = {corr(sg_pb, sg_ic):.4f}')
print(f'\n(4) Cross-component (swap detector — should be << same-component if no swap)')
print(f'    corr(us_PB, sg_ICS) = {corr(us_pb, sg_ic):.4f}')
print(f'    corr(us_ICS, sg_PB) = {corr(us_ic, sg_pb):.4f}')

# Pixel ratio distribution (sum 1% 정합이 어떻게 분포되는지)
print(f'\n(5) Pixel ratio us/sang distribution')
r_pb = us_pb / np.clip(sg_pb, 1e-30, None)
r_ic = us_ic / np.clip(sg_ic, 1e-30, None)
print(f'    PB:  median={np.median(r_pb):.4f}  std={np.std(r_pb):.4f}  '
      f'[p5={np.percentile(r_pb,5):.3f}, p95={np.percentile(r_pb,95):.3f}]')
print(f'    ICS: median={np.median(r_ic):.4f}  std={np.std(r_ic):.4f}  '
      f'[p5={np.percentile(r_ic,5):.3f}, p95={np.percentile(r_ic,95):.3f}]')

print(f'\n=== Branch decision ===')
c_same_pb, c_same_ic = corr(us_pb, sg_pb), corr(us_ic, sg_ic)
c_cross_us, c_cross_sg = corr(us_pb, sg_ic), corr(us_ic, sg_pb)
if c_cross_us > c_same_pb or c_cross_sg > c_same_ic:
    print('  cross > same → spatial-level swap (file/XML 묶음 다름)')
elif min(c_same_pb, c_same_ic) < 0.95:
    print(f'  same-corr < 0.95 → pixel-level shape 다름 (root cause = gtsrcmaps stage shape)')
else:
    print(f'  same-corr > 0.95 → conv map shape 정합 → CCUBE pixel-level 또는 mask interaction')

In [ ]:
# Cell 8w — H1 deeper: pion / bremss 분리 (bin 7, Model I)
# cell 8v 의 load_bin, corr, mask, b, US_DIR, SG_DIR 재사용

for comp in ('pion', 'bremss'):
    us_full = load_bin(US_DIR, comp, b)
    sg_full = load_bin(SG_DIR, comp, b)
    us = us_full[mask]; sg = sg_full[mask]
    r  = us / np.clip(sg, 1e-30, None)
    print(f'\n=== {comp} (bin {b}, E={E[b]:.3f} GeV) ===')
    print(f'  sum ratio (us/sg) : {us.sum()/sg.sum():.4f}')
    print(f'  corr us↔sg        : {corr(us, sg):.4f}')
    print(f'  pixel ratio       : median={np.median(r):.4f}  std={np.std(r):.4f}  '
          f'[p5={np.percentile(r,5):.3f}, p95={np.percentile(r,95):.3f}]')
    globals()[f'us_{comp}'] = us
    globals()[f'sg_{comp}'] = sg

print(f'\n=== Within-env pion↔bremss correlation ===')
print(f'  us:   corr(pion, bremss) = {corr(us_pion, us_bremss):.4f}')
print(f'  sang: corr(pion, bremss) = {corr(sg_pion, sg_bremss):.4f}')

print(f'\n=== Cross-component (pion ↔ bremss swap detector) ===')
print(f'  corr(us_pion,   sg_bremss) = {corr(us_pion, sg_bremss):.4f}')
print(f'  corr(us_bremss, sg_pion)   = {corr(us_bremss, sg_pion):.4f}')

In [ ]:
# Cell 8z (new) — Task D: 14-bin × 5-component our/paper ratio (Model I)
# Cell 8e/8h form 결합, mask = disk-only (Cell 5/8e 와 동일), 보간 linear
# 출력: RAW (c=1) + post-fit (Model I .npz baseline c) 의 14-bin 표 + 1-10 GeV mean

import os
import numpy as np
from scipy.interpolate import interp1d

MODEL_LBL = 'I'
FIG11_TXT = './2112_09706_Fig11_ModelI_ALL.txt'
NPZ_PATH  = f'./results_12yr/GCE_model_{MODEL_LBL}_front_12yr_cholis_fit.npz'

# (1) paper Fig 11 manual digitize → 우리 E grid (14-bin) 으로 linear 보간
fig11 = np.loadtxt(FIG11_TXT)
E_p = fig11[:, 0]
paper = {'PB':  fig11[:, 1], 'ICS': fig11[:, 4], 'Bub': fig11[:, 7],
         'Iso': fig11[:,10], 'GCE': fig11[:,13]}
paper_at_us = {k: interp1d(E_p, v, kind='linear', bounds_error=False,
                           fill_value=np.nan)(E) for k, v in paper.items()}

# (2) Model I template flux (Cell 5 helper 직접, model='I' 강제)
comp_flux_array = _loader_info[0]
pion_I   = comp_flux_array(comp_path('pion',         MODEL_LBL, convol=False))
bremss_I = comp_flux_array(comp_path('bremss',       MODEL_LBL, convol=False))
ics_I    = comp_flux_array(comp_path('ics',          MODEL_LBL, convol=False))
GCE_I    = comp_flux_array(comp_path('GCE',          None,      convol=False))
bub_I    = comp_flux_array(comp_path('fermi_bubble', None,      convol=False))
iso_I    = comp_flux_array(comp_path('isotropic',    None,      convol=False))

E2_dE = E**2 / delta_E
raw_us = {
    'PB':  (pion_I + bremss_I) * E2_dE,
    'ICS': ics_I                * E2_dE,
    'Bub': bub_I                * E2_dE,
    'Iso': iso_I                * E2_dE,
    'GCE': GCE_I                * E2_dE,
}
COMPS = ['PB', 'ICS', 'Bub', 'Iso', 'GCE']

# (3) Model I .npz 의 baseline c (있을 경우 post-fit)
post_us = None
if os.path.exists(NPZ_PATH):
    npz = np.load(NPZ_PATH)
    fp  = npz['fitted_params']
    nE  = len(E)
    if fp.ndim == 2 and fp.shape[0] == 5:
        c_I = {'PB': fp[0], 'ICS': fp[1], 'GCE': fp[2], 'Bub': fp[3], 'Iso': fp[4]}
    elif fp.ndim == 1 and fp.size == 5*nE:
        c_I = {'PB': fp[0:nE], 'ICS': fp[nE:2*nE], 'GCE': fp[2*nE:3*nE],
               'Bub': fp[3*nE:4*nE], 'Iso': fp[4*nE:5*nE]}
    else:
        c_I = None
        print(f'[warn] fitted_params shape {fp.shape} 알 수 없음 → post-fit skip')
    if c_I is not None:
        post_us = {k: c_I[k] * raw_us[k] for k in COMPS}
        print(f'[loaded] {NPZ_PATH}')
        print('  baseline c (Model I, 1-10 GeV mean): ' +
              ' '.join(f'{k}={np.mean(c_I[k][(E>=1)&(E<=10)]):.3f}' for k in COMPS))
else:
    print(f'[skip] {NPZ_PATH} 부재 → RAW (c=1) 만 출력')

m = (E >= 1) & (E <= 10)

def _print_table(label, our_dict):
    print(f'\n=== {label} / paper Fig 11 Model I — 14-bin × 5-comp ratio ===')
    print(f'{"bin":>3} {"E[GeV]":>8} | ' + ' | '.join(f'{k:>9}' for k in COMPS))
    print('-' * (15 + 12*len(COMPS)))
    for b in range(len(E)):
        line = f'{b:>3} {E[b]:>8.3f} |'
        for k in COMPS:
            pv = paper_at_us[k][b]
            r  = our_dict[k][b] / pv if (pv and np.isfinite(pv) and pv != 0) else np.nan
            line += f' {r:>9.4f} |'
        print(line)
    print(f'\n1-10 GeV stats per comp:')
    print(f'{"comp":>5} {"mean":>9} {"std":>9} {"min":>9} {"max":>9} {"max/min":>9}')
    for k in COMPS:
        rb = our_dict[k][m] / paper_at_us[k][m]
        rb = rb[np.isfinite(rb)]
        if rb.size == 0:
            print(f'  {k:>4}: (all NaN)')
            continue
        rmin, rmax = rb.min(), rb.max()
        print(f'{k:>5} {rb.mean():>9.4f} {rb.std():>9.4f} '
              f'{rmin:>9.4f} {rmax:>9.4f} {rmax/rmin:>9.3f}')

_print_table('RAW (c=1) template', raw_us)
if post_us is not None:
    _print_table('post-fit (baseline c × template)', post_us)

In [ ]:
#pre-phase 3
import os, numpy as np, re

GCEPY_TPL  = '/home/haebarg/GCE-Chi-square-fitting/gcepy/gcepy/inputs/templates_lowdim'
GCEPY_CODE = '/home/haebarg/GCE-Chi-square-fitting/gcepy/gcepy/lowdim_model.py'

# (a) paper template .npy 분포
print('=== paper template files inventory ===\n')
print(f'{"file":<60} {"shape":<22} {"dtype":<10} {"sum":<14} {"max":<12} {"nonzero":<12}')
paper_arrays = {}
for f in sorted(os.listdir(GCEPY_TPL)):
    arr = np.load(os.path.join(GCEPY_TPL, f))
    paper_arrays[f] = arr
    print(f'{f:<60} {str(arr.shape):<22} {str(arr.dtype):<10} {arr.sum():<14.3e} {arr.max():<12.3e} {int((arr!=0).sum()):<12}')

# (b) bin 7 spatial argmax 우리 mapcube/conv vs paper template
print(f'\n=== bin 7 spatial argmax (orientation check) ===\n')
for f, arr in paper_arrays.items():
    if arr.ndim < 3: continue
    slab = arr[7]
    am = np.unravel_index(np.argmax(np.abs(slab)), slab.shape)
    cy = slab.shape[0]/2 - 0.5; cx = slab.shape[1]/2 - 0.5
    dy, dx = am[0]-cy, am[1]-cx
    # CDELT 부호 가정 (paper RA convention): array x 증가 → l 감소 (CDELT1 < 0)
    # paper 가 우리와 같은 convention 이면 +l side argmax → dx < 0
    print(f'  {f[:55]:<55}  shape={slab.shape}  argmax={am}  Δ from center=({dy:+.0f}, {dx:+.0f})')

# (c) lowdim_model.py 의 prior, ROI, mask 정의 라인만 추출
print(f'\n\n=== lowdim_model.py 핵심 라인 (prior / ROI / mask / likelihood) ===\n')
with open(GCEPY_CODE) as f:
    src = f.read()
print(f'(total {src.count(chr(10))+1} lines)\n')

KEYWORDS = ('prior', 'bound', 'mask', 'roi', 'crop', 'window',
            'lnlike', 'lnprob', 'log_like', 'dist.', 'Uniform', 'LogUniform',
            'sigma_bub', 'sigma_iso', 'chi2', 'cube', 'def ', 'class ',
            'import', 'from ')

for i, ln in enumerate(src.split('\n'), 1):
    s = ln.strip()
    if not s or s.startswith('#'): continue
    low = s.lower()
    if any(re.search(rf'\b{re.escape(kw)}', low) for kw in KEYWORDS):
        if len(ln) < 180:
            print(f'{i:>4}: {ln}')

In [ ]:
# Phase 3A — 4-layer comparison (한 셀)
import os, numpy as np, yaml
GP = '/home/haebarg/GCE-Chi-square-fitting/gcepy/gcepy'

# (1) paper prior yaml
print('=== (1) paper prior bounds (lowdim_priors.yaml) ===')
with open(f'{GP}/inputs/priors/lowdim_priors.yaml') as f:
    py = yaml.safe_load(f)
print(yaml.dump(py, default_flow_style=False))

# (2) paper mask
print('\n=== (2) paper mask file inventory ===')
mask_dir = f'{GP}/inputs/utils'
if os.path.exists(mask_dir):
    for f in os.listdir(mask_dir):
        p = f'{mask_dir}/{f}'
        if f.endswith('.npy'):
            arr = np.load(p, mmap_mode='r')
            print(f'  {f}  shape={arr.shape}  dtype={arr.dtype}')

paper_mask_path = f'{GP}/inputs/utils/mask_4FGL-DR2_14_Ebin_20x20window_normal.npy'
if os.path.exists(paper_mask_path):
    pm = np.load(paper_mask_path)
    print(f'\npaper mask: shape={pm.shape}, dtype={pm.dtype}, unique={np.unique(pm)[:5]}')
    our_mask = np.load('./GC_analysis_DR2/Model/GC_mask_60x60_definitions_DR2.npy')
    # 우리 mask는 600×600. paper는 (14, 160000) flat 또는 (14, 400, 400)
    if pm.ndim == 2 and pm.shape == (14, 160000):
        pm_r = pm.reshape(14, 400, 400)
    elif pm.ndim == 3:
        pm_r = pm
    else:
        pm_r = None
        print(f'paper mask shape unknown')
    
    if pm_r is not None:
        b = 7
        our_crop = our_mask[b, 100:500, 100:500].astype(int)
        paper_b = pm_r[b].astype(int)
        print(f'\nbin {b} mask compare (cropped 400×400):')
        for name, px in [('id',     paper_b),
                         ('flip_l', paper_b[:, ::-1]),
                         ('flip_b', paper_b[::-1, :]),
                         ('flip_2', paper_b[::-1, ::-1])]:
            eq = (our_crop == px).sum()
            print(f'  {name:<10} {eq/our_crop.size*100:>6.2f}%')

# (3) paper likelihood code body (lines 124-165)
print('\n\n=== (3) paper jlnlike (lines 124-165) ===')
with open(f'{GP}/lowdim_model.py') as f:
    src = f.read().split('\n')
for i, ln in enumerate(src[123:165], start=124):
    print(f'{i:>4}: {ln}')

# (4) paper conv template — 우리 Model 7p/8t 와 직접 1:1 비교
print('\n\n=== (4) paper conv template direct binary diff ===')
print('(skip if we have no Model 7p/8t in results_12yr/)')
# 우리 80-model 결과 폴더에 model 7p, 8t 가 있나?
import glob
r12 = glob.glob('./results_12yr/GCE_model_*_front_12yr_cholis.dat')
naming_alts = ['7p', '8t', 'VII', 'VIII']   # NAMING_CONVENTION 결과 후 정확 매칭
for alt in naming_alts:
    hits = [r for r in r12 if f'_{alt}_' in r]
    print(f'  results with "_{alt}_": {len(hits)}')

In [ ]:
# Cell 8y — H1 root: srcmap FITS header diff (gtsrcmaps 호출 옵션 추적)
from astropy.io import fits
import difflib

def hdr_dump(p):
    out = []
    with fits.open(p) as hdul:
        for i, hdu in enumerate(hdul):
            out.append(f'### HDU {i}: {hdu.name}  ({hdu.data.shape if hdu.data is not None else "no data"})')
            for card in hdu.header.cards:
                out.append(str(card).rstrip())
    return out

for comp in ('pion', 'bremss', 'ics'):
    us_p = f'{US_DIR}/GC_{comp}_modelI_12yr_front_clean.fits'
    sg_p = f'{SG_DIR}/GC_{comp}_modelI_12yr_front_clean.fits'
    us_h = hdr_dump(us_p)
    sg_h = hdr_dump(sg_p)
    diff = [ln for ln in difflib.unified_diff(us_h, sg_h, lineterm='', n=0)
            if ln.startswith(('+', '-')) and not ln.startswith(('+++', '---'))]
    print(f'\n========== {comp}  ({len(diff)} differing lines) ==========')
    print(f'  us : {us_p}')
    print(f'  sg : {sg_p}')
    for ln in diff[:80]:   # max 80 줄
        print(f'  {ln}')
    if len(diff) > 80:
        print(f'  ... ({len(diff)-80} more lines)')

In [ ]:
import os, hashlib

def md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

pairs = [
    ('GC_pion_modelI_12yr_front_clean.fits',     'pion'),
    ('GC_bremss_modelI_12yr_front_clean.fits',   'bremss'),
    ('GC_ics_modelI_12yr_front_clean.fits',      'ics'),
    ('GC_Extended_srcmap_12yr_front_clean_model_I.fits', 'srcmap'),
]
print(f'{"file":<14} {"bk size":>12} {"cur size":>12} {"bk mtime":>12} {"cur mtime":>12} {"md5 same?":>10}')
for f, lbl in pairs:
    bk = f'/tmp/baseline_modelI/{f}'
    cu = f'./GC_analysis_DR2/{f}'
    if not (os.path.exists(bk) and os.path.exists(cu)):
        print(f'{lbl:<14}  missing: bk={os.path.exists(bk)} cu={os.path.exists(cu)}')
        continue
    bk_sz, cu_sz = os.path.getsize(bk), os.path.getsize(cu)
    bk_mt, cu_mt = int(os.path.getmtime(bk)), int(os.path.getmtime(cu))
    same = md5(bk) == md5(cu)
    print(f'{lbl:<14} {bk_sz:>12} {cu_sz:>12} {bk_mt:>12} {cu_mt:>12} {str(same):>10}')

# run_one_model.py 의 resample line 적용 확인
print('\n--- run_one_model.py: gtsrcmaps resample line ---')
import subprocess
out = subprocess.run(['grep', '-n', '-B1', '-A1', "resample", 'run_one_model.py'],
                     capture_output=True, text=True)
print(out.stdout if out.stdout else '(no resample line found)')

In [ ]:
import subprocess, glob, os

# 우리 XML: ./GC_analysis_DR2/Model/GC_Extended_modelI_test.xml
# Sang XML: 정확한 path 모름 → Model I XML candidate 검색
our_xml = './GC_analysis_DR2/Model/GC_Extended_modelI_test.xml'
print(f'--- 우리 XML : {our_xml} (size {os.path.getsize(our_xml)} bytes) ---\n')

# Sang XML 위치 자동 검색 (1099 XML 있다 했으니 Model I 만 좁히기)
print('--- Sang XML candidates (Model I) ---')
found = []
for base in ('/home/sanghwan/FermiLAT/Sanghwan/GC_analysis',
             '/home/sanghwan/FermiLAT/Sanghwan'):
    out = subprocess.run(
        ['find', base, '-maxdepth', '4', '-name', 'GC_Extended_modelI*.xml',
         '-not', '-name', '*_l*.xml'],     # cov 의 per-ROI XML 제외
        capture_output=True, text=True, timeout=10)
    for ln in out.stdout.strip().split('\n'):
        if ln and ln not in found:
            found.append(ln)
            print(f'  {os.path.getsize(ln):>8} bytes  {ln}')

# 가장 main 후보 (size 비슷, _l 없음, Model 또는 GC_analysis 하위)
if found:
    print(f'\n--- diff 우리 vs Sang (첫 candidate) ---')
    sg_xml = found[0]
    diff = subprocess.run(['diff', '-u', our_xml, sg_xml],
                          capture_output=True, text=True, timeout=10)
    out = diff.stdout if diff.stdout else '(identical)'
    # 너무 길면 처음 100줄만
    lines = out.split('\n')
    print('\n'.join(lines[:120]))
    if len(lines) > 120:
        print(f'... ({len(lines)-120} more lines)')

# map_based_integral 명시 여부 직접 확인
print(f'\n--- map_based_integral / SpatialFunction 정의 카운트 ---')
for label, p in [('us', our_xml)] + ([('sg', found[0])] if found else []):
    with open(p) as f:
        content = f.read()
    print(f'{label}: map_based_integral 등장 {content.count("map_based_integral")}회, '
          f'MapCubeFunction {content.count("MapCubeFunction")}회, '
          f'total size {len(content)} chars')

In [ ]:
from astropy.io import fits
import numpy as np, os

US_DIR_MC = './MapCubes'
SG_XML_MC = '/home/sanghwan/FermiLAT/Mapcube_convert_test'   # Sang XML 이 실제 가리키는 곳
SG_V12D   = '/home/sanghwan/FermiLAT/Fermi-LAT-GCE-Analysis/Fermi-LAT-Galactic-Center-Excess-analysis/Model_maps/GDE_maps/GALPROP_Mapcubes/Mapcubes'

# Sang V12d-path 의 실제 file 이름 확인
print('--- Sang V12d-path file listing (Pion sample) ---')
import subprocess
for sub in ('Pion', 'Bremss', 'ICs'):
    out = subprocess.run(['ls', f'{SG_V12D}/{sub}/'], capture_output=True, text=True)
    print(f'{sub}: {out.stdout.split()[:5]} ...' if out.stdout else f'{sub}: (empty)')

def load(p):
    return fits.open(p)[0].data

us_paths = {c: f'{US_DIR_MC}/{c}_mapcube_modelI.fits' for c in ('pion', 'bremss', 'ics')}
sg_xml_paths = {c: f'{SG_XML_MC}/{c}_mapcube_modelI.gz' for c in ('pion', 'bremss', 'ics')}

print(f'\n--- 우리 vs Sang(XML actual = Mapcube_convert_test) ---')
print(f'{"comp":<8} {"path exists":<14} {"shape":>20} {"sum ratio":>14} {"max |diff|":>14} {"identical?":>12}')
for c in ('pion', 'bremss', 'ics'):
    if not os.path.exists(sg_xml_paths[c]):
        print(f'{c:<8} {"MISSING":<14} {sg_xml_paths[c]}')
        continue
    us = load(us_paths[c]).astype(np.float64)
    sg = load(sg_xml_paths[c]).astype(np.float64)
    shape_match = us.shape == sg.shape
    if not shape_match:
        print(f'{c:<8} {"SHAPE DIFF":<14} us={us.shape} sg={sg.shape}')
        continue
    rel_diff = np.abs(us - sg).max() / np.abs(us).max()
    same = bool(rel_diff < 1e-6)
    print(f'{c:<8} {"OK":<14} {str(us.shape):>20} {us.sum()/sg.sum():>14.6f} {np.abs(us-sg).max():>14.4e} {str(same):>12}')

# bin 7 only 비교 (gtsrcmaps 가 처리하는 단일 bin)
print(f'\n--- bin 7 pixel-level (mapcube bin 7 ≠ srcmap bin 7 이지만 PB-only signal 확인) ---')
# mapcube 의 38-bin 중 1.96 GeV 근처 bin
for c in ('pion', 'bremss', 'ics'):
    if not os.path.exists(sg_xml_paths[c]):
        continue
    us = load(us_paths[c])
    sg = load(sg_xml_paths[c])
    # mapcube 의 bin 7 (대응 energy 가 다를 수 있지만 spatial shape compare 만)
    us_b, sg_b = us[7].astype(np.float64), sg[7].astype(np.float64)
    a = us_b - us_b.mean()
    b = sg_b - sg_b.mean()
    corr = float((a*b).sum() / np.sqrt((a**2).sum() * (b**2).sum()))
    print(f'  {c}: corr us↔sg(actual) = {corr:.6f}')

In [ ]:
import numpy as np
from astropy.io import fits

us_paths = {c: f'./MapCubes/{c}_mapcube_modelI.fits' for c in ('pion', 'bremss', 'ics')}
sg_paths = {c: f'/home/sanghwan/FermiLAT/Mapcube_convert_test/{c}_mapcube_modelI.gz'
            for c in ('pion', 'bremss', 'ics')}

def cor(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a*b).sum() / np.sqrt((a**2).sum() * (b**2).sum()))

print(f'{"comp":<8} {"raw corr":>10} {"corr(us[::-1], sg)":>22} {"max |diff|(flipped)":>22}')
for c in ('pion', 'bremss', 'ics'):
    us = fits.open(us_paths[c])[0].data.astype(np.float64)
    sg = fits.open(sg_paths[c])[0].data.astype(np.float64)
    us_flipped = us[:, :, ::-1]
    # bin 7 만
    raw_corr = cor(us[7], sg[7])
    flp_corr = cor(us_flipped[7], sg[7])
    max_diff = np.abs(us_flipped - sg).max() / max(np.abs(us).max(), 1e-30)
    print(f'{c:<8} {raw_corr:>10.6f} {flp_corr:>22.6f} {max_diff:>22.4e}')

# 만약 corr(flipped) ≈ 1.0 → axis=2 flip 단독이 binary 차이의 전부
# 만약 corr(flipped) < 1.0 → flip 외 추가 변환 존재

In [ ]:
from astropy.io import fits
import numpy as np

# (a) WCS 비교
print('=== WCS card comparison ===\n')
files = [
    ('us mapcube',  './MapCubes/pion_mapcube_modelI.fits'),
    ('sg mapcube',  '/home/sanghwan/FermiLAT/Mapcube_convert_test/pion_mapcube_modelI.gz'),
    ('us ccube',    './GC_analysis_DR2/GC_ccube_12yr_front_clean.fits'),
    ('sg ccube',    '/home/sanghwan/FermiLAT/Sanghwan/GC_analysis/GC_ccube_12yr_front_clean.fits'),
    ('us srcmap I', './GC_analysis_DR2/GC_pion_modelI_12yr_front_clean.fits'),
    ('sg srcmap I', '/home/sanghwan/FermiLAT/Sanghwan/GC_analysis/GC_pion_modelI_12yr_front_clean.fits'),
]
keys = ('NAXIS1','CRPIX1','CRVAL1','CDELT1','CTYPE1',
        'NAXIS2','CRPIX2','CRVAL2','CDELT2','CTYPE2')
print(f'{"file":<14}  ' + '  '.join(f'{k:>9}' for k in keys))
for label, p in files:
    h = fits.open(p)[0].header
    vals = []
    for k in keys:
        v = h.get(k, '?')
        if isinstance(v, float): v = f'{v:>9.4g}'
        else: v = f'{str(v):>9}'
        vals.append(v)
    print(f'{label:<14}  ' + '  '.join(vals))

# (b) mask convention V44 적용 지점 추적
print('\n\n=== mask load + flip 적용 지점 grep ===\n')
import subprocess
for pat in (r'np\.flip.*mask', r'flip.*axis.*2', r'mask.*flip',
            r'GC_mask_60x60', r'psc_mask\s*='):
    out = subprocess.run(['grep', '-rn', '-l', pat,
                          'cholis_masking.py', 'run_one_model.py', 'prepare_common.py'],
                         capture_output=True, text=True)
    if out.stdout.strip():
        print(f'pattern: {pat}')
        for f in out.stdout.strip().split('\n'):
            sub = subprocess.run(['grep', '-n', pat, f], capture_output=True, text=True)
            print(sub.stdout, end='')
        print()

In [ ]:
from astropy.io import fits
import numpy as np, subprocess, os

# (A) GC argmax position — 모든 spatial array 의 GC peak 위치 확인
print('=== GC peak array index check (WCS-predicted vs argmax) ===\n')
files = [
    ('us mapcube pion b7',    './MapCubes/pion_mapcube_modelI.fits', 7),
    ('sg mapcube pion b7',    '/home/sanghwan/FermiLAT/Mapcube_convert_test/pion_mapcube_modelI.gz', 7),
    ('us mapcube ics b7',     './MapCubes/ics_mapcube_modelI.fits', 7),
    ('us ccube b7',           './GC_analysis_DR2/GC_ccube_12yr_front_clean.fits', 7),
    ('us srcmap pion b7',     './GC_analysis_DR2/GC_pion_modelI_12yr_front_clean.fits', 7),
    ('us GCE_template_NFW2',  './GCE_template_NFW2.fits', None),
    ('us bubble template',    './Fermi_Bubbles_template.fits', None),
]
print(f'{"file":<24} {"shape":<22} {"GC pred (y,x)":<14} {"argmax (y,x)":<14} {"on-GC?":<8}')
for label, p, b in files:
    if not os.path.exists(p):
        print(f'{label:<24} MISSING')
        continue
    hdu = fits.open(p)[0]
    h, d = hdu.header, hdu.data
    slab = d[b] if (b is not None and d.ndim >= 3) else (d if d.ndim == 2 else d[0])
    gc_pred = (h['CRPIX2'] - 1, h['CRPIX1'] - 1)
    am = np.unravel_index(np.argmax(slab), slab.shape)
    dy, dx = am[0] - gc_pred[0], am[1] - gc_pred[1]
    ok = abs(dy) < 10 and abs(dx) < 10
    print(f'{label:<24} {str(slab.shape):<22} {f"({gc_pred[0]:.0f},{gc_pred[1]:.0f})":<14} '
          f'{str(am):<14} {str(ok):<8}  Δ=({dy:+.0f},{dx:+.0f})')

# (B) np.flip 흔적 — prepare/template/mask 코드 전체
print('\n\n=== np.flip / axis=2 grep (모든 .py / 최근 사용 .ipynb 제외) ===\n')
out = subprocess.run(['grep', '-rn', '--include=*.py',
                      '-E', r'np\.flip|flip.*axis|\[::-1\]',
                      '.'],
                     capture_output=True, text=True)
for ln in out.stdout.strip().split('\n'):
    if 'archive' in ln or '__pycache__' in ln: continue
    print(f'  {ln}')

# (C) PSC mask 의 source 분포 — array center 영역에서 mask 가 어떻게 작동
print('\n\n=== PSC mask source positions (bin 7, full 600×600) ===')
pm = np.load('./GC_analysis_DR2/Model/GC_mask_60x60_definitions_DR2.npy')
b7 = pm[7]
print(f'shape: {pm.shape}, dtype: {pm.dtype}, unique: {np.unique(b7)}')
# Sgr A* (l=0, b=0) = (300, 300). bright PSC ~5° 안 = array (250-350, 250-350)
# mask convention: 1=keep, 0=mask? 확인
n_kept_around_gc = int(b7[250:350, 250:350].sum())
n_total_around_gc = (b7[250:350, 250:350].size)
print(f'inner 10°×10° box (250-350, 250-350): kept={n_kept_around_gc}/{n_total_around_gc}'
      f' = {n_kept_around_gc/n_total_around_gc:.3f}')

In [ ]:
from astropy.io import fits
import numpy as np, os, glob

# paper raw flux map 경로 (build_mapcubes.py 도큐먼테이션 기준)
paper_dirs = [
    '../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg',
    './GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg',
    '/home/haebarg/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg',
]
paper_dir = next((d for d in paper_dirs if os.path.exists(d)), None)
print(f'paper_dir: {paper_dir}\n')

# Model I = 'bs' suffix (NAMING_CONVENTION)
patterns = [
    ('pi0',    'pi0_bs_*.fits'),
    ('bremss', 'bremss_bs_*.fits'),
    ('ICS',    'ICS_bs_*.fits'),
]
print(f'{"comp":<8} {"path":<60} {"shape":<22} {"GC pred":<10} {"argmax (b=any)":<18}')

for comp, pat in patterns:
    files = sorted(glob.glob(f'{paper_dir}/{pat}'))
    if not files:
        print(f'{comp:<8} (no match for {pat})')
        continue
    p = files[0]
    hdu = fits.open(p)[0]
    h, d = hdu.header, hdu.data
    gc_pred = (h.get('CRPIX2', '?'), h.get('CRPIX1', '?'))
    # paper raw 는 38-bin (50-814008 MeV), bin 7 근처는 1.9 GeV bin
    # 가장 강한 spatial signal 보이는 low-E bin (paper raw 에서 inner disk dominant)
    for b in (5, 7, 10):
        slab = d[b] if d.ndim >= 3 else d
        am = np.unravel_index(np.argmax(slab), slab.shape)
        if b == 5:
            print(f'{comp:<8} {p.split("/")[-1]:<60} {str(slab.shape):<22} '
                  f'{f"({gc_pred[0]:.0f},{gc_pred[1]:.0f})":<10} '
                  f'bin{b}: {str(am):<14} Δ=({am[0]-gc_pred[0]:+.0f},{am[1]-gc_pred[1]:+.0f})')
        else:
            print(f'{"":<8} {"":<60} {"":<22} {"":<10} '
                  f'bin{b}: {str(am):<14} Δ=({am[0]-gc_pred[0]:+.0f},{am[1]-gc_pred[1]:+.0f})')
    print(f'{"":<8} CDELT1={h.get("CDELT1","?")}, CTYPE1={h.get("CTYPE1","?")}\n')

# WCS-level orientation 비교
print('--- 우리 mapcube vs paper raw — WCS card check ---')
for label, p in [('us', './MapCubes/pion_mapcube_modelI.fits'),
                 ('paper', files[0] if patterns and (files := sorted(glob.glob(f'{paper_dir}/pi0_bs_*.fits'))) else None)]:
    if p is None: continue
    h = fits.open(p)[0].header
    print(f'{label}: CDELT1={h["CDELT1"]:+.4f}  CRPIX1={h["CRPIX1"]:.1f}  CRVAL1={h["CRVAL1"]:.1f}  CTYPE1={h["CTYPE1"]}')

In [ ]:
import numpy as np, glob, os
import matplotlib.pyplot as plt

npz_path = './results_12yr/GCE_model_I_front_12yr_cholis_fit.npz'
d = np.load(npz_path, allow_pickle=True)
print(f'=== {npz_path} ===')
print(f'keys: {list(d.keys())}\n')
for k in d.keys():
    v = d[k]
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
        if v.dtype.kind in 'fi' and v.size < 30:
            print(f'    values: {v}')
    else:
        print(f'  {k}: {type(v)} = {v}')

# bin 7 의 (c_PB, c_ICS) chain scatter — multi-modal 여부 single test
# convention: chain shape (nwalkers × nsteps, ndim) flat 또는 (nbins, nwalkers, nsteps, ndim)
print('\n=== chain 구조 추정 + bin 7 scatter ===')
for cand_key in ('chain', 'samples', 'flat_chain', 'fit_params', 'params'):
    if cand_key in d.keys():
        arr = d[cand_key]
        print(f'  using key "{cand_key}" with shape {arr.shape}')
        break

# 위 print 결과 본 후 정확한 indexing 결정 — 일단 추측해서 plot
# 가장 common: shape (n_bins, n_samples, n_params) 또는 dict 형태

In [ ]:
import numpy as np

d = np.load('./results_12yr/GCE_model_I_front_12yr_cholis_fit.npz')
fp, fm, fs = d['fitted_params'], d['fitted_params_median'], d['fitted_params_std']
fl, fu = d['fitted_params_lower'], d['fitted_params_upper']
E = d['E']

names = ['PB', 'ICS', 'GCE', 'Bub', 'Iso']
print('Proxy A — best vs median (1σ-normalized) — large = posterior skew\n')
print(f'{"bin":>3} {"E (GeV)":>9} | ' + ' | '.join(f'{n:>20}' for n in names))
print(f'{"":>3} {"":>9} | ' + ' | '.join(f'{"best/med (Δ/σ)":>20}' for _ in names))
for b in range(14):
    row = f'{b:>3} {E[b]:>9.3f} | '
    for c in range(5):
        skew = (fp[c,b] - fm[c,b]) / fs[c,b] if fs[c,b] > 0 else 0
        row += f' {fp[c,b]:>6.3f}/{fm[c,b]:>5.3f} ({skew:>+5.2f}σ) | '
    print(row)

print('\nProxy B — 1σ range (upper-lower) / best — large = poorly constrained\n')
print(f'{"bin":>3} {"E (GeV)":>9} | ' + ' | '.join(f'{n:>10}' for n in names))
for b in range(14):
    row = f'{b:>3} {E[b]:>9.3f} | '
    for c in range(5):
        rng = (fu[c,b] - fl[c,b]) / max(abs(fp[c,b]), 1e-30)
        row += f' {rng:>9.3f} | '
    print(row)

print('\nProxy C — σ asymmetry (PB,ICS vs GCE,Bub,Iso)\n')
print(f'{"bin":>3} {"E (GeV)":>9}   {"σ(PB)":>8} {"σ(ICS)":>8} {"σ(GCE)":>8} {"σ(Bub)":>8} {"σ(Iso)":>8}  {"σ(PB+ICS)/σ(rest)":>20}')
for b in range(14):
    pb_ics = (fs[0,b] + fs[1,b]) / 2
    rest   = (fs[2,b] + fs[3,b] + fs[4,b]) / 3
    asym   = pb_ics / rest if rest > 0 else 0
    print(f'{b:>3} {E[b]:>9.3f}   {fs[0,b]:>8.4f} {fs[1,b]:>8.4f} {fs[2,b]:>8.4f} {fs[3,b]:>8.4f} {fs[4,b]:>8.4f}  {asym:>20.3f}')

In [ ]:
import numpy as np
from astropy.io import fits

# 우리 array 세 개 + WCS
ccube = fits.open('./GC_analysis_DR2/GC_ccube_12yr_front_clean.fits')[0]
psc_mask = np.load('./GC_analysis_DR2/Model/GC_mask_60x60_definitions_DR2.npy')
mapcube = fits.open('./MapCubes/pion_mapcube_modelI.fits')[0]

# 모두 같은 WCS-convention 일 거라는 가정 검증
print('=== WCS convention 비교 ===')
for label, hdu in [('mapcube', mapcube), ('ccube', ccube)]:
    h = hdu.header
    print(f'{label:>10}: NAXIS1={h["NAXIS1"]} CRPIX1={h["CRPIX1"]:.1f} '
          f'CRVAL1={h["CRVAL1"]:.1f} CDELT1={h["CDELT1"]:+.3f} CTYPE1={h["CTYPE1"]}')
print(f'{"psc_mask":>10}: shape={psc_mask.shape} (assumes same WCS as ccube)')

# bin 7 에서 mask 가 0 인 pixel cluster 찾기 → bright PSC 의 array 위치
b = 7
m = psc_mask[b]
masked = (m == 0)
print(f'\nbin {b}: masked pixels = {masked.sum()} / {m.size}')

# masked cluster 의 (y, x) 위치 — connected components 또는 단순 argmax
# 가장 큰 masked region 의 centroid: mask=0 인 곳의 (y_mean, x_mean)
ys, xs = np.where(masked)
# disk 마스크 (b<2°, y=290~310) 제외 + 큰 source 위치 만
mask_no_disk = masked.copy()
mask_no_disk[290:310, :] = False
ys2, xs2 = np.where(mask_no_disk)

# 가장 GC 에서 먼 + 큰 masked cluster
# 단순 방법: 각 source 가 ~5-10 pixel radius. histogram 으로 cluster 찾기
print(f'\nbin {b} masked pixels (disk 제외): {mask_no_disk.sum()}')
if mask_no_disk.sum() > 0:
    # GC = (300, 300). 가장 멀리 + 한 region 의 centroid 찾기 위해 distance 분포
    dy = ys2 - 300; dx = xs2 - 300
    r = np.sqrt(dy**2 + dx**2)
    # top 5 distance pixel
    idx_far = np.argsort(r)[-20:]
    print(f'top 20 farthest masked pixels (y, x, b_deg, l_deg):')
    cdelt = ccube.header['CDELT1']   # 보통 -0.1 (음수)
    for i in idx_far[::5]:
        y, x = ys2[i], xs2[i]
        b_deg = (y - 299.5) * 0.1   # CRPIX2=300.5, CDELT2=+0.1
        l_deg = (x - 299.5) * cdelt # CRPIX1=300.5, CDELT1=-0.1
        print(f'  array ({y:>3},{x:>3})  →  (b={b_deg:+.2f}°, l={l_deg:+.2f}°)')

# 이 source 들이 4FGL catalog 에서 실제 같은 위치인지 cross-check
# (수동: 위 출력 l_deg 부호 + b_deg 부호가 4FGL catalog ROI 안 bright source 위치와 일치하면 mask 와 ccube WCS 정합)
print('\n→ 위 array (y,x) 에서 변환된 (l,b) 가 실제 4FGL catalog 의 known bright source 위치와 정합하는지 수동 확인')
print('→ 정합하면 mask ↔ ccube WCS 일관 (CDELT1 부호 의미 그대로 사용)')
print('→ l 부호가 반대로 나오면 mask 가 mapcube 와 다른 orientation = silent flip')

# 같은 (y, x) 에서 우리 mapcube 0.25° grid index 환산 후 mapcube 값 확인
print('\n=== 가장 먼 masked pixel @ mapcube (downsample 0.1° → 0.25°) ===')
if mask_no_disk.sum() > 0:
    y, x = ys2[idx_far[-1]], xs2[idx_far[-1]]
    # 0.1° → 0.25° array index: ccube (300.5, CDELT -0.1) ↔ mapcube (120.5, CDELT -0.25)
    # 같은 sky 위치의 mapcube index
    l_deg = (x - 299.5) * (-0.1)   # x_ccube → l
    b_deg = (y - 299.5) * (+0.1)
    x_mc = int(round(l_deg / (-0.25) + 119.5))
    y_mc = int(round(b_deg / (+0.25) + 119.5))
    print(f'ccube (y={y}, x={x}) ↔ sky (l={l_deg:+.2f}°, b={b_deg:+.2f}°) '
          f'↔ mapcube (y={y_mc}, x={x_mc})')
    print(f'  ccube value here: {ccube.data[b, y, x]:.2e}')
    print(f'  mapcube value:    {mapcube.data[b, y_mc, x_mc]:.4e}')
    # mapcube 가 그 위치에서 enhanced (bright source 는 mapcube 에 없지만 disk gas 있으면 +)
    # 더 결정적: 우리 mapcube +l 쪽 vs -l 쪽 의 ratio = paper raw 와 같은 부호인지
    bin_mc = 7   # mapcube bin
    pos_l = mapcube.data[bin_mc, 119, 100:120].mean()  # x=100-119 = +l side
    neg_l = mapcube.data[bin_mc, 119, 120:140].mean()  # x=120-139 = -l side
    print(f'\n=== mapcube +l vs -l asymmetry (bin {bin_mc}, b=0 row) ===')
    print(f'  +l side (x=100-120) mean = {pos_l:.4e}')
    print(f'  -l side (x=120-140) mean = {neg_l:.4e}')
    print(f'  ratio +l/-l = {pos_l/neg_l:.3f}')
    # paper raw 의 같은 asymmetry 와 비교
    paper = fits.open('../GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg/pi0_bs_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')[0].data
    pp = paper[bin_mc, 119, 100:120].mean()
    pn = paper[bin_mc, 119, 120:140].mean()
    print(f'  paper +l/-l = {pp/pn:.3f}  → 우리 와 같은 부호면 우리 mapcube paper-aligned')

In [ ]:
import os, numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

files = [
    ('CURRENT (v2 flat-ellipse)',    './Fermi_Bubbles_template.fits'),
    ('OLD V49 digitized',            './Fermi_Bubbles_template.fits.ROLLBACK_OLD_digitized'),
    ('paperfaithful_ellipse_backup', './Fermi_Bubbles_template.fits.paperfaithful_ellipse_backup'),
    ('fig15_normalize_broken',       './Fermi_Bubbles_template.fits.fig15_normalize_broken'),
]

print('=== Bubble template diagnostics (b<5° gap check) ===\n')
print(f'{"file":<35} {"shape":<14} {"argmax":<14} {"|b|<5° / total":<18} {"north(b>5)":<12} {"south(b<-5)":<12} {"|b|<5° empty?":<14}')
for label, p in files:
    if not os.path.exists(p):
        print(f'{label:<35} MISSING'); continue
    d = fits.open(p)[0].data
    slab = d[0] if d.ndim == 3 else d
    am = np.unravel_index(np.argmax(slab), slab.shape)
    # CRPIX2=300.5, CDELT2=+0.1 → row 295-305 = |b|<0.5°, row 250-350 = |b|<5°
    inner_b = slab[250:350].sum()
    north   = slab[350:].sum()      # b > +5°
    south   = slab[:250].sum()      # b < -5°
    total   = slab.sum()
    gap_ok  = inner_b / total < 0.1 if total > 0 else False
    print(f'{label:<35} {str(slab.shape):<14} {str(am):<14} '
          f'{inner_b/total:.3f}            {north/total:.3f}        {south/total:.3f}        {gap_ok}')

# visualize
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, (label, p) in zip(axs, files):
    if not os.path.exists(p):
        ax.set_title(f'{label[:25]}\nMISSING'); continue
    d = fits.open(p)[0].data
    slab = d[0] if d.ndim == 3 else d
    im = ax.imshow(slab, origin='lower', extent=[30, -30, -30, 30],
                   cmap='inferno', aspect='equal')
    ax.set_title(label[:30], fontsize=9)
    ax.set_xlabel('l [deg]'); ax.set_ylabel('b [deg]')
    ax.axhline(+5, c='cyan', lw=0.5, ls='--')
    ax.axhline(-5, c='cyan', lw=0.5, ls='--')
    ax.axvline(0, c='cyan', lw=0.3)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('./bubble_templates_compare.png', dpi=120, bbox_inches='tight')
print('\nsaved: bubble_templates_compare.png')
print('\npaper Fig 12 / Su+2010 Fig 1 expected: dumbbell (북 lobe |b|>5° + 남 lobe |b|<-5°), |b|<5° gap')
print('→ "|b|<5° / total" 값이 작아야 (< 0.1) paper-faithful')
print('→ argmax y 가 250-350 사이 (|b|<5°) 면 gap 없는 단일 ellipse = paper deviation')

In [ ]:
from astropy.io import fits
import numpy as np

raw_p = './Fermi_Bubbles_template.fits'
out_p = './GC_analysis_DR2/GC_fermi_bubble_model_12yr_front_clean.fits'

raw_hdu = fits.open(raw_p)[0]
out_hdu = fits.open(out_p)[0]
raw_d, out_d = raw_hdu.data, out_hdu.data

# (1) raw data distribution — binary or all-nonzero?
print('=== (1) raw template data distribution ===')
print(f'shape: {raw_d.shape},  dtype: {raw_d.dtype}')
print(f'min: {raw_d.min():.4e}   max: {raw_d.max():.4e}   sum: {raw_d.sum():.4f}')
print(f'exactly zero pixels: {(raw_d == 0).sum()} / {raw_d.size}')
print(f'< 1e-7  pixels: {(raw_d < 1e-7).sum()}')
print(f'< 1e-10 pixels: {(raw_d < 1e-10).sum()}')
print(f'unique values (first 10): {np.unique(raw_d)[:10]}')
print(f'unique value count: {len(np.unique(raw_d))}')

# histogram
print(f'\nvalue histogram (10 log-spaced bins from min to max):')
mn = max(raw_d.min(), 1e-30)
if raw_d.max() > mn:
    bins = np.logspace(np.log10(mn), np.log10(raw_d.max()), 11)
    hist, _ = np.histogram(raw_d, bins=bins)
    for i, h in enumerate(hist):
        print(f'  [{bins[i]:.2e}, {bins[i+1]:.2e}]: {h:>8}')

# (2) WCS card 우리 raw vs gtmodel out 비교
print('\n=== (2) WCS card raw vs gtmodel ===')
for k in ('NAXIS', 'NAXIS1', 'NAXIS2', 'NAXIS3', 'CRPIX1', 'CRPIX2',
          'CRVAL1', 'CRVAL2', 'CDELT1', 'CDELT2', 'CTYPE1', 'CTYPE2'):
    r = raw_hdu.header.get(k, '—')
    o = out_hdu.header.get(k, '—')
    print(f'  {k:>8}   raw: {str(r):<14}   out: {str(o):<14}')

# (3) lobe vs background contrast
print('\n=== (3) lobe vs background contrast (bin-free 2D raw) ===')
slab = raw_d[0] if raw_d.ndim == 3 else raw_d
# paper Fig 12: 북 lobe centroid ≈ (l=0, b=+15°), 남 lobe ≈ (l=0, b=-15°)
# array idx (CRPIX2=300.5, CDELT2=+0.1): b=+15° → y≈450, b=-15° → y≈150
north = slab[440:460, 290:310].mean()   # 북 lobe 중심
south = slab[140:160, 290:310].mean()   # 남 lobe 중심
inner = slab[295:305, 295:305].mean()   # GC inner
edge  = slab[0:10,    0:10  ].mean()    # ROI corner (background)
print(f'  north lobe center:  {north:.4e}')
print(f'  south lobe center:  {south:.4e}')
print(f'  inner |b|<0.5°:     {inner:.4e}')
print(f'  ROI corner:         {edge:.4e}')
print(f'  lobe / background ratio: {north / max(edge, 1e-30):.3f}')
print(f'  lobe / inner ratio:      {north / max(inner, 1e-30):.3f}')

In [ ]:
import numpy as np, glob, os

our_p = './GC_analysis_DR2/Model/GC_mask_60x60_definitions_DR2.npy'
cands = glob.glob('/home/sanghwan/FermiLAT/Sanghwan/**/GC_mask_60x60*.npy', recursive=True)
print('Sang mask candidates:')
for c in cands: print(f'  {os.path.getsize(c):>10}  {c}')

# 우리 600×600 → fit-time cropping [100:500, 100:500] = 400×400
our_full = np.load(our_p)
print(f'\n우리 mask full shape: {our_full.shape}')

# Sang 후보 중 (14, 400, 400) shape 인 것 선택
sg_full = None
for c in cands:
    arr = np.load(c)
    print(f'  shape: {arr.shape}  ← {c}')
    if arr.shape == (14, 400, 400):
        sg_full = arr; sg_path = c; break

if sg_full is None:
    print('\nSang (14, 400, 400) mask 없음')
else:
    print(f'\n선택된 Sang mask: {sg_path}')
    # 우리 cropped == Sang 직접 비교
    b = 7
    our_crop = our_full[b, 100:500, 100:500].astype(int)
    sg_b     = sg_full[b].astype(int)
    
    print(f'\nbin {b} cropped mask comparison (shape {our_crop.shape} vs {sg_b.shape}):')
    for name, sg_x in [('identical',         sg_b),
                       ('us == flip_l (axis=1 of cropped)', sg_b[:, ::-1]),
                       ('us == flip_b (axis=0)',            sg_b[::-1, :]),
                       ('us == flip_both',                  sg_b[::-1, ::-1])]:
        eq = (our_crop == sg_x).sum()
        pct = eq / our_crop.size * 100
        print(f'  {name:<40} {eq:>8} / {our_crop.size}  ({pct:.2f}%)')

    # source l-asymmetry
    def l_asym(m, crpix_x):
        ys, xs = np.where(m == 0)
        keep = (ys < (crpix_x - 10)) | (ys > (crpix_x + 10))
        xs2 = xs[keep]
        return (xs2 < crpix_x).sum(), (xs2 > crpix_x).sum()
    
    up, un = l_asym(our_crop, 199.5)  # cropped: CRPIX1 effectively 200 - 100 + 0.5 = ?
    sp, sn = l_asym(sg_b,     199.5)
    print(f'\nmask source l-distribution (disk 제외):')
    print(f'  우리 cropped:  +l = {up:>6}   -l = {un:>6}   asym +l/-l = {up/max(un,1):.3f}')
    print(f'  Sang:          +l = {sp:>6}   -l = {sn:>6}   asym +l/-l = {sp/max(sn,1):.3f}')

# 우리 cropping window 의 sky 위치 확인 (CRPIX1=300.5, CDELT1=-0.1 기준)
# array x=100 → l = (100 - 299.5) × (-0.1) = +19.95°
# array x=500 → l = (500 - 299.5) × (-0.1) = -20.05°
# → cropped 400×400 = l ∈ [+20°, -20°], b ∈ [-20°, +20°]
print(f'\n우리 cropping window [100:500, 100:500] 의 sky 범위:')
print(f'  l: array 100 → +19.95°, array 499 → -20.05° (l ∈ [+20°, -20°])')
print(f'  b: array 100 → -19.95°, array 499 → +20.05° (b ∈ [-20°, +20°])')
print(f'  → fit-effective ROI = 40°×40° (= Sang native 와 동일 sky 범위)')

In [ ]:
import numpy as np, os, glob

our = np.load('./GC_analysis_DR2/Model/GC_mask_60x60_definitions_DR2.npy')   # (14, 600, 600)
b = 7
u_b_400 = our[b, 100:500, 100:500].astype(int)   # cropped 400×400
u_b_600 = our[b].astype(int)                      # full 600×600

def compare(u, s, label):
    print(f'  {label:<55}', end='')
    for name, sx in [('id',       s),
                     ('flip_l',   s[:, ::-1]),
                     ('flip_b',   s[::-1, :]),
                     ('flip_2',   s[::-1, ::-1])]:
        eq = (u == sx).sum()
        print(f'  {name}={eq/u.size*100:>6.2f}%', end='')
    print()

print(f'=== bin {b} mask compare (우리 14-bin slice → Sang 17-bin bin {b}) ===\n')
print('header: identical / flip_l (axis=1) / flip_b (axis=0) / flip_both\n')

for p in sorted(glob.glob('/home/sanghwan/FermiLAT/Sanghwan/GC_analysis/Model/GC_mask_60x60_definitions*.npy')):
    if '_l' in os.path.basename(p).replace('_DR', '_xR'): continue
    if '_disk' in os.path.basename(p): continue
    if '_test' in os.path.basename(p): continue
    try:
        s = np.load(p, mmap_mode='r')
        s_b = np.asarray(s[b]).astype(int)
        u = u_b_400 if s_b.shape == (400, 400) else (u_b_600 if s_b.shape == (600, 600) else None)
        if u is None:
            print(f'  {os.path.basename(p):<55}  shape mismatch {s_b.shape}')
            continue
        compare(u, s_b, f'{os.path.basename(p)} {s_b.shape}')
    except Exception as e:
        print(f'  {os.path.basename(p):<55}  ERROR: {e}')

# l-asymmetry (가장 match 율 높은 variant 와 우리)
print(f'\n=== bin {b} PSC l-asym (disk 제외, mask=0 픽셀) ===')
def l_asym(m, crpix_x):
    ys, xs = np.where(m == 0)
    keep = (ys < crpix_x - 10) | (ys > crpix_x + 10)
    xs2 = xs[keep]
    return (xs2 < crpix_x).sum(), (xs2 > crpix_x).sum()

up_400, un_400 = l_asym(u_b_400, 199.5)
up_600, un_600 = l_asym(u_b_600, 299.5)
print(f'  우리 cropped:  +l={up_400}  -l={un_400}  +l/-l={up_400/max(un_400,1):.3f}')
print(f'  우리 full:     +l={up_600}  -l={un_600}  +l/-l={up_600/max(un_600,1):.3f}')
for p in sorted(glob.glob('/home/sanghwan/FermiLAT/Sanghwan/GC_analysis/Model/GC_mask_60x60_definitions*.npy')):
    if any(k in os.path.basename(p) for k in ('_l', '_disk', '_test')):
        if '_DR' not in os.path.basename(p): continue
    try:
        s = np.load(p, mmap_mode='r')
        s_b = np.asarray(s[b]).astype(int)
        crpix = 199.5 if s_b.shape == (400, 400) else 299.5
        sp, sn = l_asym(s_b, crpix)
        print(f'  {os.path.basename(p)[:50]:<50}  +l={sp}  -l={sn}  +l/-l={sp/max(sn,1):.3f}')
    except Exception:
        pass

In [ ]:
# Cell 8z — XML file inventory: model XMLs 위치 + size
import os, glob

for env, d in [('us', US_DIR), ('sg', SG_DIR)]:
    candidates = sorted(set(
        glob.glob(f'{d}/Model/*.xml') +
        glob.glob(f'{d}/*.xml') +
        glob.glob(f'{d}/Model/**/*.xml', recursive=True)
    ))
    print(f'\n=== {env} XMLs ({len(candidates)} files) ===')
    for f in candidates:
        sz = os.path.getsize(f)
        # Model I 추정 + GDE 추정만 우선
        bn = os.path.basename(f)
        flag = '  ←' if ('I' in bn.split('.')[0].split('_')[-1] or 'model' in bn.lower()) else ''
        print(f'  {sz:>8} bytes  {f}{flag}')

In [ ]:
# Cell 8aa — EBOUNDS 비교 + prepare log 의 fermitools 호출 추적

# (1) EBOUNDS HDU bin-by-bin 우리(14) vs Sang(17): bin 0-13 영역 일치하는지
print('=== EBOUNDS bin-by-bin (우리 14-bin vs Sang 17-bin) ===\n')
print(f'{"bin":>4}  {"us E_MIN":>12} {"us E_MAX":>12}    {"sg E_MIN":>12} {"sg E_MAX":>12}    {"match?":>8}')
us_eb = fits.open(f'{US_DIR}/GC_pion_modelI_12yr_front_clean.fits')['EBOUNDS'].data
sg_eb = fits.open(f'{SG_DIR}/GC_pion_modelI_12yr_front_clean.fits')['EBOUNDS'].data
for i in range(max(len(us_eb), len(sg_eb))):
    u_lo = us_eb[i]['E_MIN'] if i < len(us_eb) else None
    u_hi = us_eb[i]['E_MAX'] if i < len(us_eb) else None
    s_lo = sg_eb[i]['E_MIN'] if i < len(sg_eb) else None
    s_hi = sg_eb[i]['E_MAX'] if i < len(sg_eb) else None
    if u_lo is not None and s_lo is not None:
        m = 'OK' if (abs(u_lo - s_lo) / s_lo < 1e-4 and abs(u_hi - s_hi) / s_hi < 1e-4) else 'DIFF'
        print(f'{i:>4}  {u_lo:>12.3f} {u_hi:>12.3f}    {s_lo:>12.3f} {s_hi:>12.3f}    {m:>8}')
    elif u_lo is not None:
        print(f'{i:>4}  {u_lo:>12.3f} {u_hi:>12.3f}    {"--":>12} {"--":>12}    us-only')
    else:
        print(f'{i:>4}  {"--":>12} {"--":>12}    {s_lo:>12.3f} {s_hi:>12.3f}    sg-only')

# (2) prepare_12yr.log 의 gtsrcmaps 호출 명령 추출
print('\n\n=== prepare_12yr.log: gtsrcmaps invocations ===\n')
import subprocess
try:
    out = subprocess.run(['grep', '-A', '3', '-B', '1', 'gtsrcmaps', 'prepare_12yr.log'],
                         capture_output=True, text=True, timeout=5)
    print(out.stdout[:3000] if out.stdout else '(no gtsrcmaps lines in prepare_12yr.log)')
except Exception as e:
    print(f'(grep failed: {e})')

# (3) gtsrcmaps.par 우리 working dir
print('\n\n=== ./gtsrcmaps.par (working dir) ===\n')
if os.path.exists('./gtsrcmaps.par'):
    with open('./gtsrcmaps.par') as f:
        print(f.read())
else:
    print('(./gtsrcmaps.par not found)')

In [ ]:
# Cell 8bb — gtsrcmaps 호출 명령 line 추출 (우리 source + Sang notebook)
import json, subprocess

print('=== 우리 run_one_model.py: gtsrcmaps invocation ===\n')
out = subprocess.run(
    ['grep', '-n', '-A', '15',
     r'\(GtApp\|gtsrcmaps\|srcMaps\|SrcMaps\)',
     'run_one_model.py'],
    capture_output=True, text=True, timeout=5)
print(out.stdout if out.stdout else '(no match in run_one_model.py)')

print('\n\n=== Sang 12yr Model I notebook: gtsrcmaps invocation cell ===\n')
sn = '/home/sanghwan/FermiLAT/Sanghwan/GC_analysis-60x60-modelI_12yr.ipynb'
with open(sn) as f:
    nb = json.load(f)
hits = 0
for i, c in enumerate(nb['cells']):
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    if any(k in src for k in ('srcMaps', 'gtsrcmaps', 'SrcMaps', 'GtApp')):
        if 'gtsrcmaps' in src.lower() or 'srcmap' in src.lower():
            hits += 1
            print(f'--- cell {i} (hit {hits}) ---')
            # 호출 관련 line 만 (전체 cell 출력 X)
            for ln in src.split('\n'):
                kw = ('srcmap', 'gtsrcmap', 'gtapp', 'expcube', 'ltcube',
                      'bexpmap', 'enumbins', 'emin', 'emax', 'edisp',
                      'evtype', '.xml', '.fits')
                if any(k in ln.lower() for k in kw):
                    print(f'  {ln.rstrip()}')
            if hits >= 3:   # 첫 3 hit 만
                break
            print()

## Plot 4 — 80-Model Envelope (improved)

Two-panel figure (Cholis+ 2022 Fig 6 / Fig 12 style):

- **Left**: all 80 models (light gray) + best-N envelope (16-84% percentile, orange band)
  + best-N median (red) + reference Model X with stat error + ±1σ_sys band
- **Right**: pairwise comparison of best-N pipeline result with Cholis Zenodo 12yr
  (matching colors per model)

The "best-N" selection uses total log-likelihood (`_likelihood_value` files); set
`N_BEST=80` to see the full distribution. Default N=5 matches Cholis Fig 12.


In [ ]:
# Cell — Plot 4: 80-model envelope (Fix 1+2: chain-median + asymmetric chain unc.)
# ----------------------------------------------------------------------------
# Fix 1: Use chain median (.npz fitted_params_median[2]) instead of MAP (.dat col 1)
#        - robust to bimodal posteriors observed at bins 0-1
# Fix 2: Show 16-84% chain credible interval per model (asymmetric)
#        - exposes intra-model fit uncertainty alongside inter-model spread
#
# Per-bin physical flux from .npz:
#   flux = c_GCE × GCE_template[b] × E[b]² / ΔE[b]
# verified self-consistent vs .dat in C2.2 diagnostic (14/14 bins, < 5%).

N_BEST   = 5
PCT_LOW  = 16
PCT_HIGH = 84

if not all_flux:
    print('[skip] no .dat files loaded')
else:
    # ============== AUGMENT: load chain percentiles from .npz ==============
    # GCE template per-bin identical across models (verified)
    ref_npz_path = GCE_NPZ_PATTERN.format(model=SELECTED_MODEL)
    have_chain = False
    if os.path.exists(ref_npz_path):
        try:
            _ref = np.load(ref_npz_path)
            GCE_pb  = _ref['GCE']
            E_npz   = _ref['E']
            dE_npz  = _ref['delta_E']
            E2dE    = E_npz**2 / dE_npz
            n_aug = 0
            for m in list(all_flux.keys()):
                p = GCE_NPZ_PATTERN.format(model=m)
                if not os.path.exists(p):
                    continue
                try:
                    d = np.load(p)
                    if 'fitted_params_median' not in d.files:
                        continue
                    c_med = d['fitted_params_median'][2]
                    c_lo  = d['fitted_params_lower'][2]
                    c_hi  = d['fitted_params_upper'][2]
                    all_flux[m]['flux_median'] = c_med * GCE_pb * E2dE
                    all_flux[m]['flux_lower']  = c_lo  * GCE_pb * E2dE
                    all_flux[m]['flux_upper']  = c_hi  * GCE_pb * E2dE
                    n_aug += 1
                except Exception:
                    continue
            have_chain = (n_aug == len(all_flux))
            print(f'  Augmented {n_aug}/{len(all_flux)} models with chain percentiles')
        except Exception as e:
            print(f'  [warn] chain augmentation failed: {e}')

    if not have_chain:
        print('  [warn] falling back to MAP-based flux for any missing model')

    # ============== Build arrays ==============
    labs     = list(all_flux.keys())
    n_mod    = len(labs)
    flux_map = np.array([all_flux[m]['flux'] for m in labs])  # MAP

    if have_chain:
        flux_med = np.array([all_flux[m]['flux_median'] for m in labs])
        flux_lo  = np.array([all_flux[m]['flux_lower']  for m in labs])
        flux_hi  = np.array([all_flux[m]['flux_upper']  for m in labs])
    else:
        # Fallback: use MAP for all
        flux_med = flux_map
        flux_lo  = np.array([all_flux[m]['lower'] for m in labs])
        flux_hi  = np.array([all_flux[m]['upper'] for m in labs])

    lh_arr  = np.array([all_loglike.get(m, np.nan) for m in labs])
    E_ref   = all_flux[labs[0]]['E']

    # ============== Best-N selection by Σ log L ==============
    if np.all(np.isnan(lh_arr)):
        best_idx = np.arange(n_mod)
        N_BEST = n_mod
        print('  [warn] no likelihood — using ALL models')
    else:
        valid = ~np.isnan(lh_arr)
        ranked = np.argsort(-lh_arr)
        best_idx = np.array([i for i in ranked if valid[i]])[:N_BEST]
        print(f'  Top {N_BEST} models by Σ log L:')
        for r, i in enumerate(best_idx, 1):
            print(f'    {r}. Model {labs[i]:<8}  Σ log L = {lh_arr[i]:.1f}  '
                  f'(c_GCE bin0 med={all_flux[labs[i]].get("flux_median", flux_map[i])[0]:.2e})')

    # ============== Envelopes ==============
    # === No stuck-model filter — paper-exact: all 80 models / 14 bins ===
    FLUX_STUCK_THRESH = -np.inf  # disabled: nothing filtered
    flux_med_filt = np.where(flux_med > FLUX_STUCK_THRESH, flux_med, np.nan)
    n_healthy = np.sum(~np.isnan(flux_med_filt), axis=0)
    print(f'\n  Per-bin healthy models (flux > {FLUX_STUCK_THRESH:.0e}):')
    for b in range(len(E_ref)):
        print(f'    bin {b:2d}  E={E_ref[b]:7.3f} GeV : {n_healthy[b]:3d}/{n_mod}')

    env_lo_all_med = np.nanpercentile(flux_med_filt, PCT_LOW,  axis=0)
    env_hi_all_med = np.nanpercentile(flux_med_filt, PCT_HIGH, axis=0)
    env_med_all    = np.nanmedian(flux_med_filt, axis=0)

    flux_med_bn_filt = flux_med_filt[best_idx]
    env_lo_bn_med  = np.nanpercentile(flux_med_bn_filt, PCT_LOW,  axis=0)
    env_hi_bn_med  = np.nanpercentile(flux_med_bn_filt, PCT_HIGH, axis=0)
    env_med_bn     = np.nanmedian(flux_med_bn_filt, axis=0)

    # NEW: combined inter-model + intra-chain uncertainty
    stuck_mask = np.isnan(flux_med_filt)
    flux_lo_f = np.where(~stuck_mask, flux_lo, np.nan)
    flux_hi_f = np.where(~stuck_mask, flux_hi, np.nan)
    env_lo_all_comb = np.nanpercentile(flux_lo_f, PCT_LOW,  axis=0)
    env_hi_all_comb = np.nanpercentile(flux_hi_f, PCT_HIGH, axis=0)

    # MAP-based all-80 (for shift diagnostic only)
    flux_map_f = np.where(~stuck_mask, flux_map, np.nan)
    env_lo_all_map = np.nanpercentile(flux_map_f, PCT_LOW,  axis=0)
    env_hi_all_map = np.nanpercentile(flux_map_f, PCT_HIGH, axis=0)

    # Cholis Zenodo refs (per best-N model)
    cholis_refs = {}
    if os.path.exists(CHOLIS_REF_DIR):
        for idx in best_idx:
            lab = labs[idx]
            p = f'{CHOLIS_REF_DIR}/GCE_Model{lab}_flux_Inner40x40_masked_disk.dat'
            if os.path.exists(p):
                r = np.loadtxt(p)
                if r.ndim == 2 and r.shape[1] >= 4:
                    cholis_refs[lab] = dict(E=r[:, 0], flux=r[:, 1],
                                            lo=r[:, 2], hi=r[:, 3])

    # ============== Plot ==============
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 6.6), sharey=True)

    # --- LEFT: All 80 (median spread) + Best-N (median spread) + Model X median±chain ---
    for i, lab in enumerate(labs):
        col   = 'lightgray' if i not in best_idx else None
        alpha = 0.20 if i not in best_idx else 0.45
        ax1.plot(E_ref, np.maximum(flux_med[i], 1e-12), '-',
                 alpha=alpha, lw=0.7, color=col, zorder=1)

    ax1.fill_between(E_ref, env_lo_all_med, env_hi_all_med, color='steelblue', alpha=0.10,
                     label=f'All {n_mod}: median spread only', zorder=2)

    ax1.fill_between(E_ref, env_lo_bn_med, env_hi_bn_med, color='orange', alpha=0.30,
                     label=f'Best {N_BEST}: median spread', zorder=4)
    ax1.plot(E_ref, env_med_bn, 'r-', lw=2.4, label='Best-N median', zorder=5)

    if SELECTED_MODEL in all_flux:
        rm = all_flux[SELECTED_MODEL]
        if have_chain and 'flux_median' in rm:
            yerr_lo = np.maximum(rm['flux_median'] - rm['flux_lower'], 0)
            yerr_hi = np.maximum(rm['flux_upper'] - rm['flux_median'], 0)
            ax1.errorbar(rm['E'], rm['flux_median'], yerr=[yerr_lo, yerr_hi],
                         fmt='o', color='black', ms=6, capsize=3, lw=1.5,
                         label=f'Model {SELECTED_MODEL} median ± chain', zorder=6)
            if sigma_sys is not None:
                ax1.fill_between(rm['E'], rm['flux_median'] - sigma_sys,
                                 rm['flux_median'] + sigma_sys,
                                 color='lightblue', alpha=0.45,
                                 label=r'$\pm 1\sigma_\mathrm{sys}$ band', zorder=3)
        else:
            ax1.errorbar(rm['E'], rm['flux'], yerr=rm['stat_err'],
                         fmt='o', color='black', ms=5, capsize=3,
                         label=f'Model {SELECTED_MODEL} (MAP, fallback)', zorder=6)

    ax1.set_xscale('log'); ax1.set_yscale('log')
    ax1.set_xlim(0.2, 60); ax1.set_ylim(1e-8, 5e-6)
    ax1.set_xlabel('E [GeV]', fontsize=12)
    ax1.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=12)
    ax1.set_title(f'GCE SED — 12yr 80 models',
                  fontsize=12)
    ax1.grid(True, which='major', alpha=0.3)
    ax1.legend(loc='lower left', fontsize=8, framealpha=0.92)

    # --- RIGHT: best-N each with its own asymmetric error bar vs Cholis ---
    palette = plt.cm.tab10.colors
    for i, idx in enumerate(best_idx):
        c = palette[i % 10]
        lab = labs[idx]
        if have_chain:
            yl = np.maximum(flux_med[idx] - flux_lo[idx], 0)
            yh = np.maximum(flux_hi[idx] - flux_med[idx], 0)
            ax2.errorbar(E_ref, flux_med[idx], yerr=[yl, yh],
                         fmt='s-', ms=4, lw=1.3, color=c, alpha=0.92, capsize=2,
                         label=f'12yr {lab} (med±chain)')
        else:
            ax2.plot(E_ref, flux_map[idx], '-', marker='s', ms=4, lw=1.2,
                     color=c, alpha=0.95, label=f'12yr Model {lab}')
        if lab in cholis_refs:
            r = cholis_refs[lab]
            ax2.plot(r['E'], r['flux'], '--', color=c, lw=1.3, alpha=0.85)
            ax2.fill_between(r['E'], r['lo'], r['hi'], alpha=0.10, color=c)
    ax2.plot([], [], '--', color='black', alpha=0.6, label='Cholis Zenodo (12yr)')
    ax2.set_xscale('log'); ax2.set_yscale('log')
    ax2.set_xlim(0.2, 60); ax2.set_ylim(1e-8, 5e-6)
    ax2.set_xlabel('E [GeV]', fontsize=12)
    ax2.set_title(f'Best-{N_BEST}: median ± chain vs Cholis 12yr',
                  fontsize=12)
    ax2.grid(True, which='major', alpha=0.3)
    ax2.legend(loc='lower left', fontsize=7, ncol=2, framealpha=0.92)

    plt.tight_layout()
    out = f'{PLOTS_DIR}/04_envelope_best{N_BEST}_FIX1_FIX2.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()

    # ============== Diagnostic: MAP→median shift table ==============
    if have_chain:
        print(f'\n  MAP-based vs Median-based 16-84% envelope shift (All {n_mod}):')
        print(f'    bin |  E[GeV] | MAP-lo    | med-lo    | shift  | MAP-hi    | med-hi    | shift')
        print(f'    ----+---------+-----------+-----------+--------+-----------+-----------+------')
        for b in range(min(8, len(E_ref))):
            ml_map, mh_map = env_lo_all_map[b], env_hi_all_map[b]
            ml_med, mh_med = env_lo_all_med[b], env_hi_all_med[b]
            sh_lo = (ml_med / max(ml_map, 1e-15) - 1) * 100
            sh_hi = (mh_med / max(mh_map, 1e-15) - 1) * 100
            print(f'    {b:3d} | {E_ref[b]:7.3f} | '
                  f'{ml_map:.3e} | {ml_med:.3e} | {sh_lo:+5.0f}% | '
                  f'{mh_map:.3e} | {mh_med:.3e} | {sh_hi:+5.0f}%')

    # Per-bin combined-envelope width
    if have_chain:
        print(f'\n  Combined envelope width (low E):')
        print(f'    bin |  E[GeV] | width (dex) | half-width / median')
        for b in range(min(8, len(E_ref))):
            comb_w = np.log10(env_hi_all_comb[b] / max(env_lo_all_comb[b], 1e-15))
            half_med = 0.5 * (env_hi_all_comb[b] - env_lo_all_comb[b]) / max(env_med_all[b], 1e-15)
            print(f'    {b:3d} | {E_ref[b]:7.3f} | {comb_w:.2f}        | {half_med:.2f}')

In [ ]:
# Cell 9b — Ranking scatter: 12yr log-L vs Cholis 12yr log-L (γ=1.2)
import os
import numpy as np
import matplotlib.pyplot as plt

CHOLIS_LOGL_FILE = f'{CHOLIS_REF_DIR.replace("/Figures_12_and_14_GCE_Spectra","")}/GCE_Models_LogLikelihoods_2021_DMprofiles_October_GCE_vs_Background.dat'
CHOLIS_BEST5  = {'X', 'XV', 'XLVIII', 'XLIX', 'LIII'}
CHOLIS_WORST5 = {'II', 'LXIV', 'LXIX', 'LXX', 'LXXI'}

# Load Cholis 12yr log-L (γ=1.2 col = index 1)
cholis_logl = {}
if os.path.exists(CHOLIS_LOGL_FILE):
    with open(CHOLIS_LOGL_FILE) as f:
        for ln in f:
            if ln.startswith('#') or not ln.strip(): continue
            tok = ln.split()
            try: cholis_logl[tok[0]] = float(tok[1])
            except: pass
    print(f'Loaded Cholis 12yr log-L: {len(cholis_logl)} models')
else:
    print(f'[warn] Cholis log-L file missing: {CHOLIS_LOGL_FILE}')

# Load 12yr per-model sum log-L (assumes Cell 5 has loaded all .dat / lh files)
# OR direct from disk if Cell 5 hasn't been run:
def color_for(M):
    if M in CHOLIS_BEST5:  return 'gold'
    if M in CHOLIS_WORST5: return 'crimson'
    return 'gray'

logl_12yr = {}
for M in ALL_MODELS:
    lh = GCE_LH_PATTERN.format(model=M)
    if os.path.exists(lh):
        logl_12yr[M] = float(np.loadtxt(lh).sum())

print(f'12yr log-L loaded: {len(logl_12yr)} / 80')
common = sorted(set(logl_12yr) & set(cholis_logl))

if len(common) >= 5:
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    for M in common:
        x = logl_12yr[M]; y = cholis_logl[M]
        c = color_for(M)
        ax.scatter(x, y, c=c, s=220, edgecolor='black', zorder=10, alpha=0.85)
        # annotate only best/worst + outlier
        if M in CHOLIS_BEST5 or M in CHOLIS_WORST5:
            ax.annotate(M, (x, y), fontsize=10, fontweight='bold',
                        xytext=(7, 7), textcoords='offset points')
    # Ideal monotone line (단, 데이터셋 다름으로 scale 다름)
    ax.set_xlabel('12yr sum log-L (this work)', fontsize=12)
    ax.set_ylabel('Cholis 12yr log-L (γ=1.2, published)', fontsize=12)
    # Custom legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gold',
               markersize=12, markeredgecolor='black', label='Cholis best 5 (★)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='crimson',
               markersize=12, markeredgecolor='black', label='Cholis worst 5 (✗)'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markersize=12, markeredgecolor='black', label='other 70 models'),
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower right')
    ax.set_title(f'Ranking 12yr vs Cholis 12yr  ({len(common)} models)\n'
                 '(monotone diagonal-like spread → ranking preserved; scatter → (β))')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/04b_ranking_12yr_vs_cholis.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient models for ranking scatter (need ≥5).')
# ============================================================


In [ ]:
# Cell 9c — Ranking table: 12yr log-L vs Cholis 12yr log-L (γ=1.2)
import pandas as pd

# 12yr 내림차순(=좋은 fit 순) rank, Cholis도 동일 common 집합 기준 rank
order_12  = sorted(common, key=lambda M: logl_12yr[M], reverse=True)
order_cho = sorted(common, key=lambda M: cholis_logl[M], reverse=True)
rank_12   = {M: i + 1 for i, M in enumerate(order_12)}
rank_cho  = {M: i + 1 for i, M in enumerate(order_cho)}

def _group(M):
    if M in CHOLIS_BEST5:  return '★ best5'
    if M in CHOLIS_WORST5: return '✗ worst5'
    return ''

rows = []
for M in order_12:
    rows.append({
        'rank_12yr':   rank_12[M],
        'Model':       M,
        'logL_12yr':   logl_12yr[M],
        'rank_Cholis': rank_cho[M],
        'logL_Cholis': cholis_logl[M],
        'Δrank':       rank_cho[M] - rank_12[M],   # +면 12yr에서 상승, −면 하락
        'group':       _group(M),
    })

rank_df = pd.DataFrame(rows)

# 콘솔/노트북 출력 (전체 행, 정수/실수 포맷)
with pd.option_context('display.max_rows', None,
                       'display.float_format', lambda v: f'{v:,.2f}'):
    print(rank_df.to_string(index=False))

# CSV 저장 (기존 플롯 저장 위치와 동일)
csv_out = f'{PLOTS_DIR}/04b_ranking_12yr_vs_cholis.csv'
rank_df.to_csv(csv_out, index=False)
print(f'\n[saved] {csv_out}')

# Δrank 절댓값이 큰 순으로도 한 번 — ranking inversion 후보 확인용
print('\n--- |Δrank| 상위 (12yr↔Cholis 순위 변동 큰 모델) ---')
print(rank_df.reindex(rank_df['Δrank'].abs().sort_values(ascending=False).index)
            .head(10).to_string(index=False))

In [ ]:
# Cell 9e — 3-way 전체 순위 + sum_logL 값 (읽기용 전체 표)
#
# 각 설정:
#   P   = our flat-ellipse bubble + Cholis prior (P+χ²)   [production, 무접미사]
#   a   = his bubble + his prior (Poisson only)            [_hisbub_noConstr]
#   b   = his bubble + Cholis prior (P+χ²)                 [_hisbub_constr]
#   Cho = Cholis 발표 γ=1.2 diffuse-fit log-L
# rank = 설정 내 순위(1=best, sum_logL 내림차순). common 위에서 매김.
# 주의 1) a는 χ² penalty가 없어 sum_logL이 체계적으로 높음(less negative).
#        설정 간 sum_logL '절대값' 직접비교는 무의미 → 순위로 비교.
#      2) d_best = (그 설정 best의 sum_logL) − (그 모델 sum_logL), ≥0. 0=best, 클수록 열위.
import os
import numpy as np
import pandas as pd

LH_PATTERNS = {
    'P': f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis_likelihood_value',
    'a': f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis_hisbub_noConstr_likelihood_value',
    'b': f'{RESULTS_DIR}/GCE_model_{{model}}_front_12yr_cholis_hisbub_constr_likelihood_value',
}
CHOLIS_LOGL_FILE = (f'{CHOLIS_REF_DIR.replace("/Figures_12_and_14_GCE_Spectra","")}'
                    '/GCE_Models_LogLikelihoods_2021_DMprofiles_October_GCE_vs_Background.dat')
CHOLIS_BEST5  = {'X', 'XV', 'XLVIII', 'XLIX', 'LIII'}
CHOLIS_WORST5 = {'II', 'LXIV', 'LXIX', 'LXX', 'LXXI'}
SANGHWAN_BEST = 'X'

# --- 우리 3개 설정 sum_logL ---
logl = {k: {} for k in LH_PATTERNS}
for k, pat in LH_PATTERNS.items():
    for M in ALL_MODELS:
        p = pat.format(model=M)
        if os.path.exists(p):
            logl[k][M] = float(np.loadtxt(p).sum())

# --- Cholis γ=1.2 log-L (col0=Model, col1=γ=1.2) ---
cholis = {}
if os.path.exists(CHOLIS_LOGL_FILE):
    for ln in open(CHOLIS_LOGL_FILE):
        if ln.startswith('#') or not ln.strip():
            continue
        t = ln.split()
        try:
            cholis[t[0]] = float(t[1])
        except Exception:
            pass

common = [M for M in ALL_MODELS if M in cholis and all(M in logl[k] for k in logl)]

def rank_and_dbest(score):
    order = sorted(common, key=lambda M: score[M], reverse=True)   # best=rank1
    rk = {M: i + 1 for i, M in enumerate(order)}
    best_val = score[order[0]]
    db = {M: best_val - score[M] for M in common}                  # ≥0
    return rk, db

R, DB = {}, {}
for k in logl:
    R[k], DB[k] = rank_and_dbest(logl[k])
R['Cho'], DB['Cho'] = rank_and_dbest(cholis)

def flag(M):
    return (('★' if M in CHOLIS_BEST5 else ' ')
            + ('✗' if M in CHOLIS_WORST5 else ' ')
            + ('S' if M == SANGHWAN_BEST else ' '))

rows = []
for M in sorted(common, key=lambda M: R['Cho'][M]):     # Cholis 순위순
    rows.append({
        'Model':          M,
        'flag':           flag(M),
        'P_rank':         R['P'][M],
        'P_sumLogL':      logl['P'][M],
        'a_rank':         R['a'][M],
        'a_sumLogL':      logl['a'][M],
        'b_rank':         R['b'][M],
        'b_sumLogL':      logl['b'][M],
        'Cholis_rank':    R['Cho'][M],
        'Cholis_logL_g1.2': cholis[M],
    })
full_df = pd.DataFrame(rows)

# 범례
print('P   = our flat-ellipse bubble + Cholis prior (P+χ²)   [production]')
print('a   = his bubble + his prior (Poisson only)')
print('b   = his bubble + Cholis prior (P+χ²)')
print('Cho = Cholis 발표 γ=1.2 diffuse-fit log-L')
print('rank: 설정 내 순위(1=best, sum_logL 내림차순) | flag: ★Cholis best5  ✗worst5  S=Sanghwan best(X)')
print('주의: a는 χ² penalty 없어 sum_logL 체계적으로 높음 → 설정 간 절대값 비교 무의미, 순위로 비교')
print(f'common models = {len(common)}  (정렬: Cholis 순위순)\n')

with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 240,
                       'display.float_format', lambda v: f'{v:,.1f}'):
    print(full_df.to_string(index=False))

# CSV (d_best 포함; 재정렬·정밀비교용)
csv_df = full_df.copy()
for k, col in [('P', 'P'), ('a', 'a'), ('b', 'b'), ('Cho', 'Cholis')]:
    csv_df[f'{col}_dBest'] = [DB[k][M] for M in csv_df['Model']]
csv_out = f'{PLOTS_DIR}/04c_ranking_3way_full_logL.csv'
csv_df.to_csv(csv_out, index=False)
print(f'\n[saved] {csv_out}  (sum_logL + rank + d_best 전체)')

In [ ]:
# Cell 9f — 5-파이프라인 순위 + 양봉(bimodal) 탐지
#   12P  = 12yr our-bubble + Cholis prior (production)      [results_12yr, 무접미사]
#   12a  = 12yr his-bubble + Poisson-only (Sanghwan 방법 아날로그) [_hisbub_noConstr]
#   17   = 17yr thesis (our-bubble + Cholis prior)          [../GCE_17yr_reproduce/results_17yr]
#   SW16 = Sanghwan 16yr FRONT (his-bubble + Poisson-only)  [../GCE_16yr_data/Sanghwan_result]
#   Cho  = Cholis 발표 γ=1.2 diffuse-fit log-L
# 규약: sum_logL 내림차순 = best(rank1). 연도별 데이터가 달라 sum_logL '절대값' 교차비교 불가
#       -> 순위(Spearman)와 파이프라인 내부 양봉 구조만 해석.
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE = '/home/haebarg/GCE-Chi-square-fitting'
LH = {
    '12P':  BASE + '/GCE_12yr_reproduce/results_12yr/GCE_model_{M}_front_12yr_cholis_likelihood_value',
    '12a':  BASE + '/GCE_12yr_reproduce/results_12yr/GCE_model_{M}_front_12yr_cholis_hisbub_noConstr_likelihood_value',
    '17':   BASE + '/GCE_17yr_reproduce/results_17yr/GCE_model_{M}_front_17yr_cholis_likelihood_value',
    'SW16': BASE + '/GCE_16yr_data/Sanghwan_result/GCE_model_{M}_front_16yr_cholis_likelihood_value',
}
LABELS = {
    '12P':  '12yr our-bub + Cholis (production)',
    '12a':  '12yr his-bub + Poisson (Sanghwan-method analog)',
    '17':   '17yr thesis (our-bub + Cholis)',
    'SW16': 'Sanghwan 16yr FRONT (his-bub + Poisson)',
    'Cho':  'Cholis published gamma=1.2',
}
CHOLIS_BEST5  = {'X', 'XV', 'XLVIII', 'XLIX', 'LIII'}
CHOLIS_WORST5 = {'II', 'LXIV', 'LXIX', 'LXX', 'LXXI'}
SANGHWAN_BEST = 'X'
PIPES = ['12P', '12a', '17', 'SW16', 'Cho']

# --- our-format 파이프라인 로드 (14 per-bin log-L 합) ---
logl = {}
for k, pat in LH.items():
    d = {}
    for M in ALL_MODELS:
        p = pat.format(M=M)
        if os.path.exists(p):
            d[M] = float(np.loadtxt(p).sum())
    logl[k] = d
    miss = [M for M in ALL_MODELS if M not in d]
    print(f'[{k:5s}] {LABELS[k]:44s} {len(d):2d}/80' + (f'  누락:{miss}' if miss else ''))

# --- Cholis gamma=1.2 (col0=Model, col1=g1.2) ---
CHOLIS_LOGL_FILE = (f'{CHOLIS_REF_DIR.replace("/Figures_12_and_14_GCE_Spectra","")}'
                    '/GCE_Models_LogLikelihoods_2021_DMprofiles_October_GCE_vs_Background.dat')
cho = {}
if os.path.exists(CHOLIS_LOGL_FILE):
    for ln in open(CHOLIS_LOGL_FILE):
        if ln.startswith('#') or not ln.strip():
            continue
        t = ln.split()
        try:
            cho[t[0]] = float(t[1])
        except Exception:
            pass
logl['Cho'] = cho
print(f'[Cho  ] {LABELS["Cho"]:44s} {len(cho):2d}/80')

common = [M for M in ALL_MODELS if all(M in logl[k] for k in PIPES)]
print(f'\ncommon (전 파이프라인): {len(common)}')
if len(common) < 5:
    print('[stop] common < 5 — 경로/완료 확인 필요.')
else:
    # ===== (1) 양봉 탐지: 정렬 sum_logL의 최대 gap =====
    print('\n=== (1) 양봉 탐지: 내림차순 sum_logL의 최대 gap ===')
    print(f'{"pipe":5s} {"n":>3s} {"max_gap":>11s} {"gap/median":>11s} {"above|below":>13s}  gap')
    bad_sets = {}
    for k in PIPES:
        s = sorted((logl[k][M] for M in common), reverse=True)
        gaps = [s[i] - s[i + 1] for i in range(len(s) - 1)]
        gmax = max(gaps); gi = gaps.index(gmax); med = float(np.median(gaps))
        ratio = gmax / med if med > 0 else float('inf')
        n_above, n_below = gi + 1, len(s) - (gi + 1)
        desc = 'gap 뚜렷' if (ratio > 8 and n_below >= 3) else '완만'
        print(f'{k:5s} {len(s):3d} {gmax:11.0f} {ratio:11.1f} {n_above:5d}|{n_below:<7d}  {desc}')
        thr = s[gi + 1]
        bad_sets[k] = set(M for M in common if logl[k][M] <= thr)

    # bad cluster(최대 gap 아래) 멤버십 교차 — 12yr가 Sanghwan을 물려받았나
    print('\n=== bad cluster 멤버십 교차 (최대 gap 아래 모델 집합) ===')
    for k in PIPES:
        print(f'  {k:5s}: {len(bad_sets[k]):2d}개')
    def jac(a, b):
        u = len(a | b); return len(a & b) / u if u else 1.0
    for a_, b_ in [('12a', 'SW16'), ('12P', 'SW16'), ('12a', '17'), ('SW16', '17'), ('12a', '12P')]:
        inter = len(bad_sets[a_] & bad_sets[b_])
        print(f'  {a_:4s} ∩ {b_:4s} = {inter:2d}개  (Jaccard {jac(bad_sets[a_], bad_sets[b_]):.2f})')

    # ===== (2) Spearman ρ vs Cholis =====
    def rk(k):
        order = sorted(common, key=lambda M: logl[k][M], reverse=True)
        return {M: i + 1 for i, M in enumerate(order)}
    RANK = {k: rk(k) for k in PIPES}
    print('\n=== (2) Spearman ρ vs Cholis (순위 상관, common) ===')
    cho_vec = [RANK['Cho'][M] for M in common]
    for k in ['12P', '12a', '17', 'SW16']:
        rho, _ = spearmanr([RANK[k][M] for M in common], cho_vec)
        print(f'  {k:5s} vs Cholis: ρ = {rho:+.3f}   ({LABELS[k]})')
    # 파이프라인 상호 ρ (Sanghwan 방법 계열 vs 우리)
    for a_, b_ in [('12a', 'SW16'), ('12P', '17'), ('12a', '17')]:
        rho, _ = spearmanr([RANK[a_][M] for M in common], [RANK[b_][M] for M in common])
        print(f'  {a_:5s} ↔ {b_:5s}: ρ = {rho:+.3f}')

    # ===== (3) 순위 + sum_logL 표 (Cholis 순) =====
    def flag(M):
        return (('★' if M in CHOLIS_BEST5 else '') + ('✗' if M in CHOLIS_WORST5 else '')
                + ('S' if M == SANGHWAN_BEST else ''))
    rows = []
    for M in sorted(common, key=lambda M: RANK['Cho'][M]):
        r = {'Model': M, 'flag': flag(M)}
        for k in PIPES:
            r[f'r_{k}'] = RANK[k][M]
        for k in PIPES:
            r[f'L/1e3_{k}'] = logl[k][M] / 1000.0
        rows.append(r)
    tbl = pd.DataFrame(rows)
    print('\n=== (3) 순위 r_ + sum_logL/1000 (L/1e3_), Cholis 순 정렬 ===')
    print('    flag: ★Cholis best5  ✗worst5  S=Sanghwan best(X)')
    print('    주의: L/1e3 절대값은 연도별 데이터 달라 교차비교 무의미 (같은 pipe 내 상대차/순위만)')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.width', 320, 'display.float_format', lambda v: f'{v:,.1f}'):
        print(tbl.to_string(index=False))
    csv = f'{PLOTS_DIR}/04d_ranking_5pipe_bimodal.csv'
    tbl.to_csv(csv, index=False)
    print(f'\n[saved] {csv}')

In [ ]:
# Cell 9g — 4-파이프라인 비교: 12yr(채택) / 17yr / SW16 / Cholis  [04e]
#   12   = 12yr his-bubble + Poisson-only [_hisbub_noConstr] ← 채택된 12yr 분석 방법
#          (mirror-fix 재fit 완료본; production 무접미사 12P는 폐기)
#   17   = 17yr thesis (our-bubble + Cholis prior)
#   SW16 = Sanghwan 16yr FRONT (his-bubble + Poisson-only)
#   Cho  = Cholis 발표 γ=1.2 diffuse-fit log-L
# 규약: sum_logL 내림차순 = best(rank1). 연도별 데이터가 달라 절대값 교차비교 무의미
#       -> 순위(Spearman)와 같은 pipe 내 Δbest만 해석. (4)의 전체값 나열은 기록/참조용.
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE = '/home/haebarg/GCE-Chi-square-fitting'
LH = {
    '12':   BASE + '/GCE_12yr_reproduce/results_12yr/GCE_model_{M}_front_12yr_cholis_hisbub_noConstr_likelihood_value',
    '17':   BASE + '/GCE_17yr_reproduce/results_17yr/GCE_model_{M}_front_17yr_cholis_likelihood_value',
    'SW16': BASE + '/GCE_16yr_data/Sanghwan_result/GCE_model_{M}_front_16yr_cholis_likelihood_value',
}
LABELS = {
    '12':   '12yr his-bub + Poisson (채택 12yr method, mirror-fix 후)',
    '17':   '17yr thesis (our-bub + Cholis)',
    'SW16': 'Sanghwan 16yr FRONT (his-bub + Poisson)',
    'Cho':  'Cholis published gamma=1.2',
}
CHOLIS_BEST5  = {'X', 'XV', 'XLVIII', 'XLIX', 'LIII'}
CHOLIS_WORST5 = {'II', 'LXIV', 'LXIX', 'LXX', 'LXXI'}
SANGHWAN_BEST = 'X'
PIPES = ['12', '17', 'SW16', 'Cho']

# --- our-format 파이프라인 로드 (14 per-bin log-L 합) ---
logl = {}
for k, pat in LH.items():
    d = {}
    for M in ALL_MODELS:
        p = pat.format(M=M)
        if os.path.exists(p):
            d[M] = float(np.loadtxt(p).sum())
    logl[k] = d
    miss = [M for M in ALL_MODELS if M not in d]
    print(f'[{k:5s}] {LABELS[k]:52s} {len(d):2d}/80' + (f'  누락:{miss}' if miss else ''))

# --- Cholis gamma=1.2 (col0=Model, col1=g1.2 <- 우리 NFW² 대응 열) ---
CHOLIS_LOGL_FILE = (f'{CHOLIS_REF_DIR.replace("/Figures_12_and_14_GCE_Spectra","")}'
                    '/GCE_Models_LogLikelihoods_2021_DMprofiles_October_GCE_vs_Background.dat')
cho = {}
if os.path.exists(CHOLIS_LOGL_FILE):
    for ln in open(CHOLIS_LOGL_FILE):
        if ln.startswith('#') or not ln.strip():
            continue
        t = ln.split()
        try:
            cho[t[0]] = float(t[1])
        except Exception:
            pass
logl['Cho'] = cho
print(f'[Cho  ] {LABELS["Cho"]:52s} {len(cho):2d}/80')

common = [M for M in ALL_MODELS if all(M in logl[k] for k in PIPES)]
print(f'\ncommon (전 파이프라인): {len(common)}')
if len(common) < 5:
    print('[stop] common < 5 — 경로/완료 확인 필요.')
else:
    # ===== (1) worst-cluster 구조 sanity: 내림차순 sum_logL 최대 gap =====
    print('\n=== (1) 내림차순 sum_logL 최대 gap (worst-cluster 구조 sanity) ===')
    print(f'{"pipe":5s} {"n":>3s} {"max_gap":>11s} {"gap/median":>11s} {"above|below":>13s}  gap')
    for k in PIPES:
        s = sorted((logl[k][M] for M in common), reverse=True)
        gaps = [s[i] - s[i + 1] for i in range(len(s) - 1)]
        gmax = max(gaps); gi = gaps.index(gmax); med = float(np.median(gaps))
        ratio = gmax / med if med > 0 else float('inf')
        desc = 'gap 뚜렷' if (ratio > 8 and len(s) - gi - 1 >= 3) else '완만'
        print(f'{k:5s} {len(s):3d} {gmax:11.0f} {ratio:11.1f} {gi + 1:5d}|{len(s) - gi - 1:<7d}  {desc}')

    # ===== (2) Spearman ρ — 4×4 pairwise 행렬 =====
    def rk(k):
        order = sorted(common, key=lambda M: logl[k][M], reverse=True)
        return {M: i + 1 for i, M in enumerate(order)}
    RANK = {k: rk(k) for k in PIPES}
    print('\n=== (2) Spearman ρ (pairwise, common) ===')
    mat = pd.DataFrame(index=PIPES, columns=PIPES, dtype=float)
    for a_ in PIPES:
        va = [RANK[a_][M] for M in common]
        for b_ in PIPES:
            mat.loc[a_, b_] = spearmanr(va, [RANK[b_][M] for M in common])[0]
    with pd.option_context('display.float_format', lambda v: f'{v:+.3f}'):
        print(mat.to_string())

    # ===== (3) 순위표 (Cholis 순 정렬) =====
    def flag(M):
        return (('★' if M in CHOLIS_BEST5 else '') + ('✗' if M in CHOLIS_WORST5 else '')
                + ('S' if M == SANGHWAN_BEST else ''))
    rows = []
    for M in sorted(common, key=lambda M: RANK['Cho'][M]):
        r = {'Model': M, 'flag': flag(M)}
        for k in PIPES:
            r[f'r_{k}'] = RANK[k][M]
        rows.append(r)
    tbl_r = pd.DataFrame(rows)
    print('\n=== (3) 순위 r_ (Cholis 순 정렬) ===')
    print('    flag: ★Cholis best5  ✗worst5  S=Sanghwan best(X)')
    with pd.option_context('display.max_rows', None, 'display.width', 200):
        print(tbl_r.to_string(index=False))

    # ===== (4) likelihood 전체 값 나열 (full precision) =====
    # logL_k  = 우리 3개 pipe: 14-bin per-bin log-L 합 / Cho: 발표 diffuse-fit log-L
    # dbest_k = logL - max(logL)  (같은 pipe 내에서만 의미; -2ΔlnL = -2*dbest)
    best = {k: max(logl[k][m] for m in common) for k in PIPES}
    rows = []
    for M in sorted(common, key=lambda M: RANK['Cho'][M]):
        r = {'Model': M, 'flag': flag(M)}
        for k in PIPES:
            r[f'logL_{k}'] = logl[k][M]
        for k in PIPES:
            r[f'dbest_{k}'] = logl[k][M] - best[k]
        rows.append(r)
    tbl_L = pd.DataFrame(rows)
    print('\n=== (4) likelihood 전체 값 (Cholis 순 정렬) ===')
    print('    주의: 절대값 교차비교 무의미(연도별 데이터 상이) — 같은 pipe 내 dbest만 해석')
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.width', 360,
                           'display.float_format', lambda v: f'{v:,.1f}'):
        print(tbl_L.to_string(index=False))

    # ===== 저장: 순위 + 전체 logL 통합 CSV =====
    tbl = tbl_r.merge(tbl_L.drop(columns=['flag']), on='Model')
    csv = f'{PLOTS_DIR}/04e_ranking_4pipe.csv'
    tbl.to_csv(csv, index=False)
    print(f'\n[saved] {csv}')

## Plot 5 — Covariance Matrix Visualization (NEW)

Two-panel figure showing the structure of the systematic covariance matrix:

- **Left**: `cov_sys` heatmap — full 14×14 matrix, signed log scale
- **Right**: σ_sys = sqrt(diag(cov_sys)) vs E, compared to per-bin stat error

Useful sanity check that the cov matrix shape reproduces the expected pattern
(peak around ~0.5 GeV per Calore+ 1409.0042).


In [ ]:
# Cell 10 — Plot 5: Covariance matrix heatmap + σ_sys profile
if cov_sys is None or E is None:
    print('[skip] cov matrix or E not available')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --- Left: cov heatmap (signed log) ---
    ax = axes[0]
    vmax = np.max(np.abs(cov_sys))
    norm = SymLogNorm(linthresh=vmax * 1e-3, vmin=-vmax, vmax=vmax, base=10)
    im = ax.imshow(cov_sys, origin='lower', cmap='RdBu_r', norm=norm,
                   extent=[0.5, n_bins + 0.5, 0.5, n_bins + 0.5])
    ax.set_xlabel('bin index')
    ax.set_ylabel('bin index')
    ax.set_title(r'$\Sigma_\mathrm{sys}$  (14×14, '
                 f'cond = {np.linalg.cond(cov_sys):.1e})', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label=r'$\Sigma_{ij}$  [(GeV/cm²/s/sr)²]')

    # --- Right: σ_sys vs σ_stat ---
    ax = axes[1]
    ax.loglog(E, sigma_sys, '-o', color='C0', lw=2, ms=5,
              label=r'$\sigma_\mathrm{sys}$ (cov diag)')
    if SELECTED_MODEL in all_flux:
        ax.loglog(E, all_flux[SELECTED_MODEL]['stat_err'], '-s',
                  color='C1', lw=2, ms=5,
                  label=rf'$\sigma_\mathrm{{stat}}$ (Model {SELECTED_MODEL})')
        sig_total = np.sqrt(sigma_sys**2 + all_flux[SELECTED_MODEL]['stat_err']**2)
        ax.loglog(E, sig_total, '--', color='black', lw=1.4,
                  label=r'$\sigma_\mathrm{tot} = \sqrt{\sigma_\mathrm{stat}^2+\sigma_\mathrm{sys}^2}$')
    ax.set_xlabel('E [GeV]')
    ax.set_ylabel(r'$\sigma$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
    ax.set_title('Systematic vs statistical uncertainty', fontsize=11)
    ax.grid(True, which='major', alpha=0.3)
    ax.legend(loc='best', fontsize=10)

    plt.tight_layout()
    out = f'{PLOTS_DIR}/05_covariance_matrix.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()


## Plot 6 — Multi-Model Coefficient Comparison (NEW)

Per-bin fit coefficients (c_π+br, c_ICS, c_GCE, c_bub, c_iso) compared across the
models in `MODELS_TO_COMPARE`. Useful for checking GDE-GCE degeneracy and per-model
heterogeneity (12yr V10 patterns).


In [ ]:
# Cell 11 — Plot 6: Multi-model fit-coefficient comparison
if not coef_dict or E is None:
    print('[skip] no .npz coefficients available')
else:
    avail = [m for m in MODELS_TO_COMPARE if m in coef_dict]
    if SELECTED_MODEL in coef_dict and SELECTED_MODEL not in avail:
        avail = [SELECTED_MODEL] + avail
    if not avail:
        print('[skip] none of MODELS_TO_COMPARE has a .npz')
    else:
        print(f'Comparing models: {avail}')

        fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.4), sharex=True)
        coef_specs = [
            (r'$c_{\pi^0+\mathrm{br}}$', 'c_pion', axes[0, 0]),
            (r'$c_\mathrm{ICS}$',          'c_ics',  axes[0, 1]),
            (r'$c_\mathrm{GCE}$',          'c_gce',  axes[0, 2]),
            (r'$c_\mathrm{bubble}$',       'c_bub',  axes[1, 0]),
            (r'$c_\mathrm{iso}$',          'c_iso',  axes[1, 1]),
        ]
        palette = plt.cm.tab10.colors
        for label, key, ax in coef_specs:
            for j, m in enumerate(avail):
                ax.plot(E, coef_dict[m][key], '-o', lw=1.5, ms=4,
                        color=palette[j % 10], label=f'Model {m}')
            ax.axhline(1.0, color='gray', ls='--', lw=0.8)
            ax.set_xscale('log')
            ax.set_xlabel('E [GeV]')
            ax.set_ylabel(label)
            ax.set_title(label)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8, loc='best')

        # Use last panel for summary
        ax_sum = axes[1, 2]; ax_sum.axis('off')
        # Compute per-model means
        sumtxt = f'Mean coefficient (14-bin avg):\n\n'
        sumtxt += f"{'Model':<8} {'π+br':>6} {'ICS':>6} {'GCE':>6} {'bub':>6} {'iso':>6}\n"
        sumtxt += '-' * 42 + '\n'
        for m in avail:
            cd = coef_dict[m]
            sumtxt += (f"{m:<8} {cd['c_pion'].mean():>6.2f} {cd['c_ics'].mean():>6.2f} "
                       f"{cd['c_gce'].mean():>6.2f} {cd['c_bub'].mean():>6.2f} "
                       f"{cd['c_iso'].mean():>6.2f}\n")

        # Diagnostic interpretation
        c_gas_means = [(coef_dict[m]['c_pion'].mean() + coef_dict[m]['c_ics'].mean()) / 2
                       for m in avail]
        c_gce_means = [coef_dict[m]['c_gce'].mean() for m in avail]
        if all(0.85 < g < 1.15 for g in c_gas_means) and all(0.85 < g < 1.15 for g in c_gce_means):
            interp_str = '\n→ all c ≈ 1: fit looks healthy across models'
        elif all(g > 1.15 for g in c_gas_means) and all(g < 0.85 for g in c_gce_means):
            interp_str = '\n→ GDE-GCE degeneracy (gas > 1, GCE < 1) consistent'
        elif (max(c_gce_means) - min(c_gce_means)) > 0.3:
            interp_str = '\n→ model-specific c_GCE variation > 0.3'
        else:
            interp_str = '\n→ mixed signature; inspect individual coefficients'
        ax_sum.text(0.02, 0.98, sumtxt + interp_str,
                    transform=ax_sum.transAxes,
                    fontsize=9, family='monospace', va='top')

        plt.tight_layout()
        out = f'{PLOTS_DIR}/06_coefficient_comparison.png'
        plt.savefig(out, dpi=130, bbox_inches='tight')
        print(f'[saved] {out}')
        plt.show()


In [ ]:
# Cell — SWAPBUB overlay: XLIX bubble-morphology swap vs production vs Cholis XLIX band
# 판정: (1) 저-E GCE flux 가 Cholis Fig19 stat band 안으로 들어오나  (2) c_Bub 가 0 붕괴를 멈췄나
import numpy as np, os
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

_ref   = globals().get('CHOLIS_REF_DIR', '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra')
band_f = f'{_ref}/GCE_ModelXLIX_flux_Inner40x40_masked_disk.dat'        # 모델명 파일(랭크명 GCE_BestFitModel_ 쓰지 말 것)
prod_f = globals().get('GCE_DAT_PATTERN', './results_12yr/GCE_model_{model}_front_12yr_cholis.dat').format(model='XLIX')
swap_f = './GCE_model_XLIX_SWAPBUB_front_12yr_cholis.dat'
prod_npz_f = globals().get('GCE_NPZ_PATTERN', './results_12yr/GCE_model_{model}_front_12yr_cholis_fit.npz').format(model='XLIX')
swap_npz_f = './GCE_model_XLIX_SWAPBUB_front_12yr_cholis_fit.npz'
_plots = globals().get('PLOTS_DIR', './GCE_12yr_results_plots'); os.makedirs(_plots, exist_ok=True)

for f in [band_f, prod_f, swap_f, prod_npz_f, swap_npz_f]:
    print(f'[{"OK" if os.path.exists(f) else "MISS"}] {f}')

band = np.loadtxt(band_f)     # E, best, lo, hi  (Cholis XLIX, 40x40 inner, masked_disk)
prod = np.loadtxt(prod_f)     # E, flux(MAP), std, lo(16%), hi(84%)
swap = np.loadtxt(swap_f)
Es    = swap[:, 0]
valid = swap[:, 1] > 0        # 미실행 bin 방어

cb = interp1d(band[:,0], band[:,1], fill_value='extrapolate')(Es)   # Cholis best  (우리 E격자)
cl = interp1d(band[:,0], band[:,2], fill_value='extrapolate')(Es)   # lo
ch = interp1d(band[:,0], band[:,3], fill_value='extrapolate')(Es)   # hi

def _yerr(d):  # MAP±[16%,84%], 음수 방어
    return np.clip(np.array([d[:,1]-d[:,3], d[:,4]-d[:,1]]), 0, None)

fig, ax = plt.subplots(1, 3, figsize=(21, 6))

a = ax[0]   # (1) SED overlay
a.fill_between(band[:,0], band[:,2], band[:,3], alpha=0.30, color='gray', label='Cholis XLIX (1σ stat)')
a.plot(band[:,0], band[:,1], 'D-', color='k', ms=5, lw=1.3, label='Cholis XLIX best')
a.errorbar(prod[:,0], prod[:,1], yerr=_yerr(prod), fmt='o-', color='tab:blue', ms=6, capsize=3,
           label='ours XLIX production')
a.errorbar(swap[valid,0], swap[valid,1], yerr=_yerr(swap)[:, valid], fmt='s-', color='tab:red', ms=6, capsize=3,
           label='ours XLIX SWAPBUB (gcepy bubble shape)')
a.set_xscale('log'); a.set_yscale('log'); a.set_xlim(0.25, 60); a.set_ylim(1e-8, 3e-6)
a.set_xlabel('E [GeV]'); a.set_ylabel(r'$E^2 d\Phi/dE$ [GeV/cm²/s/sr]')
a.set_title('XLIX GCE SED — production vs bubble-morphology swap'); a.legend(loc='lower left'); a.grid(alpha=0.3)

a = ax[1]   # (2) ratio to Cholis XLIX best, stat band 을 ratio 로 표시
a.fill_between(Es, cl/cb, ch/cb, alpha=0.25, color='gray', label='Cholis 1σ band')
a.axhline(1.0, color='gray', lw=1)
a.plot(prod[:,0], prod[:,1]/cb, 'o-', color='tab:blue', ms=6, label='production / Cholis')
a.plot(Es[valid], (swap[:,1]/cb)[valid], 's-', color='tab:red', ms=6, label='SWAPBUB / Cholis')
a.set_xscale('log'); a.set_xlim(0.25, 60); a.set_ylim(0.4, 2.0)
a.set_xlabel('E [GeV]'); a.set_ylabel('ours / Cholis XLIX best')
a.set_title('GCE flux ratio to Cholis XLIX'); a.legend(); a.grid(alpha=0.3)

a = ax[2]   # (3) partition 계수: c_Bub(0 붕괴?), c_GCE
pz = np.load(prod_npz_f); sz = np.load(swap_npz_f)
pfp, sfp = pz['fitted_params'], sz['fitted_params']   # (5,14): [gas,ics,GCE,bub,iso]
a.plot(Es, pfp[3], 'o-', color='tab:blue', ms=6, label='c_Bub production')
a.plot(Es[valid], sfp[3][valid], 's-', color='tab:red', ms=6, label='c_Bub SWAPBUB')
a.plot(Es, pfp[2], 'o--', color='royalblue', ms=4, alpha=0.55, label='c_GCE production')
a.plot(Es[valid], sfp[2][valid], 's--', color='salmon', ms=4, alpha=0.55, label='c_GCE SWAPBUB')
a.axhline(0.0, color='gray', lw=0.8)
a.set_xscale('log'); a.set_xlim(0.25, 60)
a.set_xlabel('E [GeV]'); a.set_ylabel('best-fit coefficient')
a.set_title('Partition: c_Bub (0 붕괴 여부) & c_GCE'); a.legend(fontsize=8); a.grid(alpha=0.3)

plt.tight_layout()
_out = f'{_plots}/swapbub_XLIX_overlay.png'
plt.savefig(_out, dpi=130, bbox_inches='tight'); print(f'[saved] {_out}')
plt.show()

# 정량표
print(f'\n{"bin":>3} {"E":>6} {"Cholis":>10} {"prod":>10} {"swap":>10} {"prod/C":>7} {"swap/C":>7} {"cBub_p":>7} {"cBub_s":>7}')
for i in range(len(Es)):
    if not valid[i]: continue
    print(f'{i:>3} {Es[i]:>6.2f} {cb[i]:>10.3e} {prod[i,1]:>10.3e} {swap[i,1]:>10.3e} '
          f'{prod[i,1]/cb[i]:>7.3f} {swap[i,1]/cb[i]:>7.3f} {pfp[3][i]:>7.3f} {sfp[3][i]:>7.3f}')
mlo = valid & (Es <= 1.0); m110 = valid & (Es >= 1) & (Es <= 10)
print(f'\n저-E(≤1GeV) 평균 ratio  prod={np.mean((prod[:,1]/cb)[mlo]):.3f}  SWAP={np.mean((swap[:,1]/cb)[mlo]):.3f}'
      f'   |  평균 c_Bub  prod={np.mean(pfp[3][mlo]):.3f}  SWAP={np.mean(sfp[3][mlo]):.3f}')
print(f'1-10GeV  평균 ratio  prod={np.mean((prod[:,1]/cb)[m110]):.3f}  SWAP={np.mean((swap[:,1]/cb)[m110]):.3f}')

## Plot 7 — Spatial Residual Map (NEW)

Per-pixel residual `(data − model)/data` for `SELECTED_MODEL`, evaluated in 4
representative energy bins, plus latitude and longitude profiles. Ported from
12yr V11.

Reads PSF-convolved component cubes (`*_clean.fits`, **not** `_no_convol`) so the
reconstructed model matches what the runner used internally.

- RED pixels: data > model (model under-predicts)
- BLUE pixels: data < model (model over-predicts)
- gray: masked


In [ ]:
# Cell 12 — Plot 7: Spatial residual map + latitude/longitude profiles
if (E is None) or (SELECTED_MODEL not in coef_dict):
    print('[skip] need .npz coefficients for SELECTED_MODEL')
else:
    M = SELECTED_MODEL
    cd = coef_dict[M]
    cp, ci, cg, cb, co = cd['c_pion'], cd['c_ics'], cd['c_gce'], cd['c_bub'], cd['c_iso']

    # Load convolved component cubes (PSF-applied), NOT _no_convol
    needed = [
        ('pion',         comp_path('pion',         M, convol=True)),
        ('bremss',       comp_path('bremss',       M, convol=True)),
        ('ics',          comp_path('ics',          M, convol=True)),
        ('GCE',          comp_path('GCE',          None, convol=True)),
        ('fermi_bubble', comp_path('fermi_bubble', None, convol=True)),
        ('isotropic',    comp_path('isotropic',    None, convol=True)),
    ]
    missing = [p for _, p in needed if not os.path.exists(p)]
    if missing:
        print('[skip] missing convolved cubes (need *_clean.fits, NOT _no_convol):')
        for p in missing[:3]:
            print(f'  {p}')
        if len(missing) > 3:
            print(f'  ... and {len(missing) - 3} more')
    else:
        cubes = {n: fits.open(p)[0].data for n, p in needed}
        pb_cube = cubes['pion'] + cubes['bremss']

        # Build per-pixel model
        n_E_loc = pb_cube.shape[0]
        model_cube = np.zeros_like(pb_cube, dtype=float)
        for i in range(n_E_loc):
            model_cube[i] = (cp[i] * pb_cube[i]
                             + ci[i] * cubes['ics'][i]
                             + cg[i] * cubes['GCE'][i]
                             + co[i] * cubes['isotropic'][i]
                             + cb[i] * cubes['fermi_bubble'][i])

        # Inner ROI 400×400
        roi = slice(100, 500)
        data_roi  = counts_cube[:, roi, roi].astype(float)
        model_roi = model_cube[:, roi, roi]
        mask_roi  = (psc_mask[:, roi, roi] * disk_mask[roi, roi])

        # Residual fraction
        resid = np.full_like(data_roi, np.nan, dtype=float)
        pos = (data_roi > 0) & (mask_roi > 0)
        resid[pos] = (data_roi[pos] - model_roi[pos]) / data_roi[pos]

        # ---- 4-bin × 3-panel maps ----
        bins_to_show = [0, 5, 10, 13]
        fig, axes = plt.subplots(len(bins_to_show), 3,
                                 figsize=(15, 4.5 * len(bins_to_show)))
        for row, eb in enumerate(bins_to_show):
            d = data_roi[eb]; m = model_roi[eb]; r = resid[eb]
            mk = mask_roi[eb] > 0
            d_m = np.where(mk, d, np.nan)
            m_m = np.where(mk, m, np.nan)
            r_m = np.where(mk, r, np.nan)

            vmax = np.nanmax(d_m); vmin = max(0.1, np.nanmin(d_m[d_m > 0])
                                              if np.any(d_m > 0) else 0.1)
            for col, (arr, ttl, cmap) in enumerate([
                (d_m, 'data (counts)',         'inferno'),
                (m_m, 'fitted model (counts)', 'inferno'),
                (r_m, '(data − model)/data',   'RdBu_r'),
            ]):
                ax = axes[row, col]
                if col < 2:
                    nm = SymLogNorm(linthresh=1, vmin=vmin, vmax=vmax)
                    im = ax.imshow(arr, origin='lower', cmap=cmap, norm=nm)
                else:
                    im = ax.imshow(arr, origin='lower', cmap=cmap, vmin=-0.3, vmax=0.3)
                    fp = np.nansum(arr >  0.05) / mk.sum()
                    fn = np.nansum(arr < -0.05) / mk.sum()
                    mr = np.nanmean(arr)
                    ax.set_title(f'{ttl}    mean={mr:+.3f}\n'
                                 f'>+5%: {fp*100:.1f}%, <−5%: {fn*100:.1f}%',
                                 fontsize=10)
                if col < 2:
                    ax.set_title(f'{ttl}  (bin {eb}, E≈{E[eb]:.2f} GeV)', fontsize=10)
                ax.set_xlabel('pixel x ($\\ell$)')
                ax.set_ylabel('pixel y (b)')
                ax.grid(False)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        fig.suptitle(f'Spatial residual maps — Model {M}  '
                     f'(red = data > model)', fontsize=13, y=1.0)
        plt.tight_layout()
        out = f'{PLOTS_DIR}/07a_residual_maps_{M}.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        print(f'[saved] {out}')
        plt.show()

        # ---- Lat / long profiles ----
        ny_r, nx_r = resid.shape[-2:]
        # Approximate l, b axes (using astropy WCS from CCUBE)
        wcs2 = WCS(fits.open(f'{ANALYSIS_DIR}/GC_ccube_12yr{FRONT}_clean.fits')[0].header).dropaxis(2)
        l_axis = np.zeros(nx_r); b_axis = np.zeros(ny_r)
        for i in range(nx_r):
            l_axis[i], _ = wcs2.wcs_pix2world(i + 100, 300, 0)
        for j in range(ny_r):
            _, b_axis[j] = wcs2.wcs_pix2world(300, j + 100, 0)
        l_axis = ((l_axis + 180) % 360) - 180   # wrap to [-30, 30]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        cols = plt.cm.viridis(np.linspace(0, 0.9, len(bins_to_show)))
        for k, eb in enumerate(bins_to_show):
            r = resid[eb]
            b_prof = np.nanmean(r, axis=1)
            l_prof = np.nanmean(r, axis=0)
            ax1.plot(b_axis, b_prof, '-', color=cols[k], lw=1.5,
                     label=f'bin {eb}: {E[eb]:.2f} GeV')
            ax2.plot(l_axis, l_prof, '-', color=cols[k], lw=1.5,
                     label=f'bin {eb}: {E[eb]:.2f} GeV')
        for ax in (ax1, ax2):
            ax.axhline(0, color='black', ls='--', lw=0.5)
            ax.set_ylim(-0.3, 0.3); ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9)
        ax1.set_xlabel('b [deg]')
        ax1.set_ylabel(r'mean residual fraction (over $\ell$)')
        ax1.set_title('Latitude profile')
        ax2.set_xlabel(r'$\ell$ [deg]')
        ax2.set_ylabel('mean residual fraction (over b)')
        ax2.set_title('Longitude profile')
        ax2.invert_xaxis()
        fig.suptitle(f'Residual profiles — Model {M}', fontsize=13)
        plt.tight_layout()
        out = f'{PLOTS_DIR}/07b_residual_profiles_{M}.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        print(f'[saved] {out}')
        plt.show()


## DM Constraint Plots — χ² Contours

For each annihilation channel, scan the (m_DM, ⟨σv⟩) plane:

- **bb̄**: PPPC4 with EW correction (column index 13)
- **4b**: MG5 SFDM cascade (m_h2/m_χ = 0.5)
- **4τ**: MG5 SFDM cascade
- **2b2τ**: MG5 SFDM cascade

Confidence levels (2 free parameters): Δχ² = 2.30 (68%), 6.18 (95%), 11.83 (99.7%).
Test Statistic TS = χ²(no DM) − χ²(best fit), significance ≈ √TS.
Uses `cov_total = cov_stat ⊕ cov_sys`.


In [ ]:
# Cell 13 — χ² scan helpers
mass_range   = np.logspace(np.log10(10), np.log10(200), 40)   # 10-200 GeV
sigmav_range = np.logspace(-27, -25, 40)                       # cm³/s
print(f'Mass grid:    {len(mass_range)} pts, [{mass_range[0]:.1f}, {mass_range[-1]:.1f}] GeV')
print(f'σv grid:      {len(sigmav_range)} pts, [{sigmav_range[0]:.1e}, '
      f'{sigmav_range[-1]:.1e}] cm³/s')

if E is None or inv_cov is None:
    print('[abort] missing data or covariance — chi-square cannot proceed')


def model_flux_pppc4(dm_mass, sigmav, channel_idx=13, EW='Yes'):
    """E²·dN/dE on the GCE energy grid for a PPPC4 channel."""
    E_pp, dNdE_pp = exctractcirellitable(dm_mass, channel_idx, 'gammas', EW)
    valid = (E_pp > 0) & (dNdE_pp > 0)
    f = interp1d(np.log10(E_pp[valid]), np.log10(dNdE_pp[valid]),
                 bounds_error=False, fill_value=-np.inf)
    dNdE_gce = 10**f(np.log10(E))
    dNdE_gce = np.where(np.isfinite(dNdE_gce), dNdE_gce, 0.0)
    dNdE_gce = np.where(E > dm_mass, 0.0, dNdE_gce)
    return E**2 * dNdE_gce * (sigmav / dm_mass**2) * J_FACTOR / SR


def model_flux_mg5(dm_mass, sigmav, channel='4b'):
    """E²·dN/dE on the GCE grid using the MG5 cascade interpolator."""
    E_mg, dNdE_mg = MG5Interpolator(dm_mass, channel).interpolated_table()
    valid = (E_mg > 0) & (dNdE_mg > 0)
    if valid.sum() < 2:
        return np.zeros_like(E)
    f = interp1d(np.log10(E_mg[valid]), np.log10(dNdE_mg[valid]),
                 bounds_error=False, fill_value=-np.inf)
    dNdE_gce = 10**f(np.log10(E))
    dNdE_gce = np.where(np.isfinite(dNdE_gce), dNdE_gce, 0.0)
    dNdE_gce = np.where(E > dm_mass, 0.0, dNdE_gce)
    return E**2 * dNdE_gce * (sigmav / dm_mass**2) * J_FACTOR / SR


def chi_square(model_pred, data, inv_cov_, use_bins=None):
    """Chi^2 with optional bin slicing."""
    if use_bins is not None:
        diff = (data - model_pred)[use_bins]
    else:
        diff = data - model_pred
    return diff @ inv_cov_ @ diff


def scan_grid(model_func, m_arr, s_arr, data, inv_cov_, use_bins=None, **kw):
    """Scan (mass, sigmav) grid; if use_bins given, fit only those bins."""
    chi2 = np.zeros((len(m_arr), len(s_arr)))
    for i, m in enumerate(m_arr):
        for j, s in enumerate(s_arr):
            try:
                pred_full = model_func(m, s, **kw)
                chi2[i, j] = chi_square(pred_full, data, inv_cov_, use_bins=use_bins)
            except Exception:
                chi2[i, j] = np.inf
    return chi2

bb_best     = None
mg5_results = {}
print('[OK] χ² helpers ready')


In [ ]:
# Cell 14 — Plot 8: bb̄ χ² contour (PPPC4)
if E is None or inv_cov is None:
    print('[skip] no GCE data / cov')
else:
    print('Scanning bb̄ channel (PPPC4)...')
    chi2_bb = scan_grid(model_flux_pppc4, mass_range, sigmav_range,
                        flux_X, inv_cov_fit if inv_cov_fit is not None else inv_cov,
                        use_bins=CHI2_USE_BINS,
                        channel_idx=13, EW='Yes')
    min_chi2 = np.nanmin(chi2_bb)
    i_min, j_min = np.unravel_index(np.nanargmin(chi2_bb), chi2_bb.shape)
    best_m, best_s = mass_range[i_min], sigmav_range[j_min]
    if inv_cov_fit is not None:
        flux_fit = flux_X[CHI2_USE_BINS]
        chi2_null = flux_fit @ inv_cov_fit @ flux_fit
    else:
        chi2_null = flux_X @ inv_cov @ flux_X
    TS = chi2_null - min_chi2
    significance = np.sqrt(max(TS, 0))
    print(f'  best fit: m_DM={best_m:.1f} GeV, σv={best_s:.2e} cm³/s')
    print(f'  χ²_min = {min_chi2:.2f},  TS = {TS:.2f},  significance ≈ {significance:.2f}σ')

    fig, ax = plt.subplots(figsize=(8, 6.5))
    M_g, S_g = np.meshgrid(mass_range, sigmav_range, indexing='ij')
    levels = [min_chi2 + 2.30, min_chi2 + 6.18, min_chi2 + 11.83]
    cs = ax.contour(M_g, S_g, chi2_bb, levels=levels,
                    colors=['blue', 'green', 'orange'], linewidths=2)
    ax.clabel(cs, inline=True, fmt={l: lbl for l, lbl in
                                     zip(levels, ['68% CL', '95% CL', '99.7% CL'])})
    ax.plot(best_m, best_s, 'r*', ms=18, mec='black',
            label=f'best fit\n(m={best_m:.1f}, σv={best_s:.1e})')
    ax.axhline(3e-26, color='black', ls='--', alpha=0.7,
               label=r'thermal relic $\langle\sigma v\rangle = 3\times10^{-26}$ cm³/s')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
    ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm³/s]', fontsize=12)
    ax.set_title(f'bb̄ DM constraint (PPPC4, 12yr, Model {SELECTED_MODEL})\n'
                 f'TS = {TS:.1f}, significance ≈ {significance:.1f}σ', fontsize=12)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    out = f'{PLOTS_DIR}/08_chi2_bb.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()

    bb_best = {'channel': 'bb', 'm_DM': best_m, 'sigmav': best_s,
               'min_chi2': min_chi2, 'TS': TS, 'sig': significance,
               'chi2_grid': chi2_bb}


In [ ]:
# Cell 15 — Plots 9-11: 4b / 4τ / 2b2τ χ² contours (MG5 cascade)
mg5_results = {}

if E is None or inv_cov is None:
    print('[skip] no GCE data / cov')
else:
    plot_idx_map = {'4b': '09', '4tau': '10', '2b2tau': '11'}

    for channel in ['4b', '4tau', '2b2tau']:
        print(f'\n=== Scanning {channel} channel (MG5) ===')
        try:
            _ = MG5Interpolator(50.0, channel).interpolated_table()
            chi2_g = scan_grid(model_flux_mg5, mass_range, sigmav_range,
                               flux_X, inv_cov_fit if inv_cov_fit is not None else inv_cov,
                               use_bins=CHI2_USE_BINS, channel=channel)
            min_chi2 = np.nanmin(chi2_g)
            i, j = np.unravel_index(np.nanargmin(chi2_g), chi2_g.shape)
            best_m, best_s = mass_range[i], sigmav_range[j]
            if inv_cov_fit is not None:
                flux_fit = flux_X[CHI2_USE_BINS]
                chi2_null = flux_fit @ inv_cov_fit @ flux_fit
            else:
                chi2_null = flux_X @ inv_cov @ flux_X
            TS = chi2_null - min_chi2
            sig = np.sqrt(max(TS, 0))
            print(f'  best: m={best_m:.1f} GeV, σv={best_s:.2e} cm³/s, '
                  f'TS={TS:.2f}, significance={sig:.2f}σ')

            mg5_results[channel] = {'m_DM': best_m, 'sigmav': best_s,
                                    'min_chi2': min_chi2, 'TS': TS, 'sig': sig,
                                    'chi2_grid': chi2_g}

            # Plot
            fig, ax = plt.subplots(figsize=(8, 6.5))
            M_g, S_g = np.meshgrid(mass_range, sigmav_range, indexing='ij')
            levels = [min_chi2 + 2.30, min_chi2 + 6.18, min_chi2 + 11.83]
            cs = ax.contour(M_g, S_g, chi2_g, levels=levels,
                            colors=['blue', 'green', 'orange'], linewidths=2)
            ax.clabel(cs, inline=True,
                      fmt={l: lbl for l, lbl in zip(levels,
                                                    ['68% CL', '95% CL', '99.7% CL'])})
            ax.plot(best_m, best_s, 'r*', ms=18, mec='black',
                    label=f'best fit\n(m={best_m:.1f}, σv={best_s:.1e})')
            ax.axhline(3e-26, color='black', ls='--', alpha=0.7,
                       label='thermal relic')
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
            ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm³/s]', fontsize=12)
            chan_label = {'4b': r'4b', '4tau': r'4$\tau$',
                          '2b2tau': r'2b 2$\tau$'}.get(channel, channel)
            ax.set_title(f'{chan_label} DM constraint (MG5 SFDM, 12yr, '
                         f'Model {SELECTED_MODEL})\n'
                         f'TS = {TS:.1f}, significance ≈ {sig:.1f}σ', fontsize=12)
            ax.legend(loc='upper left', fontsize=10)
            ax.grid(True, alpha=0.3, which='both')
            plt.tight_layout()
            out = f'{PLOTS_DIR}/{plot_idx_map[channel]}_chi2_{channel}.png'
            plt.savefig(out, dpi=130, bbox_inches='tight')
            print(f'  [saved] {out}')
            plt.show()
        except Exception as e:
            print(f'  [skip] {channel}: {e}')


## Plot 12 — External Constraint Overlay (NEW, scaffolding)

Overlay Fermi dSph 95% UL and AMS-02 antiproton 95% UL on the bb̄ contour.
External files come from collaborators and have these expected names (in
`EXTERNAL_DATA_DIR`):

- `bb_14.0yr_30_dSphs.txt`, `bb_17.0yr_30_dSphs.txt` — bb̄ dSph UL
- `bbbb_14.0yr_30_dSphs.txt`, `bbbb_17.0yr_30_dSphs.txt` — 4b dSph UL
- `2b_phi_avg_*_data_model_cov.txt` (or similar) — antiproton bb̄ UL
- `4b_phi_avg_*_data_model_cov.txt` — antiproton 4b UL

The cell **gracefully skips** missing files — it's safe to run before/after these
arrive. Both bb̄ and 4b panels are produced in a single 2-panel figure.


In [ ]:
# Cell 16 — Plot 12: bb̄ + 4b contour + dSph + antiproton (graceful fallback)
def _try_load(prefix, fname_options):
    """Load 2-column txt from EXTERNAL_DATA_DIR; return (array, path-found) or (None, None)."""
    if EXTERNAL_DATA_DIR is None:
        return None, None
    for f in fname_options:
        p = f'{EXTERNAL_DATA_DIR}/{f}'
        if os.path.exists(p):
            try:
                d = np.loadtxt(p)
                d = d[d[:, 1] > 1.1e-28]   # filter sentinel rows
                return d, p
            except Exception:
                continue
    return None, None

# dSph
dsph_bb_14, _   = _try_load('bb14',   ['bb_14.0yr_30_dSphs.txt', 'bb_14_0yr_30_dSphs.txt'])
dsph_bb_17, _   = _try_load('bb17',   ['bb_17.0yr_30_dSphs.txt', 'bb_17_0yr_30_dSphs.txt'])
dsph_4b_14, _   = _try_load('4b14',   ['bbbb_14.0yr_30_dSphs.txt', 'bbbb_14_0yr_30_dSphs.txt'])
dsph_4b_17, _   = _try_load('4b17',   ['bbbb_17.0yr_30_dSphs.txt', 'bbbb_17_0yr_30_dSphs.txt'])
# Antiproton
ap_bb_17, _     = _try_load('apbb',   ['2b_phi_avg_prior_True_data_model_cov.txt',
                                       '2b_phi_avg_data_model_cov.txt',
                                       'antiproton_2b_12yr.txt'])
ap_4b_17, _     = _try_load('ap4b',   ['4b_phi_avg_prior_True_data_model_cov.txt',
                                       '4b_phi_avg_data_model_cov.txt',
                                       'antiproton_4b_12yr.txt'])

print('External constraint files:')
print(f'  dSph bb̄ 14yr: {"✓ " + str(len(dsph_bb_14)) + " pts" if dsph_bb_14 is not None else "✗"}')
print(f'  dSph bb̄ 12yr: {"✓ " + str(len(dsph_bb_17)) + " pts" if dsph_bb_17 is not None else "✗"}')
print(f'  dSph 4b 14yr: {"✓ " + str(len(dsph_4b_14)) + " pts" if dsph_4b_14 is not None else "✗"}')
print(f'  dSph 4b 12yr: {"✓ " + str(len(dsph_4b_17)) + " pts" if dsph_4b_17 is not None else "✗"}')
print(f'  AMS-02 bb̄  : {"✓ " + str(len(ap_bb_17)) + " pts" if ap_bb_17 is not None else "✗"}')
print(f'  AMS-02 4b   : {"✓ " + str(len(ap_4b_17)) + " pts" if ap_4b_17 is not None else "✗"}')

if bb_best is None:
    print('\n[skip] bb̄ contour not computed yet — run Cell 14')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

    panel_specs = [
        ('bb̄', r'$b\bar{b}$', bb_best, 'chi2_bb',
         dsph_bb_14, dsph_bb_17, ap_bb_17),
        ('4b',  r'$\chi\chi\to\phi\phi\to 4b$', mg5_results.get('4b'), 'chi2_4b',
         dsph_4b_14, dsph_4b_17, ap_4b_17),
    ]
    for ax, (label, latex, info, _, d14, d17, ap17) in zip(axes, panel_specs):
        if info is None:
            ax.text(0.5, 0.5, f'{label} contour not available',
                    ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.axis('off')
            continue

        chi2_g = info['chi2_grid']
        min_c  = info['min_chi2']
        bm, bs = info['m_DM'], info['sigmav']
        M_g, S_g = np.meshgrid(mass_range, sigmav_range, indexing='ij')
        levels = [min_c + 2.30, min_c + 6.18]
        try:
            ax.contour(M_g, S_g, chi2_g, levels=levels, colors='darkorange',
                       linestyles=['--', '-'], linewidths=[1.7, 2.1])
        except Exception:
            pass
        ax.plot(bm, bs, 'o', color='darkorange', ms=11, mec='black',
                label=f'12yr best fit\n($m={bm:.0f}$ GeV, $\\sigma v={bs:.1e}$)',
                zorder=20)
        ax.plot([], [], '--', color='darkorange', lw=1.7, label=r'12yr 1$\sigma$')
        ax.plot([], [], '-',  color='darkorange', lw=2.1, label=r'12yr 2$\sigma$')

        if d14 is not None:
            ax.plot(d14[:, 0], d14[:, 1], '-', color='black', lw=1.6,
                    alpha=0.85, label='Fermi dSph 95% UL (14yr)')
        if d17 is not None:
            ax.plot(d17[:, 0], d17[:, 1], '--', color='red', lw=1.6,
                    alpha=0.9, label='Fermi dSph 95% UL (12yr)')
        if ap17 is not None:
            ax.plot(ap17[:, 0], ap17[:, 1], '-', color='magenta', lw=1.7,
                    alpha=0.9, label='AMS-02 p̄ 95% UL')

        ax.axhline(3e-26, color='gray', ls=':', lw=1.2, alpha=0.7,
                   label=r'thermal relic')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlim(20, 200); ax.set_ylim(1e-27, 5e-25)
        ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
        ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm³/s]', fontsize=12)
        ax.text(0.04, 0.95, latex, transform=ax.transAxes, ha='left', va='top',
                fontsize=20, fontweight='bold')
        ax.set_title(f'{latex}: GCE 12yr + external '
                     f'(Model {SELECTED_MODEL})', fontsize=11)
        ax.grid(True, which='major', ls='--', lw=0.5, alpha=0.5)
        ax.legend(loc='lower right', fontsize=9, framealpha=0.92)

    plt.tight_layout()
    out = f'{PLOTS_DIR}/12_external_constraints.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    print(f'\n[saved] {out}')
    plt.show()


## Plot 13 — Cross-Comparison with Cholis 2022 Published Results (NEW)

Two-panel comparison showing analysis evolution from Cholis 2022 (12yr) to this
work (12yr). Useful as a thesis defense slide.

- **Left**: GCE flux for `SELECTED_MODEL` — Cholis Zenodo 12yr vs haebarg 12yr
- **Right**: bb̄ DM contour — Cholis 2022 Fig 18 published curves vs this analysis


In [ ]:
# Cell 17 — Plot 13: Cholis 12yr (published) vs haebarg 12yr
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.5))

# ---- Left: GCE flux comparison ----
ax = axes[0]
plotted_left = False
cholis_p = f'{CHOLIS_REF_DIR}/GCE_Model{SELECTED_MODEL}_flux_Inner40x40_masked_disk.dat'
if os.path.exists(cholis_p) and SELECTED_MODEL in all_flux:
    cd = np.loadtxt(cholis_p)
    if cd.shape[1] >= 4:
        E_c, f_c, lo_c, hi_c = cd[:, 0], cd[:, 1], cd[:, 2], cd[:, 3]
        yerr_c = [np.maximum(f_c - lo_c, 0), np.maximum(hi_c - f_c, 0)]
        ax.errorbar(E_c, f_c, yerr=yerr_c, marker='s', color='black',
                    ms=6, lw=1.6, capsize=4, label='Cholis+ 2022 (12yr)', zorder=10)
        ax.fill_between(E_c, lo_c, hi_c, color='black', alpha=0.10)
        plotted_left = True

if SELECTED_MODEL in all_flux:
    rm = all_flux[SELECTED_MODEL]
    yerr_h = [np.maximum(rm['flux'] - rm['lower'], 0),
              np.maximum(rm['upper'] - rm['flux'], 0)]
    ax.errorbar(rm['E'], rm['flux'], yerr=yerr_h, marker='^', color='darkorange',
                ms=6, lw=1.8, capsize=4, alpha=0.95,
                label='12yr (this work)', zorder=9)
    ax.fill_between(rm['E'], rm['lower'], rm['upper'],
                    color='darkorange', alpha=0.15)
    if sigma_sys is not None:
        ax.fill_between(rm['E'], rm['flux'] - sigma_sys, rm['flux'] + sigma_sys,
                        color='lightblue', alpha=0.40,
                        label=r'12yr $\pm\sigma_\mathrm{sys}$')
    plotted_left = True

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.3, 60); ax.set_ylim(1e-8, 5e-6)
ax.set_xlabel('E [GeV]', fontsize=12)
ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=12)
ax.set_title(f'GCE flux — Cholis 12yr vs 12yr (Model {SELECTED_MODEL})', fontsize=12)
ax.grid(True, which='major', ls='--', lw=0.5, alpha=0.5)
ax.legend(loc='lower left', fontsize=10, framealpha=0.92)
if not plotted_left:
    ax.text(0.5, 0.5, 'Cholis Zenodo flux not found\n(see CHOLIS_REF_DIR)',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)
    
# Sanghwan 16yr impl baseline overlay (graceful fallback; NOT scientific reference, per memory #12)
SANG_16YR_DAT = f'/home/sanghwan/FermiLAT/Sanghwan/GCE_model_{SELECTED_MODEL}_front_16yr_cholis.dat'
if os.path.exists(SANG_16YR_DAT):
    s16 = np.loadtxt(SANG_16YR_DAT)
    E_16, F_16 = s16[:,0], s16[:,1]
    S_16 = s16[:,2] if s16.shape[1] >= 3 else None
    if S_16 is not None:
        plt.errorbar(E_16, F_16, yerr=S_16, fmt='^--', color='blue', ms=5,
                     lw=1.0, capsize=2, alpha=0.5,
                     label=f'Sanghwan 16yr Model {SELECTED_MODEL} (impl baseline only)')
    else:
        plt.plot(E_16, F_16, '^--', color='blue', alpha=0.5,
                 label=f'Sanghwan 16yr Model {SELECTED_MODEL} (impl baseline only)')
else:
    print(f'  (Sanghwan 16yr {SELECTED_MODEL}: skipped, not found)')
# ============================================================

# ---- Right: bb̄ DM contour comparison ----
ax = axes[1]
plotted_right = False
if CHOLIS_FIG18_DIR is not None:
    def _safe_loadtxt(path):
        if not os.path.exists(path):
            return None
        for sk in (0, 1, 2):
            try:
                return np.loadtxt(path, skiprows=sk)
            except (ValueError, StopIteration):
                continue
        return None
    bf  = _safe_loadtxt(f'{CHOLIS_FIG18_DIR}/2112fig18left_best_fit_point.txt')
    da  = _safe_loadtxt(f'{CHOLIS_FIG18_DIR}/2112fig18left_dashed_line.txt')
    so  = _safe_loadtxt(f'{CHOLIS_FIG18_DIR}/2112fig18left_solid_line.txt')
    SCALE = 1e-26   # Cholis Fig 18 y-axis is in 10⁻²⁶ cm³/s
    for arr in (bf, da, so):
        if arr is None: continue
        if arr.ndim == 1:
            arr[1] *= SCALE
        else:
            arr[:, 1] *= SCALE
    if da is not None and da.ndim == 2 and da.shape[1] >= 2:
        ax.plot(da[:, 0], da[:, 1], '--', color='black', lw=1.6, alpha=0.85,
                label='Cholis 12yr 1σ')
    if so is not None and so.ndim == 2 and so.shape[1] >= 2:
        ax.plot(so[:, 0], so[:, 1], '-',  color='black', lw=1.9, alpha=0.85,
                label='Cholis 12yr 2σ')
    if bf is not None:
        if bf.ndim == 1:
            ax.plot(bf[0], bf[1], '*', color='black', ms=18, mec='gold',
                    label='Cholis 12yr best fit', zorder=12)
        else:
            ax.plot(bf[:, 0], bf[:, 1], '*', color='black', ms=14, mec='gold',
                    label='Cholis 12yr best fit', zorder=12)
    plotted_right = True

if bb_best is not None:
    M_g, S_g = np.meshgrid(mass_range, sigmav_range, indexing='ij')
    lvls = [bb_best['min_chi2'] + 2.30, bb_best['min_chi2'] + 6.18]
    try:
        ax.contour(M_g, S_g, bb_best['chi2_grid'], levels=lvls,
                   colors='darkorange', linestyles=['--', '-'],
                   linewidths=[1.7, 2.1])
    except Exception:
        pass
    ax.plot(bb_best['m_DM'], bb_best['sigmav'], 'o', color='darkorange',
            ms=11, mec='black',
            label=f"12yr best fit (m={bb_best['m_DM']:.0f} GeV, "
                  f"σv={bb_best['sigmav']:.1e})", zorder=11)
    plotted_right = True

ax.axhline(3e-26, color='gray', ls=':', lw=1.2, alpha=0.7, label='thermal relic')
ax.set_yscale('log')
ax.set_xlim(10, 200); ax.set_ylim(1e-27, 1e-24)
ax.set_xlabel(r'$m_\chi$ [GeV]', fontsize=12)
ax.set_ylabel(r'$\langle\sigma v\rangle$ [cm³/s]', fontsize=12)
ax.text(0.04, 0.95, r'$b\bar{b}$', transform=ax.transAxes,
        ha='left', va='top', fontsize=20, fontweight='bold')
ax.set_title(f'bb̄ DM contour: Cholis 12yr (Fig 18) vs 12yr', fontsize=12)
ax.grid(True, which='major', ls='--', lw=0.5, alpha=0.5)
ax.legend(loc='upper left', fontsize=9.5, framealpha=0.92)
if not plotted_right:
    ax.text(0.5, 0.5, 'Cholis Fig 18 reference\nnot available\n'
            '(see CHOLIS_FIG18_CANDIDATES in Cell 1)',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
out = f'{PLOTS_DIR}/13_cholis_comparison.png'
plt.savefig(out, dpi=140, bbox_inches='tight')
print(f'[saved] {out}')
plt.show()


## Plot 14 — Best-fit DM SED Overlay

All best-fit DM channel curves drawn on the GCE data, with stat ⊕ sys errors.


In [ ]:
# Cell 18 — Plot 14: Best-fit DM SED overlay
if E is None:
    print('[skip] no GCE data')
else:
    fig, ax = plt.subplots(figsize=(10.5, 7))

    sigma_total = (np.sqrt(stat_err**2 + sigma_sys**2) if sigma_sys is not None
                   else stat_err)
    ax.errorbar(E, flux_X, yerr=sigma_total, fmt='o', color='black',
                ms=5, capsize=3,
                label=f'Model {SELECTED_MODEL} (12yr, stat ⊕ sys)', zorder=4)
    if sigma_sys is not None:
        ax.fill_between(E, flux_X - sigma_sys, flux_X + sigma_sys,
                        color='lightblue', alpha=0.40,
                        label=r'$\pm 1\sigma_\mathrm{sys}$', zorder=2)

    colors = {'bb': 'red', '4b': 'green', '4tau': 'orange', '2b2tau': 'purple'}
    chan_label = {'bb': r'b$\bar{b}$', '4b': '4b', '4tau': r'4$\tau$',
                  '2b2tau': r'2b 2$\tau$'}

    if bb_best is not None:
        E_fine = np.logspace(np.log10(E[0]), np.log10(E[-1]), 200)
        E_pp, dNdE_pp = exctractcirellitable(bb_best['m_DM'], 13, 'gammas', 'Yes')
        valid = (E_pp > 0) & (dNdE_pp > 0)
        f = interp1d(np.log10(E_pp[valid]), np.log10(dNdE_pp[valid]),
                     bounds_error=False, fill_value=-np.inf)
        dNdE_fine = 10**f(np.log10(E_fine))
        dNdE_fine = np.where(np.isfinite(dNdE_fine) & (E_fine < bb_best['m_DM']),
                             dNdE_fine, 0.0)
        flux_dm = (E_fine**2 * dNdE_fine
                   * (bb_best['sigmav'] / bb_best['m_DM']**2) * J_FACTOR / SR)
        ax.plot(E_fine, flux_dm, color=colors['bb'], lw=2,
                label=f"best-fit {chan_label['bb']} "
                      f"(m={bb_best['m_DM']:.0f} GeV, σv={bb_best['sigmav']:.1e})")

    for ch, info in mg5_results.items():
        try:
            E_fine = np.logspace(np.log10(E[0]), np.log10(E[-1]), 200)
            E_mg, dNdE_mg = MG5Interpolator(info['m_DM'], ch).interpolated_table()
            valid = (E_mg > 0) & (dNdE_mg > 0)
            f = interp1d(np.log10(E_mg[valid]), np.log10(dNdE_mg[valid]),
                         bounds_error=False, fill_value=-np.inf)
            dNdE_fine = 10**f(np.log10(E_fine))
            dNdE_fine = np.where(np.isfinite(dNdE_fine) & (E_fine < info['m_DM']),
                                 dNdE_fine, 0.0)
            flux_dm = (E_fine**2 * dNdE_fine
                       * (info['sigmav'] / info['m_DM']**2) * J_FACTOR / SR)
            ax.plot(E_fine, flux_dm, color=colors.get(ch, 'gray'), lw=2, ls='--',
                    label=f"best-fit {chan_label.get(ch, ch)} "
                          f"(m={info['m_DM']:.0f} GeV)")
        except Exception:
            continue

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(0.2, 60); ax.set_ylim(1e-8, 5e-6)
    ax.set_xlabel('E [GeV]', fontsize=12)
    ax.set_ylabel(r'$E^2\,dN/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]', fontsize=12)
    ax.set_title(f'GCE 12yr — best-fit DM models overlay (Model {SELECTED_MODEL})',
                 fontsize=12)
    ax.grid(True, which='major', alpha=0.3)
    ax.legend(loc='lower left', fontsize=10, framealpha=0.92)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/14_best_fit_overlay.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    print(f'[saved] {out}')
    plt.show()


In [ ]:
# Cell 19 — Summary
print('=' * 64)
print(' Generated visualization outputs')
print('=' * 64)
for f in sorted(glob.glob(f'{PLOTS_DIR}/*.png')):
    sz = os.path.getsize(f) / 1024
    print(f'  {f}   ({sz:.0f} KB)')

print()
print(f'Reference Model:   {SELECTED_MODEL}')
print(f'Models compared:   {[m for m in MODELS_TO_COMPARE if m in coef_dict]}')
print()
print('Best-fit DM summary:')
if bb_best is not None:
    print(f'  bb̄    : m={bb_best["m_DM"]:>6.1f} GeV, '
          f'σv={bb_best["sigmav"]:.2e} cm³/s, TS={bb_best["TS"]:>5.1f}, '
          f'sig={bb_best["sig"]:.1f}σ')
for ch, info in mg5_results.items():
    print(f'  {ch:6s}: m={info["m_DM"]:>6.1f} GeV, '
          f'σv={info["sigmav"]:.2e} cm³/s, TS={info["TS"]:>5.1f}, '
          f'sig={info["sig"]:.1f}σ')


In [ ]:
# ======================================================================
# Plot: Cross-year GCE SED overlay (selected model)
#   Left  : 12yr Cholis (published) vs 16yr Sanghwan vs 12yr reproduce
#   Right : 12yr reproduce vs 16yr reproduce vs 12yr results
# Bands: Cholis = template-fit 1sigma ; reproduce = MCMC 16/84 percentile
# ======================================================================
import os
import numpy as np
import matplotlib.pyplot as plt

# --- model name (defend against _model pollution; default X) ---
if 'SELECTED_MODEL' in dir() and isinstance(SELECTED_MODEL, str):
    COMPARE_MODEL = SELECTED_MODEL
else:
    COMPARE_MODEL = 'X'
M = COMPARE_MODEL

# --- output dir ---
_PLOTS = PLOTS_DIR if ('PLOTS_DIR' in dir() and isinstance(PLOTS_DIR, str)) \
         else './GCE_12yr_results_plots'
os.makedirs(_PLOTS, exist_ok=True)

# --- root = parent of working dir (notebook runs in GCE_12yr_reproduce/) ---
_ROOT = os.path.abspath('..')
_SANG12 = '/home/sanghwan/FermiLAT/Sanghwan'

def _p(*parts):
    return os.path.join(_ROOT, *parts)

# (label, [candidate paths], kind)  kind: 'cholis4' (4-col) or 'repro5' (5-col)
SRC_REF = [
    ('12yr Cholis (published)',
     [_p('GCE_TEMPLATES_FILES_v3', 'Figures_12_and_14_GCE_Spectra',
         f'GCE_Model{M}_flux_Inner40x40_masked_disk.dat')],
     'cholis4'),
    ('16yr Sanghwan (front)',
     [_p('GCE_16yr_data', 'Sanghwan_result',
         f'GCE_model_{M}_front_16yr_cholis.dat')],
     'repro5'),
    ('12yr reproduce (this work)',
     [os.path.abspath(f'./results_12yr/GCE_model_{M}_front_12yr_cholis.dat')],
     'repro5'),
]

SRC_REPRO = [
    ('12yr reproduce',
     [f'{_SANG12}/GCE_model_{M}_12yr_cholis.dat',   # X 등: 먼저 시도
      f'{_SANG12}/GCE_model_{M}_12yr.dat'],          # Model I 예외: _cholis 없음
     'repro5'),
    ('16yr reproduce',
     [_p('GCE_16yr_reproduce', f'GCE_model_{M}_front_16yr_cholis.dat'),
      _p('GCE_16yr_reproduce', 'results_16yr',
         f'GCE_model_{M}_front_16yr_cholis.dat')],
     'repro5'),
    ('12yr results (this work)',
     [os.path.abspath(f'./results_12yr/GCE_model_{M}_front_12yr_cholis.dat')],
     'repro5'),
]

def _load(paths, kind):
    """Return (E, flux, lo, hi) or None. lo/hi are absolute band edges."""
    fp = next((q for q in paths if os.path.isfile(q)), None)
    if fp is None:
        return None, None
    d = np.loadtxt(fp, comments='#', ndmin=2)
    if d.size == 0:
        return None, fp
    E = d[:, 0]
    flux = d[:, 1]
    nc = d.shape[1]
    if kind == 'cholis4' and nc >= 4:          # E, bestfit, 1s_lo, 1s_hi
        lo, hi = d[:, 2], d[:, 3]
    elif kind == 'repro5' and nc >= 5:         # E, flux, stat, lo16, hi84
        lo, hi = d[:, 3], d[:, 4]
    elif nc >= 3:                              # E, flux, err  -> symmetric
        lo, hi = flux - d[:, 2], flux + d[:, 2]
    else:
        lo, hi = flux, flux
    return (E, flux, lo, hi), fp

_COL = {0: 'tab:red', 1: 'tab:blue', 2: 'k'}
_LS  = {0: '--',      1: '-.',       2: '-'}

def _draw(ax, sources, title):
    any_plotted = False
    for i, (lab, paths, kind) in enumerate(sources):
        res, fp = _load(paths, kind)
        if res is None:
            print(f'  [skip] {lab}: not found ({M})')
            continue
        E, fl, lo, hi = res
        m = np.isfinite(fl) & (fl > 0)
        c, ls = _COL[i], _LS[i]
        ax.plot(E[m], fl[m], ls, color=c, marker='o', ms=4, lw=1.6,
                label=lab, zorder=3)
        bm = m & np.isfinite(lo) & np.isfinite(hi) & (hi > 0)
        if bm.any():
            ax.fill_between(E[bm], np.clip(lo[bm], 1e-30, None), hi[bm],
                            color=c, alpha=0.15, lw=0, zorder=1)
        any_plotted = True
        print(f'  [ok]   {lab}: {os.path.relpath(fp, _ROOT)}  '
              f'({E.size} bins, ncols inferred)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('E [GeV]')
    ax.set_ylabel(r'$E^2\,d\Phi/dE$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]')
    ax.set_title(title, fontsize=11)
    ax.grid(True, which='both', alpha=0.25)
    if any_plotted:
        ax.legend(fontsize=8, framealpha=0.9)
    else:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                transform=ax.transAxes)

print(f'=== Cross-year overlay: Model {M} ===')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
print('[Left] reference / legacy')
_draw(axes[0], SRC_REF,
      f'Model {M}: 12yr Cholis vs 16yr Sanghwan vs 12yr reproduce')
print('[Right] own reproduction across years')
_draw(axes[1], SRC_REPRO,
      f'Model {M}: 12yr vs 16yr vs 12yr (this work)\n'
      f'(12yr results currently invalidated — buggy GCE template)')

# common y-range for fair visual comparison
yl = [ax.get_ylim() for ax in axes]
ymin = min(y[0] for y in yl); ymax = max(y[1] for y in yl)
for ax in axes:
    ax.set_ylim(ymin, ymax)

fig.suptitle(f'GCE SED cross-comparison — GDE Model {M}', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
_out = f'{_PLOTS}/15_crossyear_overlay_model_{M}.png'
fig.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'saved -> {_out}')

In [ ]:
import os, glob, time, numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from scipy.interpolate import interp1d

WD = '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce'; os.chdir(WD)
AN = './GC_analysis_DR2'; R = './results_12yr'
ZE = '../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
mt = lambda p: time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))) if os.path.exists(p) else 'MISSING'

# ---- 1) 현재 bubble template 정체 + provenance ----
print('### 1) bubble template ###')
bnc = f'{AN}/GC_fermi_bubble_model_12yr_front_clean_no_convol.fits'
bc  = f'{AN}/GC_fermi_bubble_model_12yr_front_clean.fits'
print('  no_convol mtime :', mt(bnc))
print('  convol    mtime :', mt(bc))
bk = (glob.glob(f'{AN}/*fermi_bubble*OLD*') + glob.glob(f'{AN}/*fermi_bubble*backup*')
      + glob.glob(f'{AN}/*fermi_bubble*_old*'))
print('  backup files    :', bk if bk else 'none')
h = fits.open(bnc); d = h[0].data; d = d[7] if d.ndim == 3 else d
w = WCS(h[0].header); w = w.dropaxis(2) if w.naxis >= 3 else w
H, Wd = d.shape
bb = np.array([w.wcs_pix2world(0, i, 0)[1] for i in range(H)])
sub = d[100:500, 100:500]; bsub = bb[100:500][:, None] * np.ones((1, 400))
m10 = np.abs(bsub) < 10
print(f'  |b|<10 ROI nonzero support = {(sub[m10] != 0).mean() * 100:5.1f}%   (new ~23%, old ~2%)')
col = sub[:, 200]; brow = bb[100:500]; nz = np.abs(brow[col != 0])
print(f'  inner |b| edge (central col) ~ {nz.min() if nz.size else float("nan"):.2f} deg   (new ~4.15, old ~7.45)')

# ---- 2) GCE flux: production vs swappbics vs paperbkg ----
print('\n### 2) GCE flux per-bin  (fp[2] x GCE x E^2/dE) ###')
def flux(p):
    if not os.path.exists(p): return None
    z = np.load(p); E = z['E']; return E, z['fitted_params'][2] * z['GCE'] * E**2 / z['delta_E']
prod = f'{R}/GCE_model_XLIX_front_12yr_cholis_fit.npz'
swpl = glob.glob(f'{R}/GCE_model_XLIX_12yr_swappbics_fit.npz') + glob.glob('./GCE_model_XLIX_12yr_swappbics_fit.npz')
pbk  = f'{R}/GCE_model_XLIX_12yr_paperbkg_fit.npz'
print('  production npz mtime :', mt(prod))
print('  swappbics npz        :', (swpl[0] if swpl else 'MISSING'))
def chol():
    cs = [f'{ZE}/GCE_ModelXLIX_flux_Inner40x40_masked_disk.dat',
          f'{ZE}/GCE_Model_XLIX_flux_Inner40x40_masked_disk.dat'] + sorted(glob.glob(f'{ZE}/GCE_ModelXLIX_*flux_Inner40x40_masked_disk.dat'))
    for c in cs:
        if os.path.exists(c): return np.loadtxt(c)
    return None
C = chol(); pa = interp1d(C[:, 0], C[:, 1], fill_value='extrapolate') if C is not None else (lambda x: np.nan)
fp_, fs_, fb_ = flux(prod), (flux(swpl[0]) if swpl else None), flux(pbk)
E = fp_[0]
print(f'{"bin":>3}{"E":>8}{"Cholis":>11}{"prod":>11}{"r":>6}{"swappbics":>12}{"r":>6}{"paperbkg":>11}{"r":>6}')
for j in range(len(E)):
    cv = pa(E[j]); row = f'{j:>3}{E[j]:>8.3f}{cv:>11.3e}{fp_[1][j]:>11.3e}{fp_[1][j]/cv:>6.2f}'
    row += f'{fs_[1][j]:>12.3e}{fs_[1][j]/cv:>6.2f}' if fs_ else f'{"-":>12}{"-":>6}'
    row += f'{fb_[1][j]:>11.3e}{fb_[1][j]/cv:>6.2f}'
    print(row)
for lab, m in [('<1 GeV  ', E < 1), ('1-10 GeV', (E >= 1) & (E <= 10)), ('>10 GeV ', E > 10)]:
    s = f'  [{lab}] prod={ (fp_[1][m]/pa(E[m])).mean():.3f}'
    s += f'  swappbics={ (fs_[1][m]/pa(E[m])).mean():.3f}' if fs_ else '  swappbics=NA'
    s += f'  paperbkg={ (fb_[1][m]/pa(E[m])).mean():.3f}'
    print(s)

# ---- 3) production our-bkg fit 의 저-E c (붕괴가 production 에도 있나) ----
print('\n### 3) production our-bkg fit c (bins 0-5) ###')
fp = np.load(prod)['fitted_params']
print(f'{"bin":>3} {"c_PB":>7} {"c_ICS":>7} {"c_GCE":>7} {"c_Bub":>7} {"c_Iso":>7}')
for j in range(6):
    print(f'{j:>3} {fp[0][j]:>7.3f} {fp[1][j]:>7.3f} {fp[2][j]:>7.3f} {fp[3][j]:>7.3f} {fp[4][j]:>7.3f}')

In [ ]:
import os, sys, numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from scipy.special import gammaln
import emcee

WD='/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce'; os.chdir(WD)
AN='./GC_analysis_DR2'; front='_front'; MODEL='XLIX'
PROD=f'./results_12yr/GCE_model_{MODEL}_front_12yr_cholis_fit.npz'
NW,NS,NB_=100,1000,400

z=np.load(PROD); E=z['E']; dE=z['delta_E']; gce_tmpl=z['GCE']; NB=len(E); E2dE=E**2/dE

_h=fits.open(f'{AN}/GC_ccube_12yr{front}_clean.fits')[0].header
_w=WCS(_h).dropaxis(2)
b_arr=np.array([_w.wcs_pix2world(0,i,0)[1] for i in range(600)])
srp=(np.radians(0.1)**2*np.cos(np.radians(b_arr)))[:,None]*np.ones((1,600))
psc=np.load(f'{AN}/Model/GC_mask_60x60_definitions_DR2.npy').astype(int)
disk=np.load(f'{AN}/Model/GC_disk_mask_60x60_definitions.npy')[100:500,100:500].astype(int)

def Lc(rel): return fits.open(f'{AN}/{rel}')[0].data
data_c=Lc(f'GC_ccube_12yr{front}_clean.fits')
PB_c =(Lc(f'GC_pion_model{MODEL}_12yr{front}_clean.fits')+Lc(f'GC_bremss_model{MODEL}_12yr{front}_clean.fits'))
ICS_c=Lc(f'GC_ics_model{MODEL}_12yr{front}_clean.fits')
GCE_c=Lc(f'GC_GCE_model_12yr{front}_clean.fits')
bub_c=Lc(f'GC_fermi_bubble_model_12yr{front}_clean.fits')
iso_c=Lc(f'GC_isotropic_model_12yr{front}_clean.fits')

class L:
    def __init__(self,eb,cfix):
        sl=(eb,slice(100,500),slice(100,500)); self.eb=eb; self.cfix=cfix
        self.data=data_c[sl]; self.PB=PB_c[sl]; self.ICS=ICS_c[sl]; self.GCE=GCE_c[sl]
        self.bub=bub_c[sl]; self.iso=iso_c[sl]
        self.fm=psc[eb,100:500,100:500]*disk
        self.olf=gammaln(self.data[self.fm==1].astype(float)+1.0)
        # 고정 성분(bubble+iso)의 기여를 미리 더함
        cb,ci=cfix
        self.fixed=cb*self.bub+ci*self.iso
    def m2lnL(self,p):
        cpb,cics,cgce=p
        exp=cpb*self.PB+cics*self.ICS+cgce*self.GCE+self.fixed
        om=self.data[self.fm==1]; em=exp[self.fm==1]
        if (em<0).any(): return np.inf
        return np.sum(2*(em-om*np.log(em)+self.olf))   # Poisson only (constraint off)

def logp(t,lk):
    if (t<0).any(): return -np.inf
    v=lk.m2lnL(t); return -np.inf if not np.isfinite(v) else -0.5*v

def run(cfix,tag):
    rng=np.random.default_rng(0); cg=np.zeros(NB); cpb=np.zeros(NB); cics=np.zeros(NB)
    for eb in range(NB):
        lk=L(eb,cfix); p0=np.abs(1.0+0.1*rng.standard_normal((NW,3)))
        s=emcee.EnsembleSampler(NW,3,logp,args=(lk,)); s.run_mcmc(p0,NS,progress=False)
        ch=s.get_chain(discard=NB_,flat=True); lp=s.get_log_prob(discard=NB_,flat=True)
        best=ch[np.argmax(lp)]; cpb[eb],cics[eb],cg[eb]=best
    flux=cg*gce_tmpl*E2dE
    print(f'\n--- {tag} ---')
    print(f'{"bin":>3}{"E":>8}{"c_PB":>7}{"c_ICS":>7}{"c_GCE":>7}{"GCEflux":>11}')
    for j in range(NB):
        print(f'{j:>3}{E[j]:>8.3f}{cpb[j]:>7.3f}{cics[j]:>7.3f}{cg[j]:>7.3f}{flux[j]:>11.3e}')
    return flux

# Cholis 저-E 비교용
ZE='../GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
import glob
cs=[f'{ZE}/GCE_ModelXLIX_flux_Inner40x40_masked_disk.dat']+sorted(glob.glob(f'{ZE}/GCE_ModelXLIX_*flux_Inner40x40_masked_disk.dat'))
C=np.loadtxt([c for c in cs if os.path.exists(c)][0]); chol=C[:,1]

print('=== bubble/iso degeneracy test (Poisson only, our input) ===')
fA=run((0,0),'A: bubble/iso 자유 아님 — 완전 제거 (c_Bub=c_Iso=0)')   # 먼저 C 케이스
# 위 run 은 cfix=(0,0) 이므로 bubble/iso 제거. 자유 baseline 은 5-param 이라 아래서 별도 처리 불필요:
fB=run((1,1),'B: bubble/iso 를 c=1 고정')

m=E<1
print('\n=== 저-E(<1 GeV) 평균 GCE/Cholis ===')
print(f'  production baseline (5 free) : 1.393  (이전 측정)')
print(f'  C: bubble/iso 제거(c=0)      : {(fA[m]/chol[m]).mean():.3f}')
print(f'  B: bubble/iso 고정(c=1)      : {(fB[m]/chol[m]).mean():.3f}')
print('해석: B 가 1.39 -> ~1.0 로 내려가면 degeneracy(GCE<-bubble/iso) 가 저-E 초과 주因.')
print('      B 도 여전히 ~1.3-1.4 면 data 가 GCE 를 직접 높이 요구 (bubble/iso 무관).')

In [ ]:
# check_bub_iso_railing.py — 절대값/band로 경계-railing 확인 (read-only)
import numpy as np
d = np.load('./results_12yr/GCE_model_XLIX_front_12yr_cholis_fit.npz')
MAP, MED = d['fitted_params'], d['fitted_params_median']   # (5,14)
LO, HI, SD = d['fitted_params_lower'], d['fitted_params_upper'], d['fitted_params_std']
E = d['E']
for gi, nm in [(3, 'Bub'), (4, 'Iso')]:
    print(f'\n=== {nm} (param row {gi}) ===')
    print(f'{"bin":>3}{"E[GeV]":>9}{"MAP":>11}{"median":>11}{"p16":>11}{"p84":>11}{"std":>11}')
    for b in range(len(E)):
        print(f'{b:>3}{E[b]:>9.3f}{MAP[gi,b]:>11.5f}{MED[gi,b]:>11.5f}'
              f'{LO[gi,b]:>11.5f}{HI[gi,b]:>11.5f}{SD[gi,b]:>11.5f}')